# Phase 3 — Ecotoxicity Modelling: Tuning, Species-Aware Learning and External Validation

**Author:** Bharath Reddy  ·  **Phase 3 plan dated:** 2 November 2026

---

## What this notebook does

Phase 2 established the baseline: organism-specific models, directly-learned biological-group
models, a hierarchical group→organism residual model, Bemis–Murcko scaffold validation and a
matched-parent halogenation analysis.

Phase 3 implements **all ten suggestions** from the Phase 3 plan. Each suggestion maps to one
step of this notebook:

| # | Suggestion from the Phase 3 plan | Step in this notebook |
|---|---|---|
| 1 | Optimise current models with Optuna; save best settings separately per organism and per group | **Step 4** |
| 2 | Test the extended RDKit descriptor set (~200) vs the current 11; check 512/1024/2048 fingerprint bits; use descriptor SHAP to trim | **Step 2** + **Step 9C** |
| 3 | Species-aware pooled models within each biological group (organism identity as a feature); RF / XGBoost / CatBoost vs independent and group models on identical splits | **Step 5** |
| 4 | Multitask neural network: shared layers, species-specific outputs, masked loss, early stopping; keep only if it clearly improves | **Step 6** |
| 5 | Identical scaffold-based held-out splits for every comparison; held-out scaffolds excluded from pooled training; nested cross-validation for tuning | **Step 3** (used by every later step) |
| 6 | Report R², MAE, RMSE and Spearman per organism with mean/median, variability and confidence intervals | **Step 7** |
| 7 | Averaging / stacking of the strongest models only, with weights learned from out-of-fold predictions | **Step 8** |
| 8 | Reliability on chemically similar vs dissimilar test compounds; verify Tanimoto against RDKit; report AD coverage and calibrated prediction intervals | **Step 9A/9B** |
| 9 | External benchmark datasets (≥2 organisms per category); halogen-only datasets for halogen patterns | **Step 10** + **Step 11** |
| 10 | Benchmarking against OECD / T.E.S.T. / ECOSAR / VEGA / ADMETSAR / ADMETlab / ProTox / Toxtree and standard ML models | **Step 11** |

---

## Design rules carried over from Phase 2 (do not change these)

1. **No explicit halogen labels inside prediction models.** `is_halogenated`, `halogen_type`,
   `F/Cl/Br/I_count` and `total_halogen_count` are used *only* for the separate halogenation
   analysis. Phase 3 extends this rule to the ~200 RDKit descriptor set, where the explicitly
   halogen-counting descriptors (`fr_halogen`, `fr_alkyl_halide`, `NumHalogens`, …) are dropped
   from model inputs — see `EXCLUDE_EXPLICIT_HALOGEN_DESCRIPTORS` in Step 0.
2. **Target is molar potency**, `pToxicity_molar = -log10(mol/L)`, converted from mg/L using
   molecular weight. mg/kg organisms are kept in a separate track and never mixed with mg/L.
3. **Group predictions are directly learned**, never averaged from organism predictions.
4. **One fixed set of scaffold splits** is created once in Step 3 and reused by every model, so
   all comparisons are on exactly the same test rows.

> Target interpretation: **higher `pToxicity_molar` = more toxic** (lower molar EC/LC50).

---

## Reproducibility

Every random process in this notebook is seeded from `RANDOM_STATE` (Step 0). Running the whole
notebook twice on the same input data gives identical tables. The environment, package versions
and the exact configuration used are written to `run_manifest.json` in the output folder, so the
numbers in the report can always be traced back to the run that produced them.

## Step 0 — Configuration, environment and run profile

**What this cell does:** installs/imports everything, fixes all random seeds, and defines a single
`RUN_PROFILE` switch that controls how heavy the run is.

**Choose your profile:**

| Profile | Approx. runtime (Kaggle CPU) | Use it for |
|---|---|---|
| `"quick"` | 30–60 min | Checking the notebook runs end-to-end; a demo |
| `"standard"` | 3–5 h | Normal experiments; what to show the professor |
| `"full"` | 8 h+ | Final publication numbers (consider splitting across two sessions) |

Set `RUN_PROFILE` in the cell below and run everything top to bottom. Nothing else needs editing
except `DATA_DIR` if your Kaggle dataset is attached at a different path.

In [1]:
# ============================================================
# STEP 0: CONFIGURATION AND ENVIRONMENT
# ============================================================
# This is the ONLY cell you normally need to edit.

from pathlib import Path
import os
import sys
import json
import time
import math
import random
import subprocess
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------
# 0.1 Run profile  --  "quick" | "standard" | "full"
# ------------------------------------------------------------------
RUN_PROFILE = "standard"

# ------------------------------------------------------------------
# 0.2 Paths
# ------------------------------------------------------------------
# Folder holding the per-organism Excel/CSV toxicity files (same dataset as Phase 2).
DATA_DIR = Path("C:\\Users\\lab\Desktop\\ecotoxicity\\Ecotoxicity ML datasets")

# Optional: if you attached the Phase-2 output folder as a Kaggle dataset, point here and the
# notebook will reuse the Phase-2 ML-ready table instead of rebuilding it (saves ~10 minutes).
# Leave as None to always rebuild from DATA_DIR.
PHASE2_ML_READY_CSV = None
# Example:
# PHASE2_ML_READY_CSV = Path("/kaggle/input/ecotox-phase2/step2_molecular_targets/step2_ml_ready.csv")

OUTPUT_DIR = Path("C:\\Users\\lab\Desktop\\ecotoxicity\\working/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 0.3 Reproducibility
# ------------------------------------------------------------------
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

# ------------------------------------------------------------------
# 0.4 Dataset-size thresholds (identical to Phase 2, so categories match)
# ------------------------------------------------------------------
STRONG_MIN = 1000
MEDIUM_MIN = 500
SMALL_MIN = 200

# Minimum requirements for an organism/group to get its own modelling task
MIN_ROWS_FOR_TASK = 500
MIN_SCAFFOLDS_FOR_TASK = 50

# ------------------------------------------------------------------
# 0.5 Profile-dependent settings
# ------------------------------------------------------------------
# N_SPLITS_MAIN      : number of outer scaffold splits (the held-out evaluation splits)
# N_INNER_FOLDS      : inner folds used for Optuna tuning and for out-of-fold stacking
# OPTUNA_TRIALS      : Optuna trials per (task, model)
# FP_BITS_TO_TEST    : fingerprint sizes compared in Step 2 (suggestion 2)
# MTL_MAX_SPLITS     : how many outer splits the multitask network is evaluated on (it is the
#                      slowest model, so "quick" evaluates it on fewer splits)
# N_BOOTSTRAP        : bootstrap resamples for the confidence intervals in Step 7
PROFILES = {
    "quick": dict(
        N_SPLITS_MAIN=3, N_INNER_FOLDS=3, OPTUNA_TRIALS=12,
        FP_BITS_TO_TEST=[1024], MTL_MAX_SPLITS=1, N_BOOTSTRAP=300,
        RUN_SHAP=True, SHAP_MAX_SAMPLES=300,
    ),
    "standard": dict(
        N_SPLITS_MAIN=5, N_INNER_FOLDS=3, OPTUNA_TRIALS=30,
        FP_BITS_TO_TEST=[512, 1024, 2048], MTL_MAX_SPLITS=2, N_BOOTSTRAP=1000,
        RUN_SHAP=True, SHAP_MAX_SAMPLES=600,
    ),
    "full": dict(
        N_SPLITS_MAIN=10, N_INNER_FOLDS=5, OPTUNA_TRIALS=60,
        FP_BITS_TO_TEST=[512, 1024, 2048], MTL_MAX_SPLITS=5, N_BOOTSTRAP=2000,
        RUN_SHAP=True, SHAP_MAX_SAMPLES=1500,
    ),
}

if RUN_PROFILE not in PROFILES:
    raise ValueError(f"RUN_PROFILE must be one of {list(PROFILES)}")

CFG = PROFILES[RUN_PROFILE]
N_SPLITS_MAIN = CFG["N_SPLITS_MAIN"]
N_INNER_FOLDS = CFG["N_INNER_FOLDS"]
OPTUNA_TRIALS = CFG["OPTUNA_TRIALS"]
FP_BITS_TO_TEST = CFG["FP_BITS_TO_TEST"]
MTL_MAX_SPLITS = CFG["MTL_MAX_SPLITS"]
N_BOOTSTRAP = CFG["N_BOOTSTRAP"]
RUN_SHAP = CFG["RUN_SHAP"]
SHAP_MAX_SAMPLES = CFG["SHAP_MAX_SAMPLES"]

TEST_SIZE = 0.20          # fraction of SCAFFOLDS held out in each outer split
N_BITS = 1024             # default fingerprint size used by the main models
CONFORMAL_ALPHA = 0.10    # 0.10 -> 90% prediction intervals

# Phase 2 rule: explicit halogen information must not enter prediction models.
# Set to False only if you deliberately want a halogen-aware ablation.
EXCLUDE_EXPLICIT_HALOGEN_DESCRIPTORS = True

print(f"RUN_PROFILE        : {RUN_PROFILE}")
print(f"Outer splits       : {N_SPLITS_MAIN}")
print(f"Inner folds        : {N_INNER_FOLDS}")
print(f"Optuna trials      : {OPTUNA_TRIALS}")
print(f"Fingerprint sizes  : {FP_BITS_TO_TEST}")
print(f"Output directory   : {OUTPUT_DIR}")

RUN_PROFILE        : standard
Outer splits       : 5
Inner folds        : 3
Optuna trials      : 30
Fingerprint sizes  : [512, 1024, 2048]
Output directory   : C:\Users\lab\Desktop\ecotoxicity\working


In [2]:
# ============================================================
# STEP 0 (continued): IMPORTS
# ============================================================
# Kaggle images normally ship RDKit, XGBoost, LightGBM and PyTorch. Anything missing is
# installed here. Turn Kaggle's "Internet" setting ON if an install is needed.

def _pip_install(package):
    """Install a package quietly; return True on success."""
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        return True
    except Exception as exc:
        print(f"  could not install {package}: {exc}")
        return False


# ---------------- RDKit (required) ----------------
try:
    from rdkit import Chem, DataStructs, RDLogger
except ImportError:
    print("Installing RDKit ...")
    _pip_install("rdkit")
    from rdkit import Chem, DataStructs, RDLogger

from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors, MACCSkeys
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.ML.Descriptors import MoleculeDescriptors

RDLogger.DisableLog("rdApp.*")   # silence the harmless valence/charge warnings

# ---------------- Core scientific stack ----------------
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                     # safe headless backend; figures are still displayed
import matplotlib.pyplot as plt

np.random.seed(RANDOM_STATE)

from scipy.stats import spearmanr, wilcoxon, mannwhitneyu

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor,
)
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------- Optional libraries ----------------
AVAILABLE = {}

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    AVAILABLE["optuna"] = True
except ImportError:
    AVAILABLE["optuna"] = _pip_install("optuna")
    if AVAILABLE["optuna"]:
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    from xgboost import XGBRegressor
    AVAILABLE["xgboost"] = True
except ImportError:
    AVAILABLE["xgboost"] = _pip_install("xgboost")
    if AVAILABLE["xgboost"]:
        from xgboost import XGBRegressor

try:
    from lightgbm import LGBMRegressor
    AVAILABLE["lightgbm"] = True
except ImportError:
    AVAILABLE["lightgbm"] = _pip_install("lightgbm")
    if AVAILABLE["lightgbm"]:
        from lightgbm import LGBMRegressor

try:
    from catboost import CatBoostRegressor
    AVAILABLE["catboost"] = True
except ImportError:
    AVAILABLE["catboost"] = _pip_install("catboost")
    if AVAILABLE["catboost"]:
        from catboost import CatBoostRegressor

try:
    import shap
    AVAILABLE["shap"] = True
except ImportError:
    AVAILABLE["shap"] = _pip_install("shap")
    if AVAILABLE["shap"]:
        import shap

try:
    import torch
    import torch.nn as nn
    AVAILABLE["torch"] = True
    torch.manual_seed(RANDOM_STATE)
except ImportError:
    AVAILABLE["torch"] = False

print("Library availability:")
for lib, ok in AVAILABLE.items():
    print(f"  {lib:<10} {'yes' if ok else 'NO  (that step will be skipped)'}")

if AVAILABLE.get("torch"):
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  torch device: {DEVICE}")

# ---------------- Small helpers used throughout ----------------
def save_table(df, directory, filename, show=True, max_rows=30):
    """Save a dataframe to CSV and optionally display the first rows."""
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / filename
    df.to_csv(path, index=False)
    print(f"  saved: {path}  ({len(df)} rows)")
    if show and len(df):
        display(df.head(max_rows))
    return path


def round_df(df, digits=4):
    """Round every numeric column, for clean report tables."""
    out = df.copy()
    numeric_cols = out.select_dtypes(include=[np.number]).columns
    out[numeric_cols] = out[numeric_cols].round(digits)
    return out


class Timer:
    """Context manager that prints how long a step took."""

    def __init__(self, label):
        self.label = label

    def __enter__(self):
        self.t0 = time.time()
        print(f"\n>>> {self.label} ...")
        return self

    def __exit__(self, *exc):
        print(f"<<< {self.label} finished in {time.time() - self.t0:.1f} s")
        return False


print("\nEnvironment ready.")

Library availability:
  optuna     yes
  xgboost    yes
  lightgbm   yes
  catboost   yes
  shap       yes
  torch      yes
  torch device: cuda

Environment ready.


## Step 1 — Load the dataset and rebuild the Phase-2 ML-ready table

**Why:** every Phase 3 comparison must start from exactly the same rows, the same molar target and
the same scaffolds as Phase 2. This step reproduces the Phase-2 preparation in one cell.

**What it produces:** `ml_ready` — one row per (chemical, organism) measurement with
canonical SMILES, the 11 basic descriptors, halogen tags (analysis only), the Bemis–Murcko
scaffold and the molar target `target_pToxicity_molar`.

If you set `PHASE2_ML_READY_CSV` in Step 0, that file is loaded instead and the rebuild is skipped.

In [3]:
# ============================================================
# STEP 1: DATA LOADING AND PHASE-2-COMPATIBLE PREPARATION
# ============================================================

# ---- Organism name spelling fixes (identical to Phase 2) ----
ORGANISM_NAME_FIXES = {
    "Raphidocelis subcapita": "Raphidocelis subcapitata",
    "Raphidocelis subcapitata": "Raphidocelis subcapitata",
    "Desmodes subcapitus": "Desmodesmus subspicatus",
    "Desmodesmus subspicatus": "Desmodesmus subspicatus",
    "Americamycis bahia": "Americamysis bahia",
    "Americamysis bahia": "Americamysis bahia",
    "Ceridaphnia dubia": "Ceriodaphnia dubia",
    "Ceriodaphnia dubia": "Ceriodaphnia dubia",
    "Galus Galus": "Gallus gallus",
    "Gallus gallus": "Gallus gallus",
    "Cortunix japonica": "Coturnix japonica",
    "Coturnix japonica": "Coturnix japonica",
    "Chironomus riparious": "Chironomus riparius",
    "Chironomus riparius": "Chironomus riparius",
    "Anas platyrynchus": "Anas platyrhynchos",
    "Anas platyrhynchos": "Anas platyrhynchos",
    "Tetrarhyncus urticae": "Tetranychus urticae",
    "Tetranychus urticae": "Tetranychus urticae",
    "Trilobium castaneum": "Tribolium castaneum",
    "Tribolium castaneum": "Tribolium castaneum",
}

# ---- Organism -> biological group (identical to Phase 2) ----
KNOWN_GROUPS = {
    "Danio rerio": "Fish",
    "Oryzias latipes": "Fish",
    "Pimephales promelas": "Fish",
    "Oncorhynchus mykiss": "Fish",
    "Daphnia magna": "Invertebrates",
    "Ceriodaphnia dubia": "Invertebrates",
    "Americamysis bahia": "Invertebrates",
    "Chironomus riparius": "Invertebrates",
    "Raphidocelis subcapitata": "Algae",
    "Desmodesmus subspicatus": "Algae",
    "Skeletonema costatum": "Algae",
    "Chlorella vulgaris": "Algae",
    "Eisenia fetida": "Terrestrial worms",
    "Folsomia candida": "Terrestrial insects and arthropods",
    "Apis mellifera": "Terrestrial insects and arthropods",
    "Tetranychus urticae": "Terrestrial insects and arthropods",
    "Drosophila melanogaster": "Terrestrial insects and arthropods",
    "Tribolium castaneum": "Terrestrial insects and arthropods",
    "Coturnix japonica": "Avian",
    "Gallus gallus": "Avian",
    "Colinus virginianus": "Avian",
    "Anas platyrhynchos": "Avian",
}

AQUATIC_GROUPS = ["Fish", "Algae", "Invertebrates"]          # mg/L track
TERRESTRIAL_GROUPS = ["Avian", "Terrestrial insects and arthropods", "Terrestrial worms"]


def normalize_text(x):
    return str(x).strip().replace("_", " ").replace("-", " ")


def infer_organism_name(file_path):
    """Work out the organism from the file name or its parent folder."""
    for candidate in (file_path.stem, file_path.parent.name):
        cleaned = normalize_text(candidate)
        cleaned = re.sub(r"\b(dataset|data|ecotoxicity|toxicity|ml|xlsx|xls|csv)\b", "",
                         cleaned, flags=re.I).strip()
        cleaned = re.sub(r"\s+", " ", cleaned)
        for wrong, right in ORGANISM_NAME_FIXES.items():
            if wrong.lower() in cleaned.lower():
                return right
        for known in KNOWN_GROUPS:
            if known.lower() in cleaned.lower():
                return known
    fallback = re.sub(r"\s+", " ", normalize_text(file_path.stem)).strip()
    return ORGANISM_NAME_FIXES.get(fallback, fallback)


def infer_group(organism_name, file_path):
    if organism_name in KNOWN_GROUPS:
        return KNOWN_GROUPS[organism_name]
    parent = normalize_text(file_path.parent.name).lower()
    if "fish" in parent:
        return "Fish"
    if "alga" in parent:
        return "Algae"
    if "avian" in parent or "bird" in parent:
        return "Avian"
    if "invertebrate" in parent:
        return "Invertebrates"
    if "terrestrial" in parent or "insect" in parent or "arthropod" in parent:
        return "Terrestrial insects and arthropods"
    return "Unknown"


def find_col(columns, patterns, must_be_numeric=False, df=None):
    """Find the first column whose name matches any of the regex patterns."""
    for pattern in patterns:
        for col in columns:
            if re.search(pattern, str(col), flags=re.I):
                if must_be_numeric and df is not None:
                    if pd.to_numeric(df[col], errors="coerce").notna().sum() == 0:
                        continue
                return col
    return None


def detect_unit(raw_df, file_path):
    """mg/L for the aquatic files, mg/kg for avian and terrestrial ones."""
    text = " ".join(map(str, raw_df.columns)) + " " + str(file_path)
    if re.search(r"mg\s*/\s*kg|mgkg|mg kg", text, flags=re.I):
        return "mg/kg"
    if re.search(r"mg\s*/\s*l|mg/l|mg l|mgL", text, flags=re.I):
        return "mg/L"
    group = KNOWN_GROUPS.get(infer_organism_name(file_path), "")
    return "mg/kg" if group in TERRESTRIAL_GROUPS else "mg/L"


def dataset_size_category(n):
    if n >= STRONG_MIN:
        return "strong"
    if n >= MEDIUM_MIN:
        return "medium"
    if n >= SMALL_MIN:
        return "small"
    return "very_small"


def load_raw_master(data_dir):
    """Read every Excel/CSV file under data_dir into one long table."""
    files = sorted(
        list(data_dir.rglob("*.xlsx")) + list(data_dir.rglob("*.xls"))
        + list(data_dir.rglob("*.csv"))
    )
    print(f"Files found under {data_dir}: {len(files)}")
    if not files:
        raise FileNotFoundError(
            f"No Excel/CSV files under {data_dir}. Fix DATA_DIR in Step 0 "
            f"(in Kaggle, check the exact path shown in the right-hand Input panel)."
        )

    parts, report = [], []
    for fp in files:
        try:
            raw = pd.read_excel(fp) if fp.suffix.lower() in (".xlsx", ".xls") else pd.read_csv(fp)
        except Exception as exc:
            report.append({"file": str(fp), "status": "read_error", "error": str(exc)})
            continue

        organism = infer_organism_name(fp)
        group = infer_group(organism, fp)
        unit = detect_unit(raw, fp)

        smiles_col = find_col(raw.columns, [r"smiles", r"canonical"], df=raw)
        inchi_col = find_col(raw.columns, [r"inchi"], df=raw)
        name_col = find_col(raw.columns, [r"chemical", r"compound", r"name"], df=raw)
        tox_col = find_col(
            raw.columns,
            [r"ic50", r"lc50", r"ec50", r"tox", r"value", r"concentration", r"endpoint"],
            must_be_numeric=True, df=raw,
        )
        if tox_col is None:
            # Fall back to the most-populated numeric column that is not an index/ID.
            counts = {
                c: pd.to_numeric(raw[c], errors="coerce").notna().sum()
                for c in raw.columns if not re.search(r"id|index|no\.?$", str(c), flags=re.I)
            }
            tox_col = max(counts, key=counts.get) if counts else None

        temp = pd.DataFrame(index=raw.index)
        temp["source_file"] = str(fp)
        temp["organism_group"] = group
        temp["organism_name"] = organism
        temp["unit"] = unit
        temp["chemical_name"] = raw[name_col] if name_col else np.nan
        temp["smiles"] = raw[smiles_col] if smiles_col else np.nan
        temp["inchi_key"] = raw[inchi_col] if inchi_col else np.nan
        temp["toxicity_value_raw"] = (
            pd.to_numeric(raw[tox_col], errors="coerce") if tox_col else np.nan
        )
        parts.append(temp)
        report.append({
            "file": str(fp), "status": "ok", "rows": len(raw),
            "organism_group": group, "organism_name": organism, "unit": unit,
            "smiles_col": smiles_col, "toxicity_col": tox_col,
        })

    master = pd.concat(parts, ignore_index=True)
    master["row_id"] = np.arange(len(master))
    return master, pd.DataFrame(report)

In [4]:
# ============================================================
# STEP 1 (continued): MOLECULES, SCAFFOLDS AND THE MOLAR TARGET
# ============================================================

def mol_from_smiles(smiles):
    """Parse a SMILES string; return None if it is missing or invalid."""
    if pd.isna(smiles):
        return None
    try:
        return Chem.MolFromSmiles(str(smiles).strip())
    except Exception:
        return None


def canonicalize_smiles(smiles):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return np.nan
    try:
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return np.nan


# The 11 basic descriptors used in Phase 2. Kept so Phase 3 can compare "11 vs ~200".
BASIC_DESCRIPTORS = [
    "mol_weight", "logP", "TPSA", "num_h_donors", "num_h_acceptors",
    "num_rotatable_bonds", "num_rings", "num_aromatic_rings",
    "heavy_atom_count", "formal_charge", "fraction_csp3",
]


def compute_basic_descriptors(smiles):
    """The Phase-2 descriptor set. mol_weight is also needed for the molar conversion."""
    mol = mol_from_smiles(smiles)
    if mol is None:
        return {name: np.nan for name in BASIC_DESCRIPTORS}
    return {
        "mol_weight": Descriptors.MolWt(mol),
        "logP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "num_h_donors": Lipinski.NumHDonors(mol),
        "num_h_acceptors": Lipinski.NumHAcceptors(mol),
        "num_rotatable_bonds": Lipinski.NumRotatableBonds(mol),
        "num_rings": rdMolDescriptors.CalcNumRings(mol),
        "num_aromatic_rings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "heavy_atom_count": mol.GetNumHeavyAtoms(),
        "formal_charge": Chem.GetFormalCharge(mol),
        "fraction_csp3": rdMolDescriptors.CalcFractionCSP3(mol),
    }


def halogen_counts(smiles):
    """Halogen tags. ANALYSIS ONLY - never used as a model input (Phase 2 rule)."""
    mol = mol_from_smiles(smiles)
    counts = {"F_count": 0, "Cl_count": 0, "Br_count": 0, "I_count": 0}
    if mol is None:
        return {**counts, "total_halogen_count": 0,
                "is_halogenated": False, "halogen_type": "invalid_smiles"}
    symbol_map = {"F": "F_count", "Cl": "Cl_count", "Br": "Br_count", "I": "I_count"}
    for atom in mol.GetAtoms():
        key = symbol_map.get(atom.GetSymbol())
        if key:
            counts[key] += 1
    total = sum(counts.values())
    present = [k.replace("_count", "") for k, v in counts.items() if v > 0]
    if total == 0:
        htype = "non_halogenated"
    elif len(present) == 1:
        htype = f"{present[0]}_only"
    else:
        htype = "mixed_halogen"
    return {**counts, "total_halogen_count": total,
            "is_halogenated": total > 0, "halogen_type": htype}


def get_scaffold(smiles):
    """Bemis-Murcko scaffold; falls back to the whole molecule for acyclic compounds."""
    mol = mol_from_smiles(smiles)
    if mol is None:
        return "invalid_scaffold"
    try:
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        return scaffold if scaffold else Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return "scaffold_error"


def convert_to_molar_target(row):
    """mg/L -> mol/L -> pToxicity_molar = -log10(mol/L). Same for mg/kg -> mol/kg."""
    value, mw = row["toxicity_value_raw"], row["mol_weight"]
    if pd.isna(value) or pd.isna(mw) or value <= 0 or mw <= 0:
        return np.nan
    molar = (value / 1000.0) / mw          # mg -> g, then g / (g/mol) = mol
    return -np.log10(molar) if molar > 0 else np.nan


def prepare_ml_ready(master_raw):
    """Add molecules, descriptors, halogen tags, scaffolds and the target; flag usable rows."""
    master = master_raw.copy()
    master["valid_smiles"] = master["smiles"].apply(lambda s: mol_from_smiles(s) is not None)
    master["canonical_smiles"] = master["smiles"].apply(canonicalize_smiles)

    print("  computing basic descriptors ...")
    master = pd.concat(
        [master, pd.DataFrame(master["canonical_smiles"].apply(compute_basic_descriptors).tolist())],
        axis=1,
    )

    print("  computing halogen tags (analysis only) ...")
    master = pd.concat(
        [master, pd.DataFrame(master["canonical_smiles"].apply(halogen_counts).tolist())],
        axis=1,
    )

    print("  computing Bemis-Murcko scaffolds ...")
    master["scaffold"] = master["canonical_smiles"].apply(get_scaffold)

    master["positive_toxicity"] = master["toxicity_value_raw"] > 0
    master["target_pToxicity_molar"] = master.apply(convert_to_molar_target, axis=1)
    master["model_target"] = master["target_pToxicity_molar"]

    master["ready_for_ml"] = (
        master["valid_smiles"]
        & master["positive_toxicity"]
        & master["model_target"].notna()
        & master[BASIC_DESCRIPTORS].notna().all(axis=1)
    )

    ready_counts = master[master["ready_for_ml"]].groupby("organism_name").size().to_dict()
    master["organism_ready_rows"] = (
        master["organism_name"].map(ready_counts).fillna(0).astype(int)
    )
    master["dataset_size_category"] = master["organism_ready_rows"].apply(dataset_size_category)
    return master


# ------------------------------------------------------------------
# Run Step 1
# ------------------------------------------------------------------
step1_dir = OUTPUT_DIR / "step1_data"

with Timer("Step 1: loading and preparing data"):
    if PHASE2_ML_READY_CSV is not None and Path(PHASE2_ML_READY_CSV).exists():
        print(f"Reusing the Phase-2 ML-ready table: {PHASE2_ML_READY_CSV}")
        ml_ready = pd.read_csv(PHASE2_ML_READY_CSV)
        # Rebuild anything the saved file may not contain.
        if "scaffold" not in ml_ready.columns:
            ml_ready["scaffold"] = ml_ready["canonical_smiles"].apply(get_scaffold)
        if "model_target" not in ml_ready.columns:
            ml_ready["model_target"] = ml_ready["target_pToxicity_molar"]
        if "row_id" not in ml_ready.columns:
            ml_ready["row_id"] = np.arange(len(ml_ready))
        master = ml_ready.copy()
    else:
        master_raw, file_report = load_raw_master(DATA_DIR)
        print(f"Raw rows loaded: {len(master_raw)}")
        master = prepare_ml_ready(master_raw)
        ml_ready = master[master["ready_for_ml"]].reset_index(drop=True).copy()
        save_table(file_report, step1_dir, "step1_file_report.csv", show=False)

# Guarantee a stable, unique row identifier for the whole notebook.
ml_ready = ml_ready.reset_index(drop=True)
ml_ready["row_id"] = np.arange(len(ml_ready))

print(f"\nML-ready rows: {len(ml_ready)}")
print(f"Unique chemicals: {ml_ready['canonical_smiles'].nunique()}")
print(f"Unique scaffolds: {ml_ready['scaffold'].nunique()}")

dataset_summary = (
    ml_ready.groupby(["organism_group", "organism_name", "unit", "dataset_size_category"])
    .agg(
        rows=("row_id", "count"),
        unique_chemicals=("canonical_smiles", "nunique"),
        unique_scaffolds=("scaffold", "nunique"),
        halogenated_rows=("is_halogenated", "sum"),
        target_min=("model_target", "min"),
        target_median=("model_target", "median"),
        target_max=("model_target", "max"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

save_table(round_df(dataset_summary), step1_dir, "step1_dataset_summary.csv")


>>> Step 1: loading and preparing data ...
Files found under C:\Users\lab\Desktop\ecotoxicity\Ecotoxicity ML datasets: 21
Raw rows loaded: 15488
  computing basic descriptors ...
  computing halogen tags (analysis only) ...
  computing Bemis-Murcko scaffolds ...
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step1_data\step1_file_report.csv  (21 rows)
<<< Step 1: loading and preparing data finished in 28.9 s

ML-ready rows: 15478
Unique chemicals: 8112
Unique scaffolds: 4840
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step1_data\step1_dataset_summary.csv  (21 rows)


,organism_group,organism_name,unit,dataset_size_category,rows,unique_chemicals,unique_scaffolds,halogenated_rows,target_min,target_median,target_max
2,Algae,Raphidocelis subcapitata,mg/L,strong,3165,3165,2163,581,0.3886,3.8981,12.3180
8,Fish,Danio rerio,mg/L,strong,2482,2482,1649,636,0.1613,3.6808,8.8192
9,Fish,Oncorhynchus mykiss,mg/L,strong,2132,2130,1476,622,0.2025,4.0136,9.9213
1,Algae,Desmodesmus subspicatus,mg/L,strong,1612,1612,1061,389,1.0809,3.8052,9.0774
11,Fish,Pimephales promelas,mg/L,strong,1581,1581,977,427,-0.0835,3.8352,11.2774
15,Invertebrates,Daphnia magna,mg/L,strong,1000,1000,690,286,0.0525,3.7408,8.5348
10,Fish,Oryzias latipes,mg/L,medium,862,862,530,255,-0.0516,3.8177,10.2816
3,Algae,Skeletonema costatum,mg/L,medium,514,514,373,221,0.0430,4.4231,9.6765
12,Invertebrates,Americamysis bahia,mg/L,small,458,458,310,225,0.4799,5.2057,11.0274
0,Algae,Chlorella vulgaris,mg/L,small,408,408,249,157,-0.2626,3.9182,8.2277


WindowsPath('C:/Users/lab/Desktop/ecotoxicity/working/step1_data/step1_dataset_summary.csv')

## Step 2 — Feature library: extended RDKit descriptors and fingerprint sizes

*Addresses suggestion 2: "maybe the current set of descriptors are not able to completely
correlate the chemical–biological implications… we might need to check the extended RDKit
descriptor sets (around 200) itself apart from the current 11 used. The lower bits of 1024 are
showing better accuracy than 2048."*

**What this cell does**

1. Computes the **full RDKit descriptor list** (~210 descriptors) for every unique chemical.
2. Cleans it: drops `Ipc` (it overflows for large molecules), replaces infinities, then drops
   descriptors that are constant or more than 95 % missing.
3. **Removes the descriptors that count halogens explicitly** (`fr_halogen`, `fr_alkyl_halide`,
   `NumHalogens`, …), because the Phase 2 rule forbids explicit halogen information inside
   prediction models. The dropped list is printed and saved so the decision is auditable.
4. Removes one of every pair of descriptors correlated above |r| = 0.95, keeping the first.
5. Builds Morgan fingerprints at **512, 1024 and 2048 bits** so the "1024 beats 2048" observation
   can be tested properly rather than assumed.

Descriptors are computed **once per unique chemical** and then joined back onto the rows, which is
far faster than computing them per measurement (the same chemical appears for many organisms).

In [5]:
# ============================================================
# STEP 2: EXTENDED DESCRIPTOR LIBRARY AND FINGERPRINTS
# ============================================================

step2_dir = OUTPUT_DIR / "step2_features"

# Halogen columns that must never enter a prediction model (Phase 2 rule).
HALOGEN_ANALYSIS_COLUMNS = [
    "F_count", "Cl_count", "Br_count", "I_count",
    "total_halogen_count", "is_halogenated", "halogen_type",
]

# RDKit descriptor names that encode halogen content explicitly.
EXPLICIT_HALOGEN_DESCRIPTORS = [
    "fr_halogen", "fr_alkyl_halide", "NumHalogens", "fr_Ar_X",
]

# ------------------------------------------------------------------
# 2.1 Full RDKit descriptor calculator
# ------------------------------------------------------------------
ALL_RDKIT_DESCRIPTOR_NAMES = [name for name, _ in Descriptors._descList]

# Ipc grows factorially and overflows to inf for larger molecules - a known RDKit pitfall.
ALL_RDKIT_DESCRIPTOR_NAMES = [n for n in ALL_RDKIT_DESCRIPTOR_NAMES if n != "Ipc"]

_descriptor_calculator = MoleculeDescriptors.MolecularDescriptorCalculator(
    ALL_RDKIT_DESCRIPTOR_NAMES
)
print(f"RDKit exposes {len(ALL_RDKIT_DESCRIPTOR_NAMES)} usable descriptors (Ipc excluded).")


def compute_extended_descriptors(smiles):
    """All RDKit descriptors for one molecule, as a plain list of floats."""
    mol = mol_from_smiles(smiles)
    if mol is None:
        return [np.nan] * len(ALL_RDKIT_DESCRIPTOR_NAMES)
    try:
        return list(_descriptor_calculator.CalcDescriptors(mol))
    except Exception:
        return [np.nan] * len(ALL_RDKIT_DESCRIPTOR_NAMES)


# ------------------------------------------------------------------
# 2.2 Compute once per UNIQUE chemical, then join back onto the rows
# ------------------------------------------------------------------
unique_chemicals = (
    ml_ready[["canonical_smiles"]].drop_duplicates().reset_index(drop=True)
)
print(f"Computing extended descriptors for {len(unique_chemicals)} unique chemicals ...")

with Timer("extended descriptor calculation"):
    descriptor_values = np.array(
        [compute_extended_descriptors(s) for s in unique_chemicals["canonical_smiles"]],
        dtype=float,
    )

extended_desc = pd.DataFrame(descriptor_values, columns=ALL_RDKIT_DESCRIPTOR_NAMES)
extended_desc.insert(0, "canonical_smiles", unique_chemicals["canonical_smiles"].values)

# ------------------------------------------------------------------
# 2.3 Clean the descriptor block
# ------------------------------------------------------------------
numeric_block = extended_desc[ALL_RDKIT_DESCRIPTOR_NAMES]
numeric_block = numeric_block.replace([np.inf, -np.inf], np.nan)

cleaning_log = []

# (a) mostly-missing descriptors
missing_fraction = numeric_block.isna().mean()
drop_missing = missing_fraction[missing_fraction > 0.05].index.tolist()
cleaning_log.append({"rule": "missing > 5%", "n_dropped": len(drop_missing),
                     "descriptors": ", ".join(drop_missing[:40])})

# (b) constant descriptors - they carry no information
remaining = numeric_block.drop(columns=drop_missing)
variances = remaining.var(numeric_only=True)
drop_constant = variances[(variances.isna()) | (variances < 1e-12)].index.tolist()
cleaning_log.append({"rule": "zero variance", "n_dropped": len(drop_constant),
                     "descriptors": ", ".join(drop_constant[:40])})

remaining = remaining.drop(columns=drop_constant)

# (c) explicit halogen descriptors - excluded by the Phase 2 modelling rule
drop_halogen = []
if EXCLUDE_EXPLICIT_HALOGEN_DESCRIPTORS:
    drop_halogen = [c for c in remaining.columns if c in EXPLICIT_HALOGEN_DESCRIPTORS]
    remaining = remaining.drop(columns=drop_halogen)
cleaning_log.append({"rule": "explicit halogen descriptor (Phase 2 rule)",
                     "n_dropped": len(drop_halogen),
                     "descriptors": ", ".join(drop_halogen)})

# (d) one of each pair correlated above |r| = 0.95
correlation = remaining.fillna(remaining.median()).corr().abs()
upper_triangle = correlation.where(
    np.triu(np.ones(correlation.shape, dtype=bool), k=1)
)
drop_correlated = [c for c in upper_triangle.columns if (upper_triangle[c] > 0.95).any()]
remaining = remaining.drop(columns=drop_correlated)
cleaning_log.append({"rule": "|correlation| > 0.95", "n_dropped": len(drop_correlated),
                     "descriptors": ", ".join(drop_correlated[:40])})

EXTENDED_DESCRIPTORS = list(remaining.columns)

# Fill any residual gaps with the column median so models never see NaN.
extended_clean = remaining.fillna(remaining.median())
extended_clean.insert(0, "canonical_smiles", extended_desc["canonical_smiles"].values)

print(f"\nExtended descriptors kept: {len(EXTENDED_DESCRIPTORS)} "
      f"(from {len(ALL_RDKIT_DESCRIPTOR_NAMES)})")
if drop_halogen:
    print(f"Explicit halogen descriptors removed: {drop_halogen}")

cleaning_log_df = pd.DataFrame(cleaning_log)
save_table(cleaning_log_df, step2_dir, "step2_descriptor_cleaning_log.csv")
save_table(pd.DataFrame({"descriptor": EXTENDED_DESCRIPTORS}), step2_dir,
           "step2_extended_descriptors_kept.csv", show=False)

# Lookup table: canonical_smiles -> cleaned extended descriptors
EXTENDED_LOOKUP = extended_clean.set_index("canonical_smiles")

RDKit exposes 216 usable descriptors (Ipc excluded).
Computing extended descriptors for 8112 unique chemicals ...

>>> extended descriptor calculation ...
<<< extended descriptor calculation finished in 88.7 s

Extended descriptors kept: 166 (from 216)
Explicit halogen descriptors removed: ['fr_alkyl_halide', 'fr_halogen']
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step2_features\step2_descriptor_cleaning_log.csv  (4 rows)


,rule,n_dropped,descriptors
0,missing > 5%,8,"BCUT2D_MWHI, BCUT2D_MWLOW, BCUT2D_CHGHI, BCUT2..."
1,zero variance,5,"SMR_VSA8, SlogP_VSA9, fr_benzodiazepine, fr_di..."
2,explicit halogen descriptor (Phase 2 rule),2,"fr_alkyl_halide, fr_halogen"
3,|correlation| > 0.95,35,"MaxEStateIndex, HeavyAtomMolWt, ExactMolWt, Nu..."


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step2_features\step2_extended_descriptors_kept.csv  (166 rows)


In [6]:
# ============================================================
# STEP 2 (continued): FINGERPRINTS AND FEATURE-MATRIX BUILDER
# ============================================================

def fp_to_numpy(fingerprint, n_bits):
    """RDKit bit vector -> numpy uint8 array."""
    array = np.zeros((n_bits,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fingerprint, array)
    return array


# Cache of fingerprint matrices, keyed by (fingerprint type, number of bits).
# Rows are aligned to UNIQUE_SMILES_ORDER below.
UNIQUE_SMILES_ORDER = list(unique_chemicals["canonical_smiles"])
SMILES_TO_POSITION = {s: i for i, s in enumerate(UNIQUE_SMILES_ORDER)}
_FP_CACHE = {}


def _build_fp_matrix(kind, n_bits):
    """Compute a fingerprint matrix once per unique chemical and cache it."""
    key = (kind, n_bits)
    if key in _FP_CACHE:
        return _FP_CACHE[key]

    if kind == "morgan":
        generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=n_bits)
    elif kind == "atom_pair":
        generator = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=n_bits)
    elif kind == "torsion":
        generator = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=n_bits)
    elif kind == "maccs":
        generator = None
        n_bits = 167
    else:
        raise ValueError(f"Unknown fingerprint type: {kind}")

    rows = []
    for smi in UNIQUE_SMILES_ORDER:
        mol = mol_from_smiles(smi)
        if mol is None:
            rows.append(np.zeros(n_bits, dtype=np.uint8))
        elif kind == "maccs":
            rows.append(fp_to_numpy(MACCSkeys.GenMACCSKeys(mol), 167))
        else:
            rows.append(fp_to_numpy(generator.GetFingerprint(mol), n_bits))

    matrix = np.vstack(rows).astype(np.uint8)
    _FP_CACHE[key] = matrix
    return matrix


def get_fingerprint_frame(smiles_series, kind="morgan", n_bits=N_BITS):
    """Fingerprint rows for the given SMILES, taken from the cached unique-chemical matrix."""
    matrix = _build_fp_matrix(kind, n_bits)
    actual_bits = matrix.shape[1]
    positions = [SMILES_TO_POSITION[s] for s in smiles_series]
    return pd.DataFrame(
        matrix[positions],
        columns=[f"{kind}{actual_bits}_{i}" for i in range(actual_bits)],
    ).reset_index(drop=True)


def get_extended_descriptor_frame(smiles_series):
    """Cleaned extended descriptors for the given SMILES."""
    return EXTENDED_LOOKUP.loc[list(smiles_series)].reset_index(drop=True)


def build_feature_matrix(data, feature_set="basic11_morgan1024"):
    """
    Build the model input matrix.

    Naming: <descriptor block>_<fingerprint block>
      basic11      - the 11 Phase-2 descriptors
      ext          - the cleaned extended RDKit descriptor set (~150-190 kept)
      morgan512/1024/2048, maccs, atompair1024, torsion1024

    Explicit halogen columns are never included, in any feature set.
    """
    data = data.reset_index(drop=True)
    smiles = data["canonical_smiles"]
    blocks = []

    if feature_set.startswith("basic11"):
        blocks.append(data[BASIC_DESCRIPTORS].reset_index(drop=True))
    elif feature_set.startswith("ext"):
        blocks.append(get_extended_descriptor_frame(smiles))
    else:
        raise ValueError(f"Feature set must start with 'basic11' or 'ext': {feature_set}")

    if "morgan512" in feature_set:
        blocks.append(get_fingerprint_frame(smiles, "morgan", 512))
    if "morgan1024" in feature_set:
        blocks.append(get_fingerprint_frame(smiles, "morgan", 1024))
    if "morgan2048" in feature_set:
        blocks.append(get_fingerprint_frame(smiles, "morgan", 2048))
    if "maccs" in feature_set:
        blocks.append(get_fingerprint_frame(smiles, "maccs", 167))
    if "atompair1024" in feature_set:
        blocks.append(get_fingerprint_frame(smiles, "atom_pair", 1024))
    if "torsion1024" in feature_set:
        blocks.append(get_fingerprint_frame(smiles, "torsion", 1024))

    matrix = pd.concat(blocks, axis=1)

    # Safety net: assert that no explicit halogen column slipped in.
    forbidden = [c for c in matrix.columns
                 if c in HALOGEN_ANALYSIS_COLUMNS
                 or (EXCLUDE_EXPLICIT_HALOGEN_DESCRIPTORS and c in EXPLICIT_HALOGEN_DESCRIPTORS)]
    if forbidden:
        raise AssertionError(f"Explicit halogen columns leaked into the features: {forbidden}")

    return matrix.astype(np.float32)


# ------------------------------------------------------------------
# Which feature sets to compare in Step 3's screening
# ------------------------------------------------------------------
FEATURE_SETS_TO_SCREEN = ["basic11", "ext"]
for bits in FP_BITS_TO_TEST:
    FEATURE_SETS_TO_SCREEN.append(f"basic11_morgan{bits}")
    FEATURE_SETS_TO_SCREEN.append(f"ext_morgan{bits}")

# The feature set used by the main models; Step 3 may override it with the screening winner.
MAIN_FEATURE_SET = "ext_morgan1024"

print("Feature sets to screen:")
for fs in FEATURE_SETS_TO_SCREEN:
    print("  -", fs)
print(f"\nDefault main feature set: {MAIN_FEATURE_SET} (Step 3 may replace this with the winner)")

Feature sets to screen:
  - basic11
  - ext
  - basic11_morgan512
  - ext_morgan512
  - basic11_morgan1024
  - ext_morgan1024
  - basic11_morgan2048
  - ext_morgan2048

Default main feature set: ext_morgan1024 (Step 3 may replace this with the winner)


## Step 3 — One fixed set of scaffold splits for every model

*Addresses suggestion 5: "Use the same scaffold-based held-out splits for all model comparisons.
Keep chemicals with held-out scaffolds out of pooled training data, and use nested
cross-validation for tuning where practical."*

**Why this step is the backbone of Phase 3.** If each model were evaluated on its own random
split, a difference in R² could simply mean one model got an easier test set. Here the splits are
created **once**, saved to disk, and every later step — Optuna, species-aware models, the
multitask network, stacking, the applicability domain — reads the same `row_id` lists.

**Two protections against leakage:**

1. **Outer splits** group by Bemis–Murcko scaffold (`GroupShuffleSplit`), so a scaffold is either
   entirely in train or entirely in test — never both.
2. **Pooled-training rule.** When a pooled model (group, species-aware, hierarchical, multitask)
   is trained to predict organism *O*'s test rows, every row in the pool whose scaffold appears in
   *O*'s test set is removed first. The helper `pooled_training_rows()` enforces this, and Step 3C
   asserts that the overlap really is zero.

**Nested cross-validation.** Inside each outer training set, `inner_folds()` builds
`N_INNER_FOLDS` scaffold-grouped folds. Optuna (Step 4) scores on these inner folds and the
stacking weights (Step 8) are learned from their out-of-fold predictions. The outer test rows are
never touched during tuning.

In [7]:
# ============================================================
# STEP 3A: CREATE AND SAVE THE FIXED SCAFFOLD SPLITS
# ============================================================

step3_dir = OUTPUT_DIR / "step3_splits"

# ------------------------------------------------------------------
# Modelling tracks. mg/L and mg/kg are never mixed.
# ------------------------------------------------------------------
aquatic = ml_ready[ml_ready["unit"] == "mg/L"].reset_index(drop=True).copy()
terrestrial = ml_ready[ml_ready["unit"] == "mg/kg"].reset_index(drop=True).copy()

print(f"Aquatic (mg/L) rows     : {len(aquatic)}")
print(f"Terrestrial (mg/kg) rows: {len(terrestrial)}")

# Organisms large enough for their own model (the Phase 2 criterion).
organism_sizes = aquatic.groupby("organism_name").agg(
    rows=("row_id", "count"), scaffolds=("scaffold", "nunique")
)
MODELLED_ORGANISMS = sorted(
    organism_sizes[
        (organism_sizes["rows"] >= MIN_ROWS_FOR_TASK)
        & (organism_sizes["scaffolds"] >= MIN_SCAFFOLDS_FOR_TASK)
    ].index.tolist()
)
print(f"\nOrganisms with their own model ({len(MODELLED_ORGANISMS)}):")
for org in MODELLED_ORGANISMS:
    print(f"  {org:<30} {organism_sizes.loc[org, 'rows']:>5} rows, "
          f"{organism_sizes.loc[org, 'scaffolds']:>4} scaffolds")

# Biological groups large enough for a directly-learned group model.
group_sizes = ml_ready.groupby(["organism_group", "unit"]).agg(
    rows=("row_id", "count"), scaffolds=("scaffold", "nunique")
)
MODELLED_GROUPS = [
    (grp, unit) for (grp, unit), row in group_sizes.iterrows()
    if row["rows"] >= MIN_ROWS_FOR_TASK and row["scaffolds"] >= MIN_SCAFFOLDS_FOR_TASK
]
print(f"\nBiological groups with their own model ({len(MODELLED_GROUPS)}): {MODELLED_GROUPS}")


def make_scaffold_splits(data, n_splits=N_SPLITS_MAIN, test_size=TEST_SIZE,
                         random_state=RANDOM_STATE):
    """
    Split rows so that whole scaffolds are held out.

    Returns a long dataframe: one line per (split_id, row_id, train/test).
    Using row_id (not positional index) means any later step can re-apply the split
    to a differently-ordered dataframe without any risk of misalignment.
    """
    data = data.reset_index(drop=True)
    splitter = GroupShuffleSplit(
        n_splits=n_splits, test_size=test_size, random_state=random_state
    )
    records = []
    for split_id, (train_idx, test_idx) in enumerate(
        splitter.split(np.zeros(len(data)), data["model_target"], groups=data["scaffold"]), 1
    ):
        for role, idx in (("train", train_idx), ("test", test_idx)):
            for i in idx:
                records.append({
                    "split_id": split_id,
                    "row_id": int(data.loc[i, "row_id"]),
                    "split_role": role,
                })
    return pd.DataFrame(records)


# ------------------------------------------------------------------
# Build one split set per organism task and per group task
# ------------------------------------------------------------------
SPLITS = {}          # task_id -> split dataframe
split_frames = []

for organism in MODELLED_ORGANISMS:
    subset = aquatic[aquatic["organism_name"] == organism]
    task_id = f"organism__{organism}"
    splits = make_scaffold_splits(subset)
    splits["task_id"] = task_id
    splits["entity_type"] = "organism"
    splits["entity_name"] = organism
    SPLITS[task_id] = splits
    split_frames.append(splits)

for grp, unit in MODELLED_GROUPS:
    subset = ml_ready[(ml_ready["organism_group"] == grp) & (ml_ready["unit"] == unit)]
    task_id = f"group__{grp}__{unit}"
    splits = make_scaffold_splits(subset)
    splits["task_id"] = task_id
    splits["entity_type"] = "group"
    splits["entity_name"] = f"{grp} | {unit}"
    SPLITS[task_id] = splits
    split_frames.append(splits)

all_splits = pd.concat(split_frames, ignore_index=True)
save_table(all_splits, step3_dir, "step3_fixed_scaffold_splits.csv", show=False)
print(f"\nSaved {all_splits['task_id'].nunique()} split tasks "
      f"x {N_SPLITS_MAIN} splits to {step3_dir}")

# Fast lookup: row_id -> position in ml_ready
ROWID_TO_INDEX = {int(r): i for i, r in enumerate(ml_ready["row_id"].values)}

Aquatic (mg/L) rows     : 14593
Terrestrial (mg/kg) rows: 885

Organisms with their own model (8):
  Danio rerio                     2482 rows, 1649 scaffolds
  Daphnia magna                   1000 rows,  690 scaffolds
  Desmodesmus subspicatus         1612 rows, 1061 scaffolds
  Oncorhynchus mykiss             2132 rows, 1476 scaffolds
  Oryzias latipes                  862 rows,  530 scaffolds
  Pimephales promelas             1581 rows,  977 scaffolds
  Raphidocelis subcapitata        3165 rows, 2163 scaffolds
  Skeletonema costatum             514 rows,  373 scaffolds

Biological groups with their own model (3): [('Algae', 'mg/L'), ('Fish', 'mg/L'), ('Invertebrates', 'mg/L')]
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step3_splits\step3_fixed_scaffold_splits.csv  (139705 rows)

Saved 11 split tasks x 5 splits to C:\Users\lab\Desktop\ecotoxicity\working\step3_splits


In [8]:
# ============================================================
# STEP 3B: SPLIT HELPERS USED BY EVERY LATER STEP
# ============================================================

def get_split_rows(task_id, split_id):
    """Return (train_row_ids, test_row_ids) for one split of one task."""
    splits = SPLITS[task_id]
    selected = splits[splits["split_id"] == split_id]
    train_ids = selected.loc[selected["split_role"] == "train", "row_id"].values
    test_ids = selected.loc[selected["split_role"] == "test", "row_id"].values
    return train_ids, test_ids


def rows_by_id(data, row_ids):
    """Select rows of `data` by row_id, preserving the given order."""
    indexed = data.set_index("row_id")
    return indexed.loc[list(row_ids)].reset_index()


def pooled_training_rows(pool, test_row_ids, reference=None):
    """
    Build a leakage-free pooled training set.

    Every row of `pool` whose scaffold appears anywhere in the test rows is removed,
    which is the rule stated in suggestion 5: "Keep chemicals with held-out scaffolds
    out of pooled training data."
    """
    reference = ml_ready if reference is None else reference
    test_scaffolds = set(rows_by_id(reference, test_row_ids)["scaffold"])
    clean = pool[~pool["scaffold"].isin(test_scaffolds)].reset_index(drop=True)
    return clean, test_scaffolds


def inner_folds(train_data, n_folds=N_INNER_FOLDS):
    """
    Scaffold-grouped inner folds for nested CV (Optuna tuning and OOF stacking).

    Yields (train_positions, validation_positions) as positional indices into train_data.
    Falls back to a single 80/20 scaffold split when there are too few scaffolds.
    """
    train_data = train_data.reset_index(drop=True)
    n_scaffolds = train_data["scaffold"].nunique()
    if n_scaffolds < n_folds * 2:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
        yield from splitter.split(np.zeros(len(train_data)), groups=train_data["scaffold"])
        return
    splitter = GroupKFold(n_splits=n_folds)
    yield from splitter.split(np.zeros(len(train_data)), groups=train_data["scaffold"])


def evaluate_predictions(y_true, y_pred):
    """The four metrics requested in suggestion 6."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2 or np.std(y_true) == 0:
        return {"r2": np.nan, "mae": np.nan, "rmse": np.nan, "spearman": np.nan}
    rho = spearmanr(y_true, y_pred).correlation
    return {
        "r2": r2_score(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "spearman": float(rho) if rho is not None and not np.isnan(rho) else np.nan,
    }


# ------------------------------------------------------------------
# Verify the no-leakage guarantee on every organism task
# ------------------------------------------------------------------
leak_checks = []
for organism in MODELLED_ORGANISMS:
    task_id = f"organism__{organism}"
    for split_id in range(1, N_SPLITS_MAIN + 1):
        train_ids, test_ids = get_split_rows(task_id, split_id)
        train_scaffolds = set(rows_by_id(ml_ready, train_ids)["scaffold"])
        test_scaffolds = set(rows_by_id(ml_ready, test_ids)["scaffold"])

        group_name = ml_ready.loc[
            ml_ready["organism_name"] == organism, "organism_group"
        ].iloc[0]
        pool = aquatic[aquatic["organism_group"] == group_name]
        clean_pool, _ = pooled_training_rows(pool, test_ids)

        leak_checks.append({
            "organism_name": organism,
            "split_id": split_id,
            "train_rows": len(train_ids),
            "test_rows": len(test_ids),
            "train_scaffolds": len(train_scaffolds),
            "test_scaffolds": len(test_scaffolds),
            "own_split_scaffold_overlap": len(train_scaffolds & test_scaffolds),
            "pooled_rows_after_filter": len(clean_pool),
            "pooled_scaffold_overlap": len(set(clean_pool["scaffold"]) & test_scaffolds),
        })

leak_check_df = pd.DataFrame(leak_checks)
save_table(round_df(leak_check_df), step3_dir, "step3_leakage_checks.csv", max_rows=12)

assert leak_check_df["own_split_scaffold_overlap"].max() == 0, \
    "Scaffold leaked between train and test inside an organism split."
assert leak_check_df["pooled_scaffold_overlap"].max() == 0, \
    "Test scaffold survived in a pooled training set."
print("\nLeakage checks passed: zero scaffold overlap in every split and every pooled set.")

  saved: C:\Users\lab\Desktop\ecotoxicity\working\step3_splits\step3_leakage_checks.csv  (40 rows)


,organism_name,split_id,train_rows,test_rows,train_scaffolds,test_scaffolds,own_split_scaffold_overlap,pooled_rows_after_filter,pooled_scaffold_overlap
0,Danio rerio,1,2070,412,1319,330,0,6292,0
1,Danio rerio,2,2053,429,1319,330,0,6352,0
2,Danio rerio,3,2066,416,1319,330,0,6295,0
3,Danio rerio,4,2067,415,1319,330,0,6344,0
4,Danio rerio,5,1709,773,1319,330,0,5061,0
5,Daphnia magna,1,621,379,552,138,0,1234,0
6,Daphnia magna,2,850,150,552,138,0,1634,0
7,Daphnia magna,3,851,149,552,138,0,1593,0
8,Daphnia magna,4,632,368,552,138,0,1252,0
9,Daphnia magna,5,854,146,552,138,0,1639,0



Leakage checks passed: zero scaffold overlap in every split and every pooled set.


### Step 3C — Feature-set screening: do ~200 descriptors and smaller fingerprints help?

This is the empirical answer to suggestion 2. A single Random Forest (fixed, untuned settings so
the comparison is about the *features*, not the hyper-parameters) is run over every candidate
feature set on the shared scaffold splits.

The winner by mean Spearman becomes `MAIN_FEATURE_SET` for the rest of the notebook, so the later
steps automatically use whichever representation actually performs best on your data rather than
an assumption.

In [9]:
# ============================================================
# STEP 3C: FEATURE-SET SCREENING
# ============================================================
# One fixed Random Forest, many feature sets, identical splits.

SCREEN_RF = dict(
    n_estimators=300, min_samples_leaf=2, max_features="sqrt",
    random_state=RANDOM_STATE, n_jobs=-1,
)

screen_rows = []

with Timer("Step 3C: feature-set screening"):
    for organism in MODELLED_ORGANISMS:
        subset = aquatic[aquatic["organism_name"] == organism].reset_index(drop=True)
        task_id = f"organism__{organism}"

        for feature_set in FEATURE_SETS_TO_SCREEN:
            X_all = build_feature_matrix(subset, feature_set)
            # Map row_id -> position inside this organism's own matrix
            position = {int(r): i for i, r in enumerate(subset["row_id"].values)}
            y_all = subset["model_target"].values

            for split_id in range(1, N_SPLITS_MAIN + 1):
                train_ids, test_ids = get_split_rows(task_id, split_id)
                tr = [position[int(r)] for r in train_ids]
                te = [position[int(r)] for r in test_ids]

                model = RandomForestRegressor(**SCREEN_RF)
                model.fit(X_all.iloc[tr], y_all[tr])
                metrics = evaluate_predictions(y_all[te], model.predict(X_all.iloc[te]))

                screen_rows.append({
                    "organism_name": organism,
                    "organism_group": subset["organism_group"].iloc[0],
                    "feature_set": feature_set,
                    "n_features": X_all.shape[1],
                    "split_id": split_id,
                    **{f"test_{k}": v for k, v in metrics.items()},
                })
        print(f"  screened: {organism}")

feature_screen_df = pd.DataFrame(screen_rows)

feature_screen_summary = (
    feature_screen_df
    .groupby(["feature_set", "n_features"])
    .agg(
        mean_r2=("test_r2", "mean"),
        median_r2=("test_r2", "median"),
        mean_mae=("test_mae", "mean"),
        mean_rmse=("test_rmse", "mean"),
        mean_spearman=("test_spearman", "mean"),
        std_spearman=("test_spearman", "std"),
        n_organisms=("organism_name", "nunique"),
    )
    .reset_index()
    .sort_values("mean_spearman", ascending=False)
)

save_table(feature_screen_df, step3_dir, "step3_feature_screen_per_split.csv", show=False)
save_table(round_df(feature_screen_summary), step3_dir, "step3_feature_screen_summary.csv")

# Per-organism winner, useful for the report
feature_screen_by_organism = (
    feature_screen_df
    .groupby(["organism_name", "feature_set"])["test_spearman"].mean()
    .reset_index()
    .sort_values(["organism_name", "test_spearman"], ascending=[True, False])
)
save_table(round_df(feature_screen_by_organism), step3_dir,
           "step3_feature_screen_by_organism.csv", show=False)

# ------------------------------------------------------------------
# Adopt the winning feature set for the rest of the notebook
# ------------------------------------------------------------------
MAIN_FEATURE_SET = feature_screen_summary.iloc[0]["feature_set"]
print(f"\nWinning feature set: {MAIN_FEATURE_SET}")
print(f"  mean Spearman = {feature_screen_summary.iloc[0]['mean_spearman']:.4f}")
print(f"  mean R2       = {feature_screen_summary.iloc[0]['mean_r2']:.4f}")
print(f"  features      = {int(feature_screen_summary.iloc[0]['n_features'])}")
print("\nThis feature set is used by every model from Step 4 onwards.")

# Direct answers to the two questions in suggestion 2
basic_vs_ext = (
    feature_screen_summary.assign(
        descriptor_block=np.where(
            feature_screen_summary["feature_set"].str.startswith("ext"),
            "extended RDKit (~200)", "basic 11 (Phase 2)",
        )
    )
    .groupby("descriptor_block")[["mean_r2", "mean_spearman", "mean_mae"]].mean()
    .reset_index()
)
print("\n11 descriptors vs the extended RDKit set:")
display(round_df(basic_vs_ext))

fp_rows = feature_screen_summary[feature_screen_summary["feature_set"].str.contains("morgan")].copy()
if len(fp_rows):
    fp_rows["fingerprint_bits"] = fp_rows["feature_set"].str.extract(r"morgan(\d+)").astype(int)
    bits_comparison = (
        fp_rows.groupby("fingerprint_bits")[["mean_r2", "mean_spearman", "mean_mae"]]
        .mean().reset_index()
    )
    print("Fingerprint size comparison (tests the '1024 beats 2048' observation):")
    display(round_df(bits_comparison))
    save_table(round_df(bits_comparison), step3_dir, "step3_fingerprint_bits_comparison.csv",
               show=False)


>>> Step 3C: feature-set screening ...
  screened: Danio rerio
  screened: Daphnia magna
  screened: Desmodesmus subspicatus
  screened: Oncorhynchus mykiss
  screened: Oryzias latipes
  screened: Pimephales promelas
  screened: Raphidocelis subcapitata
  screened: Skeletonema costatum
<<< Step 3C: feature-set screening finished in 154.4 s
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step3_splits\step3_feature_screen_per_split.csv  (320 rows)
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step3_splits\step3_feature_screen_summary.csv  (8 rows)


,feature_set,n_features,mean_r2,median_r2,mean_mae,mean_rmse,mean_spearman,std_spearman,n_organisms
5,ext_morgan1024,1190,0.3739,0.3638,0.8169,1.0794,0.6305,0.0631,8
7,ext_morgan512,678,0.3766,0.3658,0.8125,1.0768,0.6303,0.0625,8
4,ext,166,0.3808,0.3685,0.8078,1.0727,0.6300,0.0632,8
6,ext_morgan2048,2214,0.3665,0.3603,0.8240,1.0862,0.6276,0.0642,8
3,basic11_morgan512,523,0.3498,0.3292,0.8378,1.0997,0.6155,0.0712,8
1,basic11_morgan1024,1035,0.3428,0.3285,0.8433,1.1061,0.6119,0.0711,8
2,basic11_morgan2048,2059,0.3307,0.3193,0.8542,1.1171,0.6089,0.0702,8
0,basic11,11,0.3272,0.3033,0.8457,1.1181,0.5907,0.0677,8


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step3_splits\step3_feature_screen_by_organism.csv  (64 rows)

Winning feature set: ext_morgan1024
  mean Spearman = 0.6305
  mean R2       = 0.3739
  features      = 1190

This feature set is used by every model from Step 4 onwards.

11 descriptors vs the extended RDKit set:


,descriptor_block,mean_r2,mean_spearman,mean_mae
0,basic 11 (Phase 2),0.3376,0.6068,0.8453
1,extended RDKit (~200),0.3744,0.6296,0.8153


Fingerprint size comparison (tests the '1024 beats 2048' observation):


,fingerprint_bits,mean_r2,mean_spearman,mean_mae
0,512,0.3632,0.6229,0.8252
1,1024,0.3583,0.6212,0.8301
2,2048,0.3486,0.6183,0.8391


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step3_splits\step3_fingerprint_bits_comparison.csv  (3 rows)


## Step 4 — Optuna hyper-parameter tuning, saved per organism and per group

*Addresses suggestion 1: "Optimize the current models using Optuna where feasible. Save the best
settings separately for each organism and biological-group model."*

**How the tuning is kept honest.** For each (task, model) pair, Optuna searches on the **inner
folds of the outer training set only** — the held-out scaffolds of the outer split are never seen
during tuning. This is the nested cross-validation asked for in suggestion 5.

To keep the runtime sane the search is run on outer split 1's training data and the winning
parameters are then applied to all outer splits. This is the standard compromise; set
`TUNE_EVERY_SPLIT = True` below for a fully nested search if you have the time budget.

**What is saved:** `step4_best_params.json` and `step4_best_params.csv` hold one parameter set per
(task, model), so you can report them in the thesis and reuse them without re-tuning.

In [10]:
# ============================================================
# STEP 4: OPTUNA TUNING (SAVED SEPARATELY PER TASK AND MODEL)
# ============================================================

step4_dir = OUTPUT_DIR / "step4_tuning"

TUNE_EVERY_SPLIT = False     # True = fully nested search on every outer split (much slower)

# Which models get tuned. Unavailable libraries are dropped automatically.
MODELS_TO_TUNE = ["RandomForest", "ExtraTrees", "HistGB"]
if AVAILABLE.get("xgboost"):
    MODELS_TO_TUNE.append("XGBoost")
if AVAILABLE.get("lightgbm"):
    MODELS_TO_TUNE.append("LightGBM")
if AVAILABLE.get("catboost"):
    MODELS_TO_TUNE.append("CatBoost")

print("Models to tune:", MODELS_TO_TUNE)


def suggest_params(trial, model_name):
    """The search space for each model family."""
    if model_name == "RandomForest":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
            "max_depth": trial.suggest_categorical("max_depth", [None, 10, 20, 30]),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 12),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3]),
        }
    if model_name == "ExtraTrees":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3]),
        }
    if model_name == "HistGB":
        return {
            "max_iter": trial.suggest_int("max_iter", 150, 600, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 63),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 40),
            "l2_regularization": trial.suggest_float("l2_regularization", 1e-3, 1.0, log=True),
        }
    if model_name == "XGBoost":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 900, step=100),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        }
    if model_name == "LightGBM":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 900, step=100),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 127),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        }
    if model_name == "CatBoost":
        return {
            "iterations": trial.suggest_int("iterations", 200, 900, step=100),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "depth": trial.suggest_int("depth", 4, 9),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        }
    raise ValueError(f"No search space defined for {model_name}")


def build_model(model_name, params=None):
    """Create a model instance from a parameter dictionary (empty = library defaults + seed)."""
    params = dict(params or {})

    if model_name == "MeanBaseline":
        return DummyRegressor(strategy="mean")
    if model_name == "MedianBaseline":
        return DummyRegressor(strategy="median")
    if model_name == "RandomForest":
        params.setdefault("n_estimators", 400)
        params.setdefault("min_samples_leaf", 2)
        params.setdefault("max_features", "sqrt")
        return RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
    if model_name == "ExtraTrees":
        params.setdefault("n_estimators", 400)
        params.setdefault("min_samples_leaf", 2)
        params.setdefault("max_features", "sqrt")
        return ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
    if model_name == "HistGB":
        params.setdefault("max_iter", 300)
        params.setdefault("learning_rate", 0.05)
        return HistGradientBoostingRegressor(random_state=RANDOM_STATE, **params)
    if model_name == "XGBoost":
        params.setdefault("n_estimators", 400)
        params.setdefault("max_depth", 5)
        params.setdefault("learning_rate", 0.03)
        return XGBRegressor(
            random_state=RANDOM_STATE, n_jobs=-1, objective="reg:squarederror",
            tree_method="hist", verbosity=0, **params
        )
    if model_name == "LightGBM":
        params.setdefault("n_estimators", 400)
        params.setdefault("learning_rate", 0.03)
        return LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1, **params)
    if model_name == "CatBoost":
        params.setdefault("iterations", 400)
        params.setdefault("learning_rate", 0.03)
        params.setdefault("depth", 6)
        return CatBoostRegressor(random_seed=RANDOM_STATE, verbose=False,
                                 allow_writing_files=False, **params)
    raise ValueError(f"Unknown model: {model_name}")

Models to tune: ['RandomForest', 'ExtraTrees', 'HistGB', 'XGBoost', 'LightGBM', 'CatBoost']


In [11]:
# ============================================================
# STEP 4 (continued): RUN THE SEARCHES
# ============================================================

def tune_one(task_data, task_id, model_name, split_id=1, n_trials=OPTUNA_TRIALS):
    """
    Optuna search for one (task, model), scored on inner scaffold folds of the
    outer training set. The outer test rows are never seen here.
    """
    train_ids, _ = get_split_rows(task_id, split_id)
    position = {int(r): i for i, r in enumerate(task_data["row_id"].values)}
    train_positions = [position[int(r)] for r in train_ids]
    train_data = task_data.iloc[train_positions].reset_index(drop=True)

    X_train = build_feature_matrix(train_data, MAIN_FEATURE_SET).values
    y_train = train_data["model_target"].values

    # Pre-compute the inner folds once so every trial is scored identically.
    folds = list(inner_folds(train_data))

    def objective(trial):
        params = suggest_params(trial, model_name)
        scores = []
        for inner_train, inner_valid in folds:
            model = build_model(model_name, params)
            model.fit(X_train[inner_train], y_train[inner_train])
            prediction = model.predict(X_train[inner_valid])
            scores.append(r2_score(y_train[inner_valid], prediction))
        return float(np.mean(scores))

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params, study.best_value, len(folds)


BEST_PARAMS = {}         # (task_id, model_name) -> parameter dict
tuning_rows = []

if not AVAILABLE.get("optuna"):
    print("Optuna is unavailable - every model will use the sensible defaults in build_model().")
else:
    with Timer("Step 4: Optuna tuning"):
        # ---- organism-level tasks ----
        for organism in MODELLED_ORGANISMS:
            task_data = aquatic[aquatic["organism_name"] == organism].reset_index(drop=True)
            task_id = f"organism__{organism}"
            for model_name in MODELS_TO_TUNE:
                t0 = time.time()
                try:
                    params, score, n_folds = tune_one(task_data, task_id, model_name)
                except Exception as exc:
                    print(f"  tuning failed for {organism} / {model_name}: {exc}")
                    continue
                BEST_PARAMS[(task_id, model_name)] = params
                tuning_rows.append({
                    "task_id": task_id, "entity_type": "organism", "entity_name": organism,
                    "model": model_name, "inner_folds": n_folds,
                    "best_inner_r2": score, "seconds": round(time.time() - t0, 1),
                    "best_params": json.dumps(params),
                })
            print(f"  tuned: {organism}")

        # ---- biological-group tasks ----
        for grp, unit in MODELLED_GROUPS:
            task_data = ml_ready[
                (ml_ready["organism_group"] == grp) & (ml_ready["unit"] == unit)
            ].reset_index(drop=True)
            task_id = f"group__{grp}__{unit}"
            for model_name in MODELS_TO_TUNE:
                t0 = time.time()
                try:
                    params, score, n_folds = tune_one(task_data, task_id, model_name)
                except Exception as exc:
                    print(f"  tuning failed for {grp}/{unit} / {model_name}: {exc}")
                    continue
                BEST_PARAMS[(task_id, model_name)] = params
                tuning_rows.append({
                    "task_id": task_id, "entity_type": "group", "entity_name": f"{grp} | {unit}",
                    "model": model_name, "inner_folds": n_folds,
                    "best_inner_r2": score, "seconds": round(time.time() - t0, 1),
                    "best_params": json.dumps(params),
                })
            print(f"  tuned: {grp} | {unit}")

tuning_df = pd.DataFrame(tuning_rows)

if len(tuning_df):
    save_table(round_df(tuning_df), step4_dir, "step4_best_params.csv", max_rows=25)

    # JSON version: easy to reload in a later session without re-tuning
    step4_dir.mkdir(parents=True, exist_ok=True)
    with open(step4_dir / "step4_best_params.json", "w") as handle:
        json.dump(
            {f"{task}||{model}": params for (task, model), params in BEST_PARAMS.items()},
            handle, indent=2,
        )
    print(f"\nSaved {len(BEST_PARAMS)} tuned parameter sets.")

    best_model_per_task = (
        tuning_df.sort_values(["task_id", "best_inner_r2"], ascending=[True, False])
        .groupby("task_id").head(1).reset_index(drop=True)
    )
    print("\nBest model family per task (by inner-fold R2):")
    display(round_df(best_model_per_task[
        ["entity_type", "entity_name", "model", "best_inner_r2", "seconds"]
    ]))
    save_table(round_df(best_model_per_task), step4_dir, "step4_best_model_per_task.csv",
               show=False)


def tuned_params(task_id, model_name):
    """Tuned parameters for a task/model, or {} to fall back on the defaults."""
    return BEST_PARAMS.get((task_id, model_name), {})


>>> Step 4: Optuna tuning ...
  tuned: Danio rerio
  tuned: Daphnia magna
  tuned: Desmodesmus subspicatus
  tuned: Oncorhynchus mykiss
  tuned: Oryzias latipes
  tuned: Pimephales promelas
  tuned: Raphidocelis subcapitata
  tuned: Skeletonema costatum
  tuned: Algae | mg/L
  tuned: Fish | mg/L
  tuned: Invertebrates | mg/L
<<< Step 4: Optuna tuning finished in 24621.4 s
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step4_tuning\step4_best_params.csv  (66 rows)


,task_id,entity_type,entity_name,model,inner_folds,best_inner_r2,seconds,best_params
0,organism__Danio rerio,organism,Danio rerio,RandomForest,3,0.3075,87.6,"{""n_estimators"": 800, ""max_depth"": 30, ""min_sa..."
1,organism__Danio rerio,organism,Danio rerio,ExtraTrees,3,0.2980,113.1,"{""n_estimators"": 800, ""min_samples_leaf"": 2, ""..."
2,organism__Danio rerio,organism,Danio rerio,HistGB,3,0.2837,1379.6,"{""max_iter"": 550, ""learning_rate"": 0.010525511..."
3,organism__Danio rerio,organism,Danio rerio,XGBoost,3,0.3060,247.8,"{""n_estimators"": 600, ""max_depth"": 10, ""learni..."
4,organism__Danio rerio,organism,Danio rerio,LightGBM,3,0.2971,143.7,"{""n_estimators"": 500, ""learning_rate"": 0.01077..."
5,organism__Danio rerio,organism,Danio rerio,CatBoost,3,0.3129,1162.7,"{""iterations"": 800, ""learning_rate"": 0.0385232..."
6,organism__Daphnia magna,organism,Daphnia magna,RandomForest,3,0.3766,65.4,"{""n_estimators"": 800, ""max_depth"": 20, ""min_sa..."
7,organism__Daphnia magna,organism,Daphnia magna,ExtraTrees,3,0.3177,37.4,"{""n_estimators"": 300, ""min_samples_leaf"": 4, ""..."
8,organism__Daphnia magna,organism,Daphnia magna,HistGB,3,0.3423,280.4,"{""max_iter"": 150, ""learning_rate"": 0.018362365..."
9,organism__Daphnia magna,organism,Daphnia magna,XGBoost,3,0.3769,131.0,"{""n_estimators"": 700, ""max_depth"": 6, ""learnin..."



Saved 66 tuned parameter sets.

Best model family per task (by inner-fold R2):


,entity_type,entity_name,model,best_inner_r2,seconds
0,group,Algae | mg/L,XGBoost,0.3108,388.2
1,group,Fish | mg/L,CatBoost,0.3997,1628.7
2,group,Invertebrates | mg/L,XGBoost,0.4126,153.7
3,organism,Danio rerio,CatBoost,0.3129,1162.7
4,organism,Daphnia magna,XGBoost,0.3769,131.0
5,organism,Desmodesmus subspicatus,RandomForest,0.2809,68.1
6,organism,Oncorhynchus mykiss,ExtraTrees,0.3848,79.8
7,organism,Oryzias latipes,LightGBM,0.3586,33.7
8,organism,Pimephales promelas,CatBoost,0.4095,669.0
9,organism,Raphidocelis subcapitata,CatBoost,0.2921,695.5


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step4_tuning\step4_best_model_per_task.csv  (11 rows)


## Step 5 — Species-aware pooled models (the central Phase 3 experiment)

*Addresses suggestion 3: "test species-aware pooled models separately within each biological
group… Add organism identity to molecular features and evaluate Random Forest, XGBoost, and
CatBoost against the existing independent and group-level models using identical scaffold-based
validation splits. Assess whether species-aware modeling improves prediction accuracy,
particularly for organisms with limited training data."*

**The idea.** A pooled model trained on a whole biological group sees far more chemistry than a
single-organism model, but it cannot tell *Danio rerio* from *Oncorhynchus mykiss* and so predicts
the group average. Appending **organism identity** to the molecular features lets one pooled model
learn both the shared chemistry and the per-species offset. This is the tabular equivalent of the
species-conditioning used in multimodal ecotoxicity models such as AMTox-L (Yin & Li, *J. Hazard.
Mater.*, 2026), which conditions a shared chemical encoder on taxonomic identity.

**Five model types, evaluated on identical test rows**

| Model type | Training data | Knows which species? |
|---|---|---|
| `independent_organism` | that organism only | n/a (single species) |
| `group_pooled` | whole biological group | no |
| `species_aware_pooled` | whole biological group | **yes** — organism one-hot appended |
| `hierarchical_residual` | group model + organism residual correction | implicitly |
| *(Step 6)* `multitask_nn` | all species at once, one head each | yes |

Every one of them is scored on **exactly the same held-out rows** — organism *O*'s test rows from
the fixed scaffold split — and every pooled variant has *O*'s test scaffolds stripped from its
training pool first. The comparison therefore isolates the effect of the modelling strategy.

The summary at the end breaks the result down by dataset size, which answers the specific question
of whether species-aware pooling rescues the **organisms with limited training data**.

In [12]:
# ============================================================
# STEP 5: SPECIES-AWARE POOLED MODELS VS THE PHASE-2 BASELINES
# ============================================================

step5_dir = OUTPUT_DIR / "step5_species_aware"

# Suggestion 3 names these three families explicitly.
SPECIES_AWARE_MODELS = ["RandomForest"]
if AVAILABLE.get("xgboost"):
    SPECIES_AWARE_MODELS.append("XGBoost")
if AVAILABLE.get("catboost"):
    SPECIES_AWARE_MODELS.append("CatBoost")
print("Model families compared:", SPECIES_AWARE_MODELS)

# Fixed organism vocabulary per biological group, so the one-hot block has stable columns.
GROUP_ORGANISMS = {
    grp: sorted(aquatic.loc[aquatic["organism_group"] == grp, "organism_name"].unique())
    for grp in aquatic["organism_group"].unique()
}


def organism_identity_frame(data, organism_vocabulary):
    """
    Encode organism identity.

    A one-hot indicator per species, plus log10(training rows for that species) as a
    capacity hint that lets the model lean harder on the group signal for rare species.
    """
    one_hot = pd.DataFrame(
        0.0,
        index=range(len(data)),
        columns=[f"species__{name}" for name in organism_vocabulary],
    )
    names = data["organism_name"].values
    for i, name in enumerate(names):
        column = f"species__{name}"
        if column in one_hot.columns:
            one_hot.iat[i, one_hot.columns.get_loc(column)] = 1.0

    counts = data["organism_name"].map(
        aquatic.groupby("organism_name").size()
    ).fillna(1).values
    one_hot["species__log_n_records"] = np.log10(counts.astype(float) + 1.0)
    return one_hot.astype(np.float32)


def build_species_aware_matrix(data, organism_vocabulary, feature_set=None):
    """Molecular features with the organism-identity block appended."""
    feature_set = feature_set or MAIN_FEATURE_SET
    chemistry = build_feature_matrix(data, feature_set)
    identity = organism_identity_frame(data, organism_vocabulary)
    return pd.concat([chemistry, identity], axis=1)


comparison_rows = []
# Out-of-fold style store: per-row predictions from every strategy, reused by Steps 7-9.
prediction_store = []

with Timer("Step 5: species-aware vs independent vs pooled vs hierarchical"):
    for organism in MODELLED_ORGANISMS:
        organism_data = aquatic[aquatic["organism_name"] == organism].reset_index(drop=True)
        group_name = organism_data["organism_group"].iloc[0]
        group_pool_full = aquatic[aquatic["organism_group"] == group_name].reset_index(drop=True)
        vocabulary = GROUP_ORGANISMS[group_name]
        task_id = f"organism__{organism}"
        group_task_id = f"group__{group_name}__mg/L"

        organism_position = {int(r): i for i, r in enumerate(organism_data["row_id"].values)}
        X_organism_all = build_feature_matrix(organism_data, MAIN_FEATURE_SET)
        y_organism_all = organism_data["model_target"].values

        print(f"\n  {organism}  (group: {group_name}, "
              f"{len(organism_data)} rows, pool {len(group_pool_full)} rows)")

        for split_id in range(1, N_SPLITS_MAIN + 1):
            train_ids, test_ids = get_split_rows(task_id, split_id)
            tr = [organism_position[int(r)] for r in train_ids]
            te = [organism_position[int(r)] for r in test_ids]

            X_org_train, X_org_test = X_organism_all.iloc[tr], X_organism_all.iloc[te]
            y_org_train, y_org_test = y_organism_all[tr], y_organism_all[te]
            test_rows = organism_data.iloc[te].reset_index(drop=True)

            # Leakage-free pooled training set for this split.
            pool_train, test_scaffolds = pooled_training_rows(group_pool_full, test_ids)
            y_pool_train = pool_train["model_target"].values

            # Feature matrices for the pooled variants.
            X_pool_plain = build_feature_matrix(pool_train, MAIN_FEATURE_SET)
            X_pool_aware = build_species_aware_matrix(pool_train, vocabulary)
            X_test_aware = build_species_aware_matrix(test_rows, vocabulary)

            for model_name in SPECIES_AWARE_MODELS:
                organism_params = tuned_params(task_id, model_name)
                group_params = tuned_params(group_task_id, model_name)

                predictions = {}

                # 1) Independent organism model (the Phase 2 baseline)
                independent = build_model(model_name, organism_params)
                independent.fit(X_org_train, y_org_train)
                predictions["independent_organism"] = independent.predict(X_org_test)

                # 2) Group-pooled model, species-blind
                pooled = build_model(model_name, group_params)
                pooled.fit(X_pool_plain, y_pool_train)
                predictions["group_pooled"] = pooled.predict(X_org_test)

                # 3) Species-aware pooled model  <-- the Phase 3 proposal
                aware = build_model(model_name, group_params)
                aware.fit(X_pool_aware, y_pool_train)
                predictions["species_aware_pooled"] = aware.predict(X_test_aware)

                # 4) Hierarchical residual model (Phase 2 approach, same splits)
                group_prediction_on_org_train = pooled.predict(X_org_train)
                residual_target = y_org_train - group_prediction_on_org_train
                residual_model = build_model(model_name, organism_params)
                residual_model.fit(X_org_train, residual_target)
                predictions["hierarchical_residual"] = (
                    predictions["group_pooled"] + residual_model.predict(X_org_test)
                )

                for strategy, y_pred in predictions.items():
                    metrics = evaluate_predictions(y_org_test, y_pred)
                    comparison_rows.append({
                        "organism_group": group_name,
                        "organism_name": organism,
                        "dataset_size_category": organism_data["dataset_size_category"].iloc[0],
                        "organism_rows": len(organism_data),
                        "model": model_name,
                        "strategy": strategy,
                        "feature_set": MAIN_FEATURE_SET,
                        "split_id": split_id,
                        "organism_train_rows": len(tr),
                        "pooled_train_rows": len(pool_train),
                        "test_rows": len(te),
                        "test_scaffolds": test_rows["scaffold"].nunique(),
                        "pooled_scaffold_overlap": len(
                            set(pool_train["scaffold"]) & test_scaffolds
                        ),
                        **{f"test_{k}": v for k, v in metrics.items()},
                    })

                    # Keep every per-row prediction: Step 7 builds bootstrap confidence
                    # intervals from these, and Steps 9-10 reuse them for the residual analysis.
                    for i in range(len(te)):
                        prediction_store.append({
                            "organism_group": group_name,
                            "organism_name": organism,
                            "strategy": strategy,
                            "model": model_name,
                            "split_id": split_id,
                            "row_id": int(test_rows.loc[i, "row_id"]),
                            "canonical_smiles": test_rows.loc[i, "canonical_smiles"],
                            "observed": float(y_org_test[i]),
                            "predicted": float(y_pred[i]),
                        })

            print(f"    split {split_id} done", end="\r")

comparison_df = pd.DataFrame(comparison_rows)
predictions_df = pd.DataFrame(prediction_store)

save_table(comparison_df, step5_dir, "step5_strategy_comparison_per_split.csv", show=False)
save_table(predictions_df, step5_dir, "step5_row_level_predictions.csv", show=False)
print(f"\nCollected {len(comparison_df)} evaluation records.")

Model families compared: ['RandomForest', 'XGBoost', 'CatBoost']

>>> Step 5: species-aware vs independent vs pooled vs hierarchical ...

  Danio rerio  (group: Fish, 2482 rows, pool 7057 rows)
    split 5 done
  Daphnia magna  (group: Invertebrates, 1000 rows, pool 1837 rows)
    split 5 done
  Desmodesmus subspicatus  (group: Algae, 1612 rows, pool 5699 rows)
    split 5 done
  Oncorhynchus mykiss  (group: Fish, 2132 rows, pool 7057 rows)
    split 5 done
  Oryzias latipes  (group: Fish, 862 rows, pool 7057 rows)
    split 5 done
  Pimephales promelas  (group: Fish, 1581 rows, pool 7057 rows)
    split 5 done
  Raphidocelis subcapitata  (group: Algae, 3165 rows, pool 5699 rows)
    split 5 done
  Skeletonema costatum  (group: Algae, 514 rows, pool 5699 rows)
<<< Step 5: species-aware vs independent vs pooled vs hierarchical finished in 4492.8 s
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step5_species_aware\step5_strategy_comparison_per_split.csv  (480 rows)
  saved: C:\Users\l

In [13]:
# ============================================================
# STEP 5 (continued): DID SPECIES-AWARE POOLING ACTUALLY HELP?
# ============================================================

# ---- per organism x model x strategy ----
strategy_summary = (
    comparison_df
    .groupby(["organism_group", "organism_name", "dataset_size_category",
              "organism_rows", "model", "strategy"])
    .agg(
        r2_mean=("test_r2", "mean"), r2_std=("test_r2", "std"),
        mae_mean=("test_mae", "mean"), rmse_mean=("test_rmse", "mean"),
        spearman_mean=("test_spearman", "mean"), spearman_std=("test_spearman", "std"),
        n_splits=("split_id", "nunique"),
    )
    .reset_index()
)
save_table(round_df(strategy_summary), step5_dir, "step5_strategy_summary_by_organism.csv",
           show=False)

# ---- overall ranking of the four strategies ----
overall_strategy = (
    strategy_summary
    .groupby(["model", "strategy"])
    .agg(
        mean_r2=("r2_mean", "mean"), median_r2=("r2_mean", "median"),
        mean_mae=("mae_mean", "mean"), mean_rmse=("rmse_mean", "mean"),
        mean_spearman=("spearman_mean", "mean"),
        n_organisms=("organism_name", "nunique"),
    )
    .reset_index()
    .sort_values(["model", "mean_r2"], ascending=[True, False])
)
print("Overall strategy comparison (mean across organisms):")
display(round_df(overall_strategy))
save_table(round_df(overall_strategy), step5_dir, "step5_overall_strategy_comparison.csv",
           show=False)

# ---- the key question: does it help small datasets most? ----
by_size = (
    strategy_summary
    .groupby(["dataset_size_category", "strategy"])
    .agg(mean_r2=("r2_mean", "mean"), mean_spearman=("spearman_mean", "mean"),
         mean_mae=("mae_mean", "mean"), n_organisms=("organism_name", "nunique"))
    .reset_index()
    .sort_values(["dataset_size_category", "mean_r2"], ascending=[True, False])
)
print("\nStrategy performance by dataset size "
      "(suggestion 3 asks specifically about organisms with limited data):")
display(round_df(by_size))
save_table(round_df(by_size), step5_dir, "step5_strategy_by_dataset_size.csv", show=False)

# ---- paired test: species-aware vs independent, per organism ----
paired_rows = []
for model_name in comparison_df["model"].unique():
    subset = comparison_df[comparison_df["model"] == model_name]
    wide = subset.pivot_table(
        index=["organism_name", "split_id"], columns="strategy", values="test_r2"
    ).dropna()
    if {"species_aware_pooled", "independent_organism"}.issubset(wide.columns):
        difference = wide["species_aware_pooled"] - wide["independent_organism"]
        try:
            _, p_value = wilcoxon(difference)
        except Exception:
            p_value = np.nan
        paired_rows.append({
            "model": model_name,
            "comparison": "species_aware_pooled - independent_organism",
            "n_paired_splits": len(difference),
            "mean_delta_r2": difference.mean(),
            "median_delta_r2": difference.median(),
            "pct_splits_improved": 100.0 * (difference > 0).mean(),
            "wilcoxon_p": p_value,
        })
    if {"species_aware_pooled", "group_pooled"}.issubset(wide.columns):
        difference = wide["species_aware_pooled"] - wide["group_pooled"]
        try:
            _, p_value = wilcoxon(difference)
        except Exception:
            p_value = np.nan
        paired_rows.append({
            "model": model_name,
            "comparison": "species_aware_pooled - group_pooled",
            "n_paired_splits": len(difference),
            "mean_delta_r2": difference.mean(),
            "median_delta_r2": difference.median(),
            "pct_splits_improved": 100.0 * (difference > 0).mean(),
            "wilcoxon_p": p_value,
        })
    if {"species_aware_pooled", "hierarchical_residual"}.issubset(wide.columns):
        difference = wide["species_aware_pooled"] - wide["hierarchical_residual"]
        try:
            _, p_value = wilcoxon(difference)
        except Exception:
            p_value = np.nan
        paired_rows.append({
            "model": model_name,
            "comparison": "species_aware_pooled - hierarchical_residual",
            "n_paired_splits": len(difference),
            "mean_delta_r2": difference.mean(),
            "median_delta_r2": difference.median(),
            "pct_splits_improved": 100.0 * (difference > 0).mean(),
            "wilcoxon_p": p_value,
        })

paired_tests_df = pd.DataFrame(paired_rows)
print("\nPaired comparisons across matched splits (Wilcoxon signed-rank):")
display(round_df(paired_tests_df))
save_table(round_df(paired_tests_df), step5_dir, "step5_paired_strategy_tests.csv", show=False)

# ---- plain-language verdict ----
if len(paired_tests_df):
    headline = paired_tests_df[
        paired_tests_df["comparison"] == "species_aware_pooled - independent_organism"
    ]
    if len(headline):
        delta = headline["mean_delta_r2"].mean()
        improved = headline["pct_splits_improved"].mean()
        print("\n" + "=" * 70)
        print("VERDICT - species-aware pooling vs independent organism models")
        print(f"  mean change in R2 : {delta:+.4f}")
        print(f"  splits improved   : {improved:.1f}%")
        if delta > 0.01:
            print("  -> Species-aware pooling improves accuracy. Keep it as the main model.")
        elif delta > -0.01:
            print("  -> Roughly equivalent. Prefer the simpler independent models, "
                  "or keep species-aware pooling for the small-data organisms only.")
        else:
            print("  -> Species-aware pooling does not help on this data. Report it as a "
                  "negative result; the independent models remain the baseline.")
        print("=" * 70)

  saved: C:\Users\lab\Desktop\ecotoxicity\working\step5_species_aware\step5_strategy_summary_by_organism.csv  (96 rows)
Overall strategy comparison (mean across organisms):


,model,strategy,mean_r2,median_r2,mean_mae,mean_rmse,mean_spearman,n_organisms
1,CatBoost,hierarchical_residual,0.4052,0.3926,0.7827,1.0490,0.6532,8
3,CatBoost,species_aware_pooled,0.4028,0.3897,0.7913,1.0512,0.6515,8
2,CatBoost,independent_organism,0.3878,0.3734,0.8005,1.0660,0.6344,8
0,CatBoost,group_pooled,0.3693,0.3388,0.8219,1.0841,0.6408,8
5,RandomForest,hierarchical_residual,0.4073,0.3959,0.7837,1.0492,0.6524,8
7,RandomForest,species_aware_pooled,0.3951,0.3724,0.8013,1.0614,0.6488,8
6,RandomForest,independent_organism,0.3898,0.3754,0.7961,1.0654,0.6385,8
4,RandomForest,group_pooled,0.3686,0.3482,0.8231,1.0865,0.6378,8
11,XGBoost,species_aware_pooled,0.4065,0.3938,0.7897,1.0502,0.6516,8
9,XGBoost,hierarchical_residual,0.3999,0.3856,0.7857,1.0557,0.6492,8


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step5_species_aware\step5_overall_strategy_comparison.csv  (12 rows)

Strategy performance by dataset size (suggestion 3 asks specifically about organisms with limited data):


,dataset_size_category,strategy,mean_r2,mean_spearman,mean_mae,n_organisms
3,medium,species_aware_pooled,0.4110,0.6722,0.9004,2
1,medium,hierarchical_residual,0.4012,0.6678,0.8917,2
2,medium,independent_organism,0.3994,0.6510,0.8949,2
0,medium,group_pooled,0.3583,0.6458,0.9560,2
5,strong,hierarchical_residual,0.4051,0.6462,0.7481,6
7,strong,species_aware_pooled,0.3983,0.6435,0.7587,6
6,strong,independent_organism,0.3854,0.6308,0.7647,6
4,strong,group_pooled,0.3735,0.6367,0.7763,6


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step5_species_aware\step5_strategy_by_dataset_size.csv  (8 rows)

Paired comparisons across matched splits (Wilcoxon signed-rank):


,model,comparison,n_paired_splits,mean_delta_r2,median_delta_r2,pct_splits_improved,wilcoxon_p
0,RandomForest,species_aware_pooled - independent_organism,40,0.0053,0.0036,60.0,0.2424
1,RandomForest,species_aware_pooled - group_pooled,40,0.0265,0.0205,90.0,0.0000
2,RandomForest,species_aware_pooled - hierarchical_residual,40,-0.0122,-0.0140,17.5,0.0000
3,XGBoost,species_aware_pooled - independent_organism,40,0.0173,0.0133,82.5,0.0000
4,XGBoost,species_aware_pooled - group_pooled,40,0.0352,0.0262,95.0,0.0000
5,XGBoost,species_aware_pooled - hierarchical_residual,40,0.0067,0.0017,60.0,0.2948
6,CatBoost,species_aware_pooled - independent_organism,40,0.0150,0.0143,67.5,0.0014
7,CatBoost,species_aware_pooled - group_pooled,40,0.0335,0.0269,87.5,0.0000
8,CatBoost,species_aware_pooled - hierarchical_residual,40,-0.0024,-0.0063,42.5,0.1701


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step5_species_aware\step5_paired_strategy_tests.csv  (9 rows)

VERDICT - species-aware pooling vs independent organism models
  mean change in R2 : +0.0125
  splits improved   : 70.0%
  -> Species-aware pooling improves accuracy. Keep it as the main model.


## Step 6 — Multitask neural network (shared layers, species heads, masked loss)

*Addresses suggestion 4: "test a multitask neural network with shared layers, species-specific
outputs, masked loss, and early stopping. Keep it only if it clearly improves validation
performance. Since benchmarking with other models also require a neural network baseline as well."*

**The architecture**

```
           molecular features (descriptors + Morgan bits)
                            |
                    shared trunk: 512 -> 256
                  (BatchNorm + ReLU + Dropout)
                            |
        +-----------+-------+-------+-----------+
        |           |               |           |
   head: species1  head: species2  ...     head: speciesN     (one linear head each)
```

**Why a masked loss is required.** Most chemicals were not tested on every organism, so the
species × chemical target matrix is mostly empty. For each chemical the loss is computed **only
over the species that actually have a measurement**; missing entries contribute nothing and
produce no gradient. Without the mask the network would learn to predict zeros for untested pairs.

**How it is kept comparable and leak-free.** The network is trained per organism-split: it sees
every chemical whose scaffold is *not* in that organism's held-out set, and is then scored on
exactly the same test rows as every model in Step 5. An internal scaffold-grouped validation set
drives early stopping, so the test rows play no part in choosing when to stop.

Because this is the slowest model, `MTL_MAX_SPLITS` (Step 0) limits how many outer splits it runs
on. The cell is skipped automatically if PyTorch is unavailable.

In [14]:
# ============================================================
# STEP 6: MULTITASK NEURAL NETWORK
# ============================================================

step6_dir = OUTPUT_DIR / "step6_multitask_nn"
multitask_rows = []
multitask_predictions = []

if not AVAILABLE.get("torch"):
    print("PyTorch is not available in this environment - Step 6 is skipped.")
    print("The classical models in Step 5 remain the full comparison.")
else:
    from torch.utils.data import TensorDataset, DataLoader

    # ------------------------------------------------------------------
    # 6.1 Model definition
    # ------------------------------------------------------------------
    class MultitaskNet(nn.Module):
        """Shared trunk with one linear output head per species."""

        def __init__(self, n_features, n_species, hidden=(512, 256), dropout=0.3):
            super().__init__()
            layers, in_dim = [], n_features
            for width in hidden:
                layers += [
                    nn.Linear(in_dim, width),
                    nn.BatchNorm1d(width),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                ]
                in_dim = width
            self.trunk = nn.Sequential(*layers)
            # A separate head per species = "species-specific outputs".
            self.heads = nn.ModuleList([nn.Linear(in_dim, 1) for _ in range(n_species)])

        def forward(self, x):
            shared = self.trunk(x)
            return torch.cat([head(shared) for head in self.heads], dim=1)


    def masked_mse(prediction, target, mask):
        """Mean squared error over observed (chemical, species) pairs only."""
        n_observed = mask.sum()
        if n_observed == 0:
            return torch.tensor(0.0, device=prediction.device, requires_grad=True)
        squared_error = ((prediction - target) ** 2) * mask
        return squared_error.sum() / n_observed


    # ------------------------------------------------------------------
    # 6.2 Build the chemical x species target matrix
    # ------------------------------------------------------------------
    MTL_SPECIES = MODELLED_ORGANISMS[:]             # one head per modelled organism
    SPECIES_INDEX = {name: i for i, name in enumerate(MTL_SPECIES)}

    mtl_source = aquatic[aquatic["organism_name"].isin(MTL_SPECIES)].reset_index(drop=True)

    # One row per unique chemical; the target matrix holds NaN where untested.
    mtl_chemicals = (
        mtl_source[["canonical_smiles", "scaffold"]]
        .drop_duplicates(subset="canonical_smiles")
        .reset_index(drop=True)
    )
    chemical_position = {s: i for i, s in enumerate(mtl_chemicals["canonical_smiles"])}

    Y_matrix = np.full((len(mtl_chemicals), len(MTL_SPECIES)), np.nan, dtype=np.float32)
    for smiles, organism, value in zip(
        mtl_source["canonical_smiles"], mtl_source["organism_name"], mtl_source["model_target"]
    ):
        Y_matrix[chemical_position[smiles], SPECIES_INDEX[organism]] = value

    observed_mask = ~np.isnan(Y_matrix)
    print(f"Multitask matrix: {Y_matrix.shape[0]} chemicals x {Y_matrix.shape[1]} species")
    print(f"Observed entries: {observed_mask.sum()} "
          f"({100 * observed_mask.mean():.1f}% of the matrix - hence the masked loss)")

    # Chemistry features, computed once for all chemicals.
    X_mtl_all = build_feature_matrix(mtl_chemicals, MAIN_FEATURE_SET).values.astype(np.float32)

    save_table(
        pd.DataFrame({
            "species": MTL_SPECIES,
            "observed_chemicals": observed_mask.sum(axis=0),
        }),
        step6_dir, "step6_multitask_matrix_coverage.csv",
    )

Multitask matrix: 7832 chemicals x 8 species
Observed entries: 13346 (21.3% of the matrix - hence the masked loss)
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step6_multitask_nn\step6_multitask_matrix_coverage.csv  (8 rows)


,species,observed_chemicals
0,Danio rerio,2482
1,Daphnia magna,1000
2,Desmodesmus subspicatus,1612
3,Oncorhynchus mykiss,2130
4,Oryzias latipes,862
5,Pimephales promelas,1581
6,Raphidocelis subcapitata,3165
7,Skeletonema costatum,514


In [15]:
# ============================================================
# STEP 6 (continued): TRAIN AND EVALUATE
# ============================================================

if AVAILABLE.get("torch"):

    def train_multitask(X_train, Y_train, M_train, X_valid, Y_valid, M_valid,
                        n_species, max_epochs=200, patience=20, batch_size=64,
                        learning_rate=1e-3, weight_decay=1e-5, verbose=False):
        """Train with masked MSE and early stopping on the validation mask."""
        torch.manual_seed(RANDOM_STATE)
        np.random.seed(RANDOM_STATE)

        # Standardise the inputs using TRAINING statistics only.
        mean = X_train.mean(axis=0, keepdims=True)
        std = X_train.std(axis=0, keepdims=True)
        std[std < 1e-8] = 1.0
        X_train_scaled = (X_train - mean) / std
        X_valid_scaled = (X_valid - mean) / std

        # Targets are standardised too, which makes the masked loss well-scaled.
        observed_values = Y_train[M_train]
        target_mean = float(observed_values.mean())
        target_std = float(observed_values.std()) or 1.0
        Y_train_scaled = np.nan_to_num((Y_train - target_mean) / target_std)
        Y_valid_scaled = np.nan_to_num((Y_valid - target_mean) / target_std)

        model = MultitaskNet(X_train.shape[1], n_species).to(DEVICE)
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=learning_rate, weight_decay=weight_decay
        )

        train_loader = DataLoader(
            TensorDataset(
                torch.tensor(X_train_scaled), torch.tensor(Y_train_scaled),
                torch.tensor(M_train.astype(np.float32)),
            ),
            batch_size=batch_size, shuffle=True, drop_last=len(X_train_scaled) > batch_size,
        )

        X_valid_tensor = torch.tensor(X_valid_scaled).to(DEVICE)
        Y_valid_tensor = torch.tensor(Y_valid_scaled).to(DEVICE)
        M_valid_tensor = torch.tensor(M_valid.astype(np.float32)).to(DEVICE)

        best_loss, best_state, epochs_without_improvement = np.inf, None, 0

        for epoch in range(max_epochs):
            model.train()
            for xb, yb, mb in train_loader:
                xb, yb, mb = xb.to(DEVICE), yb.to(DEVICE), mb.to(DEVICE)
                optimizer.zero_grad()
                loss = masked_mse(model(xb), yb, mb)
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.no_grad():
                validation_loss = float(
                    masked_mse(model(X_valid_tensor), Y_valid_tensor, M_valid_tensor)
                )

            if validation_loss < best_loss - 1e-5:
                best_loss = validation_loss
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= patience:
                    if verbose:
                        print(f"      early stop at epoch {epoch + 1}")
                    break

        if best_state is not None:
            model.load_state_dict(best_state)
        model.eval()

        def predict(X_new):
            X_scaled = (X_new - mean) / std
            with torch.no_grad():
                output = model(torch.tensor(X_scaled.astype(np.float32)).to(DEVICE))
            return output.cpu().numpy() * target_std + target_mean

        return predict, best_loss, epoch + 1


    with Timer(f"Step 6: multitask network ({MTL_MAX_SPLITS} split(s) per organism)"):
        for organism in MODELLED_ORGANISMS:
            organism_data = aquatic[aquatic["organism_name"] == organism].reset_index(drop=True)
            organism_position = {int(r): i for i, r in enumerate(organism_data["row_id"].values)}
            task_id = f"organism__{organism}"
            head_index = SPECIES_INDEX[organism]

            for split_id in range(1, min(MTL_MAX_SPLITS, N_SPLITS_MAIN) + 1):
                train_ids, test_ids = get_split_rows(task_id, split_id)
                te = [organism_position[int(r)] for r in test_ids]
                test_rows = organism_data.iloc[te].reset_index(drop=True)
                y_test = organism_data["model_target"].values[te]
                test_scaffolds = set(test_rows["scaffold"])

                # Chemicals usable for training: scaffold not held out.
                trainable = ~mtl_chemicals["scaffold"].isin(test_scaffolds).values
                train_positions = np.where(trainable)[0]

                # Inner scaffold-grouped validation split for early stopping.
                inner_splitter = GroupShuffleSplit(
                    n_splits=1, test_size=0.15, random_state=RANDOM_STATE
                )
                inner_train, inner_valid = next(inner_splitter.split(
                    np.zeros(len(train_positions)),
                    groups=mtl_chemicals.iloc[train_positions]["scaffold"].values,
                ))
                fit_positions = train_positions[inner_train]
                valid_positions = train_positions[inner_valid]

                predict_fn, validation_loss, epochs_run = train_multitask(
                    X_mtl_all[fit_positions], Y_matrix[fit_positions], observed_mask[fit_positions],
                    X_mtl_all[valid_positions], Y_matrix[valid_positions],
                    observed_mask[valid_positions],
                    n_species=len(MTL_SPECIES),
                )

                # Predict this organism's head for the held-out test chemicals.
                test_positions = [
                    chemical_position[s] for s in test_rows["canonical_smiles"]
                ]
                y_pred = predict_fn(X_mtl_all[test_positions])[:, head_index]

                metrics = evaluate_predictions(y_test, y_pred)
                multitask_rows.append({
                    "organism_group": organism_data["organism_group"].iloc[0],
                    "organism_name": organism,
                    "dataset_size_category": organism_data["dataset_size_category"].iloc[0],
                    "organism_rows": len(organism_data),
                    "model": "MultitaskNN",
                    "strategy": "multitask_nn",
                    "feature_set": MAIN_FEATURE_SET,
                    "split_id": split_id,
                    "train_chemicals": len(fit_positions),
                    "validation_chemicals": len(valid_positions),
                    "epochs_run": epochs_run,
                    "validation_masked_mse": validation_loss,
                    "test_rows": len(te),
                    **{f"test_{k}": v for k, v in metrics.items()},
                })

                for i in range(len(te)):
                    multitask_predictions.append({
                        "organism_group": organism_data["organism_group"].iloc[0],
                        "organism_name": organism,
                        "strategy": "multitask_nn",
                        "model": "MultitaskNN",
                        "split_id": split_id,
                        "row_id": int(test_rows.loc[i, "row_id"]),
                        "canonical_smiles": test_rows.loc[i, "canonical_smiles"],
                        "observed": float(y_test[i]),
                        "predicted": float(y_pred[i]),
                    })

            print(f"  {organism}: multitask evaluation done")

multitask_df = pd.DataFrame(multitask_rows)

if len(multitask_df):
    save_table(round_df(multitask_df), step6_dir, "step6_multitask_per_split.csv", show=False)

    multitask_summary = (
        multitask_df
        .groupby(["organism_group", "organism_name", "dataset_size_category"])
        .agg(r2_mean=("test_r2", "mean"), mae_mean=("test_mae", "mean"),
             rmse_mean=("test_rmse", "mean"), spearman_mean=("test_spearman", "mean"),
             epochs_mean=("epochs_run", "mean"), n_splits=("split_id", "nunique"))
        .reset_index()
    )
    print("\nMultitask network performance by organism:")
    display(round_df(multitask_summary))
    save_table(round_df(multitask_summary), step6_dir, "step6_multitask_summary.csv", show=False)

    # ---- "keep it only if it clearly improves" ----
    comparable_splits = multitask_df[["organism_name", "split_id"]].drop_duplicates()
    classical = comparison_df.merge(comparable_splits, on=["organism_name", "split_id"])
    classical_best = (
        classical.groupby(["organism_name", "split_id"])["test_r2"].max().reset_index()
        .rename(columns={"test_r2": "best_classical_r2"})
    )
    verdict = (
        multitask_df[["organism_name", "split_id", "test_r2"]]
        .rename(columns={"test_r2": "multitask_r2"})
        .merge(classical_best, on=["organism_name", "split_id"])
    )
    verdict["delta_r2"] = verdict["multitask_r2"] - verdict["best_classical_r2"]

    save_table(round_df(verdict), step6_dir, "step6_multitask_vs_classical.csv", show=False)

    mean_delta = verdict["delta_r2"].mean()
    print("\n" + "=" * 70)
    print("VERDICT - multitask network vs the best classical model on the same splits")
    print(f"  mean change in R2 : {mean_delta:+.4f}")
    print(f"  splits improved   : {100 * (verdict['delta_r2'] > 0).mean():.1f}%")
    if mean_delta > 0.02:
        print("  -> Clear improvement. Keep the multitask network as a main model.")
    elif mean_delta > -0.02:
        print("  -> No clear improvement. Report it as the required neural-network baseline, "
              "but keep a tree model as the primary predictor.")
    else:
        print("  -> The multitask network underperforms here. Report it as the "
              "neural-network baseline and keep the tree models.")
    print("=" * 70)


>>> Step 6: multitask network (2 split(s) per organism) ...
  Danio rerio: multitask evaluation done
  Daphnia magna: multitask evaluation done
  Desmodesmus subspicatus: multitask evaluation done
  Oncorhynchus mykiss: multitask evaluation done
  Oryzias latipes: multitask evaluation done
  Pimephales promelas: multitask evaluation done
  Raphidocelis subcapitata: multitask evaluation done
  Skeletonema costatum: multitask evaluation done
<<< Step 6: multitask network (2 split(s) per organism) finished in 130.4 s
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step6_multitask_nn\step6_multitask_per_split.csv  (16 rows)

Multitask network performance by organism:


,organism_group,organism_name,dataset_size_category,r2_mean,mae_mean,rmse_mean,spearman_mean,epochs_mean,n_splits
0,Algae,Desmodesmus subspicatus,strong,0.2474,0.8066,1.1031,0.5546,24.0,2
1,Algae,Raphidocelis subcapitata,strong,0.2466,0.8125,1.0676,0.5318,22.5,2
2,Algae,Skeletonema costatum,medium,0.3424,1.0932,1.4141,0.5823,23.5,2
3,Fish,Danio rerio,strong,0.3094,0.8140,1.0690,0.5712,22.0,2
4,Fish,Oncorhynchus mykiss,strong,0.2539,0.8594,1.1196,0.5145,24.5,2
5,Fish,Oryzias latipes,medium,0.3735,0.8029,1.0531,0.6277,22.5,2
6,Fish,Pimephales promelas,strong,0.4115,0.9055,1.2135,0.6467,22.0,2
7,Invertebrates,Daphnia magna,strong,0.2493,0.9625,1.2445,0.5011,23.0,2


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step6_multitask_nn\step6_multitask_summary.csv  (8 rows)
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step6_multitask_nn\step6_multitask_vs_classical.csv  (16 rows)

VERDICT - multitask network vs the best classical model on the same splits
  mean change in R2 : -0.1297
  splits improved   : 0.0%
  -> The multitask network underperforms here. Report it as the neural-network baseline and keep the tree models.


## Step 7 — Full metric reporting with confidence intervals

*Addresses suggestion 6: "Report R², MAE, RMSE, and Spearman correlation for each organism, along
with mean/median performance, variability, and confidence intervals."*

Two complementary kinds of uncertainty are reported, because they answer different questions:

- **Across-split variability** (mean, median, standard deviation, min, max over the outer splits)
  — how stable the result is when a different set of scaffolds is held out.
- **Bootstrap confidence intervals** (`N_BOOTSTRAP` resamples of the pooled test predictions,
  2.5th–97.5th percentile) — how precisely each metric is estimated given the number of test
  compounds. These are the intervals to quote in the thesis table.

Everything produced in Steps 5 and 6 is combined first, so one table covers every strategy.

In [16]:
# ============================================================
# STEP 7: METRICS WITH CONFIDENCE INTERVALS
# ============================================================

step7_dir = OUTPUT_DIR / "step7_metrics"

# Combine the classical strategies with the neural baseline.
all_results = comparison_df.copy()
if len(multitask_df):
    shared_columns = [c for c in all_results.columns if c in multitask_df.columns]
    all_results = pd.concat(
        [all_results, multitask_df[shared_columns]], ignore_index=True, sort=False
    )

all_predictions = predictions_df.copy()
if len(multitask_predictions):
    all_predictions = pd.concat(
        [all_predictions, pd.DataFrame(multitask_predictions)], ignore_index=True, sort=False
    )

save_table(all_predictions, step7_dir, "step7_all_row_level_predictions.csv", show=False)


def bootstrap_metric_cis(y_true, y_pred, n_bootstrap=N_BOOTSTRAP, alpha=0.05,
                         random_state=RANDOM_STATE):
    """Percentile bootstrap confidence intervals for the four metrics."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    n = len(y_true)
    if n < 10:
        return {}

    rng = np.random.default_rng(random_state)
    collected = {"r2": [], "mae": [], "rmse": [], "spearman": []}

    for _ in range(n_bootstrap):
        sample = rng.integers(0, n, n)
        yt, yp = y_true[sample], y_pred[sample]
        if np.std(yt) == 0:
            continue
        collected["r2"].append(r2_score(yt, yp))
        collected["mae"].append(mean_absolute_error(yt, yp))
        collected["rmse"].append(np.sqrt(mean_squared_error(yt, yp)))
        rho = spearmanr(yt, yp).correlation
        collected["spearman"].append(rho if rho is not None and not np.isnan(rho) else np.nan)

    output = {}
    for metric, values in collected.items():
        values = np.asarray([v for v in values if v is not None and not np.isnan(v)])
        if len(values) == 0:
            continue
        output[f"{metric}_ci_lower"] = float(np.percentile(values, 100 * alpha / 2))
        output[f"{metric}_ci_upper"] = float(np.percentile(values, 100 * (1 - alpha / 2)))
    return output


# ------------------------------------------------------------------
# 7.1 Across-split variability
# ------------------------------------------------------------------
split_variability = (
    all_results
    .groupby(["organism_group", "organism_name", "dataset_size_category", "model", "strategy"])
    .agg(
        n_splits=("split_id", "nunique"),
        r2_mean=("test_r2", "mean"), r2_median=("test_r2", "median"),
        r2_std=("test_r2", "std"), r2_min=("test_r2", "min"), r2_max=("test_r2", "max"),
        mae_mean=("test_mae", "mean"), mae_median=("test_mae", "median"),
        mae_std=("test_mae", "std"),
        rmse_mean=("test_rmse", "mean"), rmse_median=("test_rmse", "median"),
        rmse_std=("test_rmse", "std"),
        spearman_mean=("test_spearman", "mean"), spearman_median=("test_spearman", "median"),
        spearman_std=("test_spearman", "std"),
    )
    .reset_index()
)

# ------------------------------------------------------------------
# 7.2 Bootstrap CIs on the pooled test predictions
# ------------------------------------------------------------------
ci_rows = []
with Timer(f"Step 7: bootstrap confidence intervals ({N_BOOTSTRAP} resamples)"):
    for (organism, strategy, model_name), subset in all_predictions.groupby(
        ["organism_name", "strategy", "model"]
    ):
        point_estimates = evaluate_predictions(subset["observed"], subset["predicted"])
        intervals = bootstrap_metric_cis(subset["observed"].values, subset["predicted"].values)
        ci_rows.append({
            "organism_name": organism,
            "strategy": strategy,
            "model": model_name,
            "n_test_predictions": len(subset),
            **{f"{k}_point": v for k, v in point_estimates.items()},
            **intervals,
        })

ci_df = pd.DataFrame(ci_rows)

metrics_full = split_variability.merge(
    ci_df, on=["organism_name", "strategy", "model"], how="left"
)
save_table(round_df(metrics_full), step7_dir, "step7_metrics_with_confidence_intervals.csv",
           show=False)

# ------------------------------------------------------------------
# 7.3 A readable table for the report
# ------------------------------------------------------------------
report_table = metrics_full[[
    "organism_group", "organism_name", "dataset_size_category", "model", "strategy",
    "n_splits", "r2_mean", "r2_std", "r2_ci_lower", "r2_ci_upper",
    "mae_mean", "mae_ci_lower", "mae_ci_upper",
    "rmse_mean", "rmse_ci_lower", "rmse_ci_upper",
    "spearman_mean", "spearman_ci_lower", "spearman_ci_upper",
]].sort_values(["organism_name", "r2_mean"], ascending=[True, False])

print("Per-organism metrics with 95% bootstrap confidence intervals:")
display(round_df(report_table).head(40))
save_table(round_df(report_table), step7_dir, "step7_report_table.csv", show=False)

# ------------------------------------------------------------------
# 7.4 Best configuration per organism
# ------------------------------------------------------------------
best_per_organism = (
    metrics_full.sort_values(["organism_name", "r2_mean"], ascending=[True, False])
    .groupby("organism_name").head(1).reset_index(drop=True)
)
print("\nBest configuration for each organism:")
display(round_df(best_per_organism[[
    "organism_group", "organism_name", "dataset_size_category", "model", "strategy",
    "r2_mean", "r2_ci_lower", "r2_ci_upper", "mae_mean", "spearman_mean",
]]))
save_table(round_df(best_per_organism), step7_dir, "step7_best_per_organism.csv", show=False)

  saved: C:\Users\lab\Desktop\ecotoxicity\working\step7_metrics\step7_all_row_level_predictions.csv  (163933 rows)

>>> Step 7: bootstrap confidence intervals (1000 resamples) ...
<<< Step 7: bootstrap confidence intervals (1000 resamples) finished in 159.1 s
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step7_metrics\step7_metrics_with_confidence_intervals.csv  (104 rows)
Per-organism metrics with 95% bootstrap confidence intervals:


,organism_group,organism_name,dataset_size_category,model,strategy,n_splits,r2_mean,r2_std,r2_ci_lower,r2_ci_upper,mae_mean,mae_ci_lower,mae_ci_upper,rmse_mean,rmse_ci_lower,rmse_ci_upper,spearman_mean,spearman_ci_lower,spearman_ci_upper
45,Fish,Danio rerio,strong,RandomForest,hierarchical_residual,5,0.3749,0.0373,0.3417,0.4070,0.7174,0.6836,0.7342,0.9541,0.9135,0.9836,0.6279,0.5944,0.6493
49,Fish,Danio rerio,strong,XGBoost,hierarchical_residual,5,0.3659,0.0418,0.3302,0.4009,0.7190,0.6867,0.7366,0.9607,0.9223,0.9913,0.6224,0.5875,0.6440
40,Fish,Danio rerio,strong,CatBoost,hierarchical_residual,5,0.3648,0.0407,0.3265,0.3983,0.7207,0.6905,0.7391,0.9616,0.9217,0.9938,0.6236,0.5882,0.6436
51,Fish,Danio rerio,strong,XGBoost,species_aware_pooled,5,0.3634,0.0389,0.3314,0.3931,0.7218,0.6899,0.7391,0.9627,0.9236,0.9915,0.6254,0.5891,0.6455
46,Fish,Danio rerio,strong,RandomForest,independent_organism,5,0.3606,0.0334,0.3280,0.3871,0.7198,0.6884,0.7392,0.9654,0.9262,0.9956,0.6258,0.5879,0.6452
42,Fish,Danio rerio,strong,CatBoost,species_aware_pooled,5,0.3583,0.0389,0.3263,0.3904,0.7220,0.6884,0.7389,0.9666,0.9238,0.9977,0.6272,0.5917,0.6478
47,Fish,Danio rerio,strong,RandomForest,species_aware_pooled,5,0.3581,0.0365,0.3288,0.3877,0.7277,0.6939,0.7431,0.9669,0.9260,0.9938,0.6231,0.5879,0.6426
50,Fish,Danio rerio,strong,XGBoost,independent_organism,5,0.3535,0.0384,0.3197,0.3860,0.7216,0.6891,0.7397,0.9702,0.9276,0.9993,0.6152,0.5801,0.6362
41,Fish,Danio rerio,strong,CatBoost,independent_organism,5,0.3517,0.0393,0.3140,0.3797,0.7257,0.6950,0.7451,0.9719,0.9332,1.0026,0.6158,0.5804,0.6369
44,Fish,Danio rerio,strong,RandomForest,group_pooled,5,0.3265,0.0439,0.2936,0.3653,0.7480,0.7123,0.7625,0.9902,0.9483,1.0177,0.6016,0.5661,0.6254


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step7_metrics\step7_report_table.csv  (104 rows)

Best configuration for each organism:


,organism_group,organism_name,dataset_size_category,model,strategy,r2_mean,r2_ci_lower,r2_ci_upper,mae_mean,spearman_mean
0,Fish,Danio rerio,strong,RandomForest,hierarchical_residual,0.3749,0.3417,0.4070,0.7174,0.6279
1,Invertebrates,Daphnia magna,strong,RandomForest,species_aware_pooled,0.3387,0.2661,0.3799,0.8481,0.5802
2,Algae,Desmodesmus subspicatus,strong,RandomForest,hierarchical_residual,0.3920,0.3530,0.4374,0.7168,0.6656
3,Fish,Oncorhynchus mykiss,strong,RandomForest,hierarchical_residual,0.4385,0.4033,0.4807,0.7353,0.6663
4,Fish,Oryzias latipes,medium,CatBoost,species_aware_pooled,0.4254,0.3634,0.4891,0.7067,0.6777
5,Fish,Pimephales promelas,strong,CatBoost,hierarchical_residual,0.5581,0.5138,0.5866,0.7578,0.7274
6,Algae,Raphidocelis subcapitata,strong,XGBoost,species_aware_pooled,0.3751,0.3402,0.3986,0.6904,0.6392
7,Algae,Skeletonema costatum,medium,CatBoost,species_aware_pooled,0.4394,0.3705,0.4975,1.0501,0.6889


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step7_metrics\step7_best_per_organism.csv  (8 rows)


WindowsPath('C:/Users/lab/Desktop/ecotoxicity/working/step7_metrics/step7_best_per_organism.csv')

## Step 8 — Stacking the strongest models, with weights from out-of-fold predictions

*Addresses suggestion 7: "Test averaging or stacking only for the strongest models. Learn ensemble
weights from out-of-fold predictions to avoid information leakage."*

**The leakage trap this avoids.** If ensemble weights were fitted on the test set, the ensemble
would look excellent and generalise poorly. Here, for each outer split:

1. The outer training set is divided into inner scaffold-grouped folds.
2. Each base model is trained on *k−1* folds and predicts the held-out fold. Repeating this gives
   one **out-of-fold prediction per training row per model** — predictions no model saw during its
   own training.
3. A non-negative ridge meta-learner is fitted on that out-of-fold matrix. Weights are clipped at
   zero and normalised, so they read as interpretable "model shares".
4. Each base model is then refitted on the *full* outer training set and the learned weights are
   applied to the test predictions.

The outer test rows enter only at the final scoring step. A simple equal-weight average is also
reported, because it is often as good as a learned blend and much easier to defend.

Only the **top `N_STACK_MODELS` families** by Step 4's inner-fold score are stacked, as requested.

In [17]:
# ============================================================
# STEP 8: OUT-OF-FOLD STACKING
# ============================================================

step8_dir = OUTPUT_DIR / "step8_stacking"

N_STACK_MODELS = 3       # "only the strongest models"

# Pick the strongest families from the Step 4 tuning scores; fall back sensibly.
if len(tuning_df):
    family_ranking = (
        tuning_df.groupby("model")["best_inner_r2"].mean()
        .sort_values(ascending=False)
    )
    STACK_MODELS = family_ranking.head(N_STACK_MODELS).index.tolist()
else:
    candidates = ["RandomForest", "ExtraTrees", "HistGB"]
    if AVAILABLE.get("xgboost"):
        candidates.insert(0, "XGBoost")
    if AVAILABLE.get("catboost"):
        candidates.insert(1, "CatBoost")
    STACK_MODELS = candidates[:N_STACK_MODELS]

print("Models entering the ensemble:", STACK_MODELS)


def fit_stack_weights(oof_predictions, y_true):
    """
    Learn non-negative blend weights from out-of-fold predictions.

    Ridge keeps the solution stable when base models are highly correlated, which they
    usually are. Negative weights are clipped away so the result stays interpretable.
    """
    meta = RidgeCV(alphas=np.logspace(-3, 3, 13))
    meta.fit(oof_predictions, y_true)
    weights = np.clip(meta.coef_, 0, None)
    if weights.sum() <= 0:                       # degenerate case -> equal weights
        weights = np.ones(oof_predictions.shape[1])
    return weights / weights.sum()


stacking_rows = []
stacking_weight_rows = []

with Timer("Step 8: out-of-fold stacking"):
    for organism in MODELLED_ORGANISMS:
        organism_data = aquatic[aquatic["organism_name"] == organism].reset_index(drop=True)
        organism_position = {int(r): i for i, r in enumerate(organism_data["row_id"].values)}
        task_id = f"organism__{organism}"

        X_all = build_feature_matrix(organism_data, MAIN_FEATURE_SET).values
        y_all = organism_data["model_target"].values

        for split_id in range(1, N_SPLITS_MAIN + 1):
            train_ids, test_ids = get_split_rows(task_id, split_id)
            tr = [organism_position[int(r)] for r in train_ids]
            te = [organism_position[int(r)] for r in test_ids]

            X_train, X_test = X_all[tr], X_all[te]
            y_train, y_test = y_all[tr], y_all[te]
            train_data = organism_data.iloc[tr].reset_index(drop=True)

            folds = list(inner_folds(train_data))

            # ---- 1) out-of-fold predictions on the training rows ----
            oof = np.full((len(tr), len(STACK_MODELS)), np.nan)
            for model_pos, model_name in enumerate(STACK_MODELS):
                params = tuned_params(task_id, model_name)
                for inner_train, inner_valid in folds:
                    model = build_model(model_name, params)
                    model.fit(X_train[inner_train], y_train[inner_train])
                    oof[inner_valid, model_pos] = model.predict(X_train[inner_valid])

            covered = ~np.isnan(oof).any(axis=1)
            if covered.sum() < 20:
                continue

            # ---- 2) weights from the out-of-fold matrix only ----
            weights = fit_stack_weights(oof[covered], y_train[covered])

            # ---- 3) refit on all training rows, predict the test rows ----
            test_predictions = np.zeros((len(te), len(STACK_MODELS)))
            for model_pos, model_name in enumerate(STACK_MODELS):
                model = build_model(model_name, tuned_params(task_id, model_name))
                model.fit(X_train, y_train)
                test_predictions[:, model_pos] = model.predict(X_test)

            stacked = test_predictions @ weights
            averaged = test_predictions.mean(axis=1)

            for label, prediction in (
                ("stacked_oof_weights", stacked),
                ("simple_average", averaged),
            ):
                metrics = evaluate_predictions(y_test, prediction)
                stacking_rows.append({
                    "organism_group": organism_data["organism_group"].iloc[0],
                    "organism_name": organism,
                    "dataset_size_category": organism_data["dataset_size_category"].iloc[0],
                    "model": "Ensemble",
                    "strategy": label,
                    "split_id": split_id,
                    "n_base_models": len(STACK_MODELS),
                    "test_rows": len(te),
                    **{f"test_{k}": v for k, v in metrics.items()},
                })

            # Individual base models on the same split, for a like-for-like comparison.
            for model_pos, model_name in enumerate(STACK_MODELS):
                metrics = evaluate_predictions(y_test, test_predictions[:, model_pos])
                stacking_rows.append({
                    "organism_group": organism_data["organism_group"].iloc[0],
                    "organism_name": organism,
                    "dataset_size_category": organism_data["dataset_size_category"].iloc[0],
                    "model": model_name,
                    "strategy": "single_model",
                    "split_id": split_id,
                    "n_base_models": 1,
                    "test_rows": len(te),
                    **{f"test_{k}": v for k, v in metrics.items()},
                })

            stacking_weight_rows.append({
                "organism_name": organism,
                "split_id": split_id,
                "oof_rows_used": int(covered.sum()),
                **{f"weight__{name}": float(w) for name, w in zip(STACK_MODELS, weights)},
            })

        print(f"  stacked: {organism}")

stacking_df = pd.DataFrame(stacking_rows)
stacking_weights_df = pd.DataFrame(stacking_weight_rows)

save_table(stacking_df, step8_dir, "step8_stacking_per_split.csv", show=False)
save_table(round_df(stacking_weights_df), step8_dir, "step8_learned_weights.csv", max_rows=20)

stacking_summary = (
    stacking_df
    .groupby(["organism_name", "dataset_size_category", "model", "strategy"])
    .agg(r2_mean=("test_r2", "mean"), r2_std=("test_r2", "std"),
         mae_mean=("test_mae", "mean"), rmse_mean=("test_rmse", "mean"),
         spearman_mean=("test_spearman", "mean"))
    .reset_index()
)
save_table(round_df(stacking_summary), step8_dir, "step8_stacking_summary.csv", show=False)

ensemble_vs_single = (
    stacking_df.groupby("strategy")
    .agg(mean_r2=("test_r2", "mean"), median_r2=("test_r2", "median"),
         mean_mae=("test_mae", "mean"), mean_rmse=("test_rmse", "mean"),
         mean_spearman=("test_spearman", "mean"))
    .reset_index().sort_values("mean_r2", ascending=False)
)
print("\nEnsemble vs single models (all organisms, all splits):")
display(round_df(ensemble_vs_single))
save_table(round_df(ensemble_vs_single), step8_dir, "step8_ensemble_vs_single.csv", show=False)

# Mean learned weight per model: which family the blend actually relies on.
weight_columns = [c for c in stacking_weights_df.columns if c.startswith("weight__")]
if weight_columns:
    mean_weights = (
        stacking_weights_df[weight_columns].mean()
        .rename(lambda c: c.replace("weight__", "")).reset_index()
    )
    mean_weights.columns = ["model", "mean_learned_weight"]
    print("\nMean out-of-fold ensemble weight per model family:")
    display(round_df(mean_weights.sort_values("mean_learned_weight", ascending=False)))
    save_table(round_df(mean_weights), step8_dir, "step8_mean_weights.csv", show=False)

Models entering the ensemble: ['XGBoost', 'RandomForest', 'CatBoost']

>>> Step 8: out-of-fold stacking ...
  stacked: Danio rerio
  stacked: Daphnia magna
  stacked: Desmodesmus subspicatus
  stacked: Oncorhynchus mykiss
  stacked: Oryzias latipes
  stacked: Pimephales promelas
  stacked: Raphidocelis subcapitata
  stacked: Skeletonema costatum
<<< Step 8: out-of-fold stacking finished in 2474.0 s
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step8_stacking\step8_stacking_per_split.csv  (200 rows)
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step8_stacking\step8_learned_weights.csv  (40 rows)


,organism_name,split_id,oof_rows_used,weight__XGBoost,weight__RandomForest,weight__CatBoost
0,Danio rerio,1,2070,0.2126,0.3426,0.4448
1,Danio rerio,2,2053,0.2119,0.3625,0.4256
2,Danio rerio,3,2066,0.2830,0.4632,0.2538
3,Danio rerio,4,2067,0.0592,0.4292,0.5115
4,Danio rerio,5,1709,0.2006,0.3096,0.4897
5,Daphnia magna,1,621,0.3442,0.3565,0.2993
6,Daphnia magna,2,850,0.2821,0.4035,0.3144
7,Daphnia magna,3,851,0.0000,0.3645,0.6355
8,Daphnia magna,4,632,0.0000,0.6824,0.3176
9,Daphnia magna,5,854,0.2713,0.4515,0.2772


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step8_stacking\step8_stacking_summary.csv  (40 rows)

Ensemble vs single models (all organisms, all splits):


,strategy,mean_r2,median_r2,mean_mae,mean_rmse,mean_spearman
0,simple_average,0.3971,0.3893,0.7914,1.0586,0.6419
2,stacked_oof_weights,0.3963,0.3831,0.7919,1.0592,0.6412
1,single_model,0.3889,0.3779,0.7972,1.0656,0.6359


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step8_stacking\step8_ensemble_vs_single.csv  (3 rows)

Mean out-of-fold ensemble weight per model family:


,model,mean_learned_weight
2,CatBoost,0.3693
1,RandomForest,0.3591
0,XGBoost,0.2716


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step8_stacking\step8_mean_weights.csv  (3 rows)


## Step 9 — Reliability: applicability domain, verified Tanimoto, calibrated intervals

*Addresses suggestion 8: "Evaluate reliability using chemically similar versus dissimilar test
compounds. Check Tanimoto calculations against RDKit and report applicability-domain coverage and
calibrated prediction intervals."*

Three sub-steps:

**9A — Tanimoto verification.** Phase 2 computed nearest-neighbour similarity with a fast matrix
formula. Here that fast path is checked element-by-element against RDKit's own
`BulkTanimotoSimilarity` on a random sample. The maximum absolute difference is printed and
asserted below 1e-6, so the speed optimisation is proven not to change the numbers.

**9B — Applicability domain and conformal intervals.** For each test compound: its nearest-
neighbour Tanimoto similarity to the training set, the Random-Forest tree-spread as a difficulty
estimate, and a **split-conformal prediction interval**.

*Why conformal rather than the tree spread alone?* The tree spread tells you the model is unsure,
but not by how much in real units. Split conformal takes a calibration set carved out of the
training data **by scaffold**, measures the actual absolute residuals there, and sets the interval
width at the (1−α) quantile. That gives an interval with a **guaranteed** marginal coverage of
about 1−α. Two variants are produced:

- *standard*: one width for everything;
- *normalised*: width scaled by the tree spread, so chemically awkward compounds get honestly
  wider intervals while easy compounds get tighter ones.

Empirical coverage is then measured on the real test rows — this is the number to report.

**9C — Descriptor SHAP.** *Addresses the end of suggestion 2: "We can later use descriptor-based
SHAP to trim."* SHAP ranks the extended descriptors, and models retrained on the top 20/50/100
show how far the descriptor set can be trimmed before accuracy drops.

In [18]:
# ============================================================
# STEP 9A: VERIFY THE FAST TANIMOTO AGAINST RDKit
# ============================================================

step9_dir = OUTPUT_DIR / "step9_reliability"


def morgan_bit_matrix(smiles_series, n_bits=1024, radius=2):
    """Binary Morgan matrix (uint8) for fast matrix-based similarity."""
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    rows = []
    for smi in smiles_series:
        mol = mol_from_smiles(smi)
        rows.append(
            np.zeros(n_bits, dtype=np.uint8) if mol is None
            else fp_to_numpy(generator.GetFingerprint(mol), n_bits)
        )
    return np.vstack(rows).astype(np.uint8)


def rdkit_fingerprints(smiles_series, n_bits=1024, radius=2):
    """RDKit bit-vector objects, for the reference similarity calculation."""
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    output = []
    for smi in smiles_series:
        mol = mol_from_smiles(smi)
        output.append(None if mol is None else generator.GetFingerprint(mol))
    return output


def max_tanimoto_to_train(test_bits, train_bits, chunk_size=512):
    """
    Nearest-neighbour Tanimoto of each test compound to the training set.

    Tanimoto = |A and B| / (|A| + |B| - |A and B|), computed as a matrix product.
    Verified against RDKit in the check below.
    """
    train_bits = train_bits.astype(np.int32)
    train_sums = train_bits.sum(axis=1)
    results = []
    for start in range(0, len(test_bits), chunk_size):
        block = test_bits[start:start + chunk_size].astype(np.int32)
        intersection = block @ train_bits.T
        block_sums = block.sum(axis=1, keepdims=True)
        union = block_sums + train_sums.reshape(1, -1) - intersection
        similarity = np.divide(
            intersection, union,
            out=np.zeros(intersection.shape, dtype=float), where=union != 0,
        )
        results.extend(similarity.max(axis=1))
    return np.asarray(results)


# ------------------------------------------------------------------
# The verification itself
# ------------------------------------------------------------------
with Timer("Step 9A: verifying the fast Tanimoto against RDKit"):
    rng = np.random.default_rng(RANDOM_STATE)
    sample_smiles = (
        ml_ready["canonical_smiles"].drop_duplicates()
        .sample(min(400, ml_ready["canonical_smiles"].nunique()), random_state=RANDOM_STATE)
        .tolist()
    )
    reference_smiles = sample_smiles[:200]
    query_smiles = sample_smiles[200:]

    fast_values = max_tanimoto_to_train(
        morgan_bit_matrix(query_smiles), morgan_bit_matrix(reference_smiles)
    )

    reference_fps = [fp for fp in rdkit_fingerprints(reference_smiles) if fp is not None]
    rdkit_values = []
    for fp in rdkit_fingerprints(query_smiles):
        if fp is None or not reference_fps:
            rdkit_values.append(0.0)
        else:
            rdkit_values.append(max(DataStructs.BulkTanimotoSimilarity(fp, reference_fps)))
    rdkit_values = np.asarray(rdkit_values)

    max_difference = float(np.max(np.abs(fast_values - rdkit_values)))
    correlation = float(np.corrcoef(fast_values, rdkit_values)[0, 1])

print(f"\n  compounds compared       : {len(query_smiles)}")
print(f"  max absolute difference  : {max_difference:.3e}")
print(f"  correlation with RDKit   : {correlation:.8f}")

tanimoto_check = pd.DataFrame({
    "n_query_compounds": [len(query_smiles)],
    "n_reference_compounds": [len(reference_smiles)],
    "max_absolute_difference": [max_difference],
    "correlation_with_rdkit": [correlation],
    "verdict": ["identical to RDKit within floating-point tolerance"
                if max_difference < 1e-6 else "MISMATCH - investigate"],
})
save_table(tanimoto_check, step9_dir, "step9A_tanimoto_verification.csv")

assert max_difference < 1e-6, (
    "The fast Tanimoto does not match RDKit; do not trust the applicability-domain numbers."
)
print("  Verified: the fast Tanimoto reproduces RDKit exactly.")


>>> Step 9A: verifying the fast Tanimoto against RDKit ...
<<< Step 9A: verifying the fast Tanimoto against RDKit finished in 0.2 s

  compounds compared       : 200
  max absolute difference  : 0.000e+00
  correlation with RDKit   : 1.00000000
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step9_reliability\step9A_tanimoto_verification.csv  (1 rows)


,n_query_compounds,n_reference_compounds,max_absolute_difference,correlation_with_rdkit,verdict
0,200,200,0.0,1.0,identical to RDKit within floating-point toler...


  Verified: the fast Tanimoto reproduces RDKit exactly.


In [19]:
# ============================================================
# STEP 9B: APPLICABILITY DOMAIN AND CONFORMAL PREDICTION INTERVALS
# ============================================================

def split_conformal_intervals(model_name, params, X_train, y_train, X_test,
                              train_scaffolds, alpha=CONFORMAL_ALPHA,
                              calibration_fraction=0.25):
    """
    Split-conformal regression with a scaffold-grouped calibration set.

    Returns predictions, interval half-widths (standard and normalised) and the
    tree-spread difficulty estimate for the test compounds.

    Guarantee: on exchangeable data the standard interval covers the true value with
    probability at least 1 - alpha. Splitting by SCAFFOLD rather than at random makes
    the calibration residuals reflect the scaffold-shift the model really faces.
    """
    splitter = GroupShuffleSplit(
        n_splits=1, test_size=calibration_fraction, random_state=RANDOM_STATE
    )
    fit_idx, calib_idx = next(
        splitter.split(np.zeros(len(X_train)), groups=train_scaffolds)
    )

    model = build_model(model_name, params)
    model.fit(X_train[fit_idx], y_train[fit_idx])

    # --- difficulty estimate from the spread across trees ---
    def tree_spread(X):
        if hasattr(model, "estimators_"):
            try:
                per_tree = np.vstack([t.predict(X) for t in model.estimators_])
                return per_tree.std(axis=0)
            except Exception:
                pass
        return np.ones(len(X))

    calib_prediction = model.predict(X_train[calib_idx])
    calib_residual = np.abs(y_train[calib_idx] - calib_prediction)
    calib_spread = tree_spread(X_train[calib_idx])

    n_calib = len(calib_idx)
    # Finite-sample corrected quantile level.
    level = min(1.0, np.ceil((n_calib + 1) * (1 - alpha)) / n_calib)

    q_standard = float(np.quantile(calib_residual, level))
    normalised_scores = calib_residual / (calib_spread + 1e-6)
    q_normalised = float(np.quantile(normalised_scores, level))

    # Refit on everything for the final point predictions.
    final_model = build_model(model_name, params)
    final_model.fit(X_train, y_train)
    test_prediction = final_model.predict(X_test)

    if hasattr(final_model, "estimators_"):
        try:
            per_tree = np.vstack([t.predict(X_test) for t in final_model.estimators_])
            test_spread = per_tree.std(axis=0)
        except Exception:
            test_spread = np.ones(len(X_test))
    else:
        test_spread = np.ones(len(X_test))

    return {
        "prediction": test_prediction,
        "half_width_standard": np.full(len(X_test), q_standard),
        "half_width_normalised": q_normalised * (test_spread + 1e-6),
        "tree_spread": test_spread,
        "n_calibration": n_calib,
    }


AD_MODEL = "RandomForest"       # tree spread needs an ensemble of trees
reliability_rows = []

with Timer("Step 9B: applicability domain and conformal intervals"):
    for organism in MODELLED_ORGANISMS:
        organism_data = aquatic[aquatic["organism_name"] == organism].reset_index(drop=True)
        organism_position = {int(r): i for i, r in enumerate(organism_data["row_id"].values)}
        task_id = f"organism__{organism}"
        params = tuned_params(task_id, AD_MODEL)

        X_all = build_feature_matrix(organism_data, MAIN_FEATURE_SET).values
        y_all = organism_data["model_target"].values
        bits_all = morgan_bit_matrix(organism_data["canonical_smiles"], n_bits=1024)

        for split_id in range(1, N_SPLITS_MAIN + 1):
            train_ids, test_ids = get_split_rows(task_id, split_id)
            tr = [organism_position[int(r)] for r in train_ids]
            te = [organism_position[int(r)] for r in test_ids]

            conformal = split_conformal_intervals(
                AD_MODEL, params, X_all[tr], y_all[tr], X_all[te],
                train_scaffolds=organism_data.iloc[tr]["scaffold"].values,
            )

            similarity = max_tanimoto_to_train(bits_all[te], bits_all[tr])
            test_rows = organism_data.iloc[te].reset_index(drop=True)

            for i in range(len(te)):
                observed = float(y_all[te][i])
                predicted = float(conformal["prediction"][i])
                half_standard = float(conformal["half_width_standard"][i])
                half_normalised = float(conformal["half_width_normalised"][i])
                reliability_rows.append({
                    "organism_group": test_rows.loc[i, "organism_group"],
                    "organism_name": organism,
                    "split_id": split_id,
                    "row_id": int(test_rows.loc[i, "row_id"]),
                    "canonical_smiles": test_rows.loc[i, "canonical_smiles"],
                    "is_halogenated": bool(test_rows.loc[i, "is_halogenated"]),
                    "halogen_type": test_rows.loc[i, "halogen_type"],
                    "observed": observed,
                    "predicted": predicted,
                    "absolute_error": abs(observed - predicted),
                    "max_train_tanimoto": float(similarity[i]),
                    "tree_spread": float(conformal["tree_spread"][i]),
                    "pi_lower_standard": predicted - half_standard,
                    "pi_upper_standard": predicted + half_standard,
                    "pi_width_standard": 2 * half_standard,
                    "covered_standard": abs(observed - predicted) <= half_standard,
                    "pi_lower_normalised": predicted - half_normalised,
                    "pi_upper_normalised": predicted + half_normalised,
                    "pi_width_normalised": 2 * half_normalised,
                    "covered_normalised": abs(observed - predicted) <= half_normalised,
                })
        print(f"  reliability done: {organism}")

reliability_df = pd.DataFrame(reliability_rows)

# ------------------------------------------------------------------
# Similarity bands = "chemically similar vs dissimilar test compounds"
# ------------------------------------------------------------------
SIMILARITY_BANDS = [
    (0.00, 0.25, "very_dissimilar (<0.25)"),
    (0.25, 0.40, "dissimilar (0.25-0.40)"),
    (0.40, 0.55, "moderate (0.40-0.55)"),
    (0.55, 0.70, "similar (0.55-0.70)"),
    (0.70, 1.01, "very_similar (>0.70)"),
]


def similarity_band(value):
    for low, high, label in SIMILARITY_BANDS:
        if low <= value < high:
            return label
    return SIMILARITY_BANDS[-1][2]


reliability_df["similarity_band"] = reliability_df["max_train_tanimoto"].apply(similarity_band)

# The Phase-2 style three-level label, kept so the two phases can be compared.
def reliability_label(similarity, spread):
    if similarity >= 0.40 and spread <= 0.75:
        return "in_domain_high_reliability"
    if similarity >= 0.25 and spread <= 1.00:
        return "borderline_moderate_reliability"
    return "out_of_domain_low_reliability"


reliability_df["reliability_label"] = [
    reliability_label(s, u)
    for s, u in zip(reliability_df["max_train_tanimoto"], reliability_df["tree_spread"])
]

save_table(reliability_df, step9_dir, "step9B_predictions_with_reliability.csv", show=False)


>>> Step 9B: applicability domain and conformal intervals ...
  reliability done: Danio rerio
  reliability done: Daphnia magna
  reliability done: Desmodesmus subspicatus
  reliability done: Oncorhynchus mykiss
  reliability done: Oryzias latipes
  reliability done: Pimephales promelas
  reliability done: Raphidocelis subcapitata
  reliability done: Skeletonema costatum
<<< Step 9B: applicability domain and conformal intervals finished in 142.2 s
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step9_reliability\step9B_predictions_with_reliability.csv  (13264 rows)


WindowsPath('C:/Users/lab/Desktop/ecotoxicity/working/step9_reliability/step9B_predictions_with_reliability.csv')

In [20]:
# ============================================================
# STEP 9B (continued): COVERAGE AND RELIABILITY TABLES
# ============================================================

# ---- error by chemical similarity: the core reliability claim ----
band_order = [label for _, _, label in SIMILARITY_BANDS]

error_by_similarity = (
    reliability_df.groupby("similarity_band")
    .agg(
        n_predictions=("row_id", "count"),
        mean_absolute_error=("absolute_error", "mean"),
        median_absolute_error=("absolute_error", "median"),
        rmse=("absolute_error", lambda x: float(np.sqrt(np.mean(np.square(x))))),
        mean_tree_spread=("tree_spread", "mean"),
        coverage_standard=("covered_standard", "mean"),
        coverage_normalised=("covered_normalised", "mean"),
        mean_width_standard=("pi_width_standard", "mean"),
        mean_width_normalised=("pi_width_normalised", "mean"),
    )
    .reindex(band_order).dropna(how="all").reset_index()
)
print(f"Prediction error and interval coverage by chemical similarity "
      f"(nominal coverage = {1 - CONFORMAL_ALPHA:.0%}):")
display(round_df(error_by_similarity))
save_table(round_df(error_by_similarity), step9_dir, "step9B_error_by_similarity.csv", show=False)

# ---- overall coverage: does the conformal guarantee hold in practice? ----
coverage_summary = pd.DataFrame([{
    "nominal_coverage": 1 - CONFORMAL_ALPHA,
    "empirical_coverage_standard": reliability_df["covered_standard"].mean(),
    "empirical_coverage_normalised": reliability_df["covered_normalised"].mean(),
    "mean_interval_width_standard": reliability_df["pi_width_standard"].mean(),
    "mean_interval_width_normalised": reliability_df["pi_width_normalised"].mean(),
    "n_predictions": len(reliability_df),
}])
print(f"\nCalibration check:")
display(round_df(coverage_summary))
save_table(round_df(coverage_summary), step9_dir, "step9B_coverage_summary.csv", show=False)

empirical = coverage_summary["empirical_coverage_standard"].iloc[0]
if abs(empirical - (1 - CONFORMAL_ALPHA)) < 0.05:
    print(f"  Well calibrated: {empirical:.1%} observed vs {1 - CONFORMAL_ALPHA:.0%} nominal.")
else:
    print(f"  Note: observed coverage {empirical:.1%} differs from the "
          f"{1 - CONFORMAL_ALPHA:.0%} nominal level. Scaffold shift between the "
          f"calibration and test sets breaks strict exchangeability, which is expected "
          f"and worth stating in the report.")

# ---- AD coverage: what share of the chemical space the model may be used on ----
ad_coverage = (
    reliability_df.groupby(["organism_group", "organism_name", "reliability_label"])
    .agg(n_predictions=("row_id", "count"),
         mean_absolute_error=("absolute_error", "mean"),
         mean_similarity=("max_train_tanimoto", "mean"))
    .reset_index()
)
totals = ad_coverage.groupby("organism_name")["n_predictions"].transform("sum")
ad_coverage["percent_of_predictions"] = 100 * ad_coverage["n_predictions"] / totals
print("\nApplicability-domain coverage per organism:")
display(round_df(ad_coverage))
save_table(round_df(ad_coverage), step9_dir, "step9B_ad_coverage.csv", show=False)

# ---- accuracy when restricted to in-domain compounds only ----
in_domain_rows = []
for threshold in [0.0, 0.25, 0.40, 0.55]:
    subset = reliability_df[reliability_df["max_train_tanimoto"] >= threshold]
    if len(subset) < 20:
        continue
    metrics = evaluate_predictions(subset["observed"], subset["predicted"])
    in_domain_rows.append({
        "similarity_threshold": threshold,
        "n_predictions": len(subset),
        "percent_of_test_set": 100 * len(subset) / len(reliability_df),
        **metrics,
    })
in_domain_df = pd.DataFrame(in_domain_rows)
print("\nAccuracy vs applicability-domain coverage trade-off:")
display(round_df(in_domain_df))
save_table(round_df(in_domain_df), step9_dir, "step9B_accuracy_vs_coverage.csv", show=False)

Prediction error and interval coverage by chemical similarity (nominal coverage = 90%):


,similarity_band,n_predictions,mean_absolute_error,median_absolute_error,rmse,mean_tree_spread,coverage_standard,coverage_normalised,mean_width_standard,mean_width_normalised
0,very_dissimilar (<0.25),1073,0.9473,0.7281,1.2373,1.0807,0.8425,0.8565,3.5774,3.6187
1,dissimilar (0.25-0.40),4508,0.7625,0.5529,1.0290,1.0193,0.9075,0.9088,3.5020,3.3903
2,moderate (0.40-0.55),3657,0.7422,0.5543,0.9909,0.9679,0.9070,0.9076,3.4550,3.2179
3,similar (0.55-0.70),1973,0.7479,0.5636,1.0068,0.9539,0.9123,0.9108,3.4382,3.1874
4,very_similar (>0.70),2053,0.7106,0.5043,0.9867,0.9315,0.9084,0.9104,3.4409,3.0827


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step9_reliability\step9B_error_by_similarity.csv  (5 rows)

Calibration check:


,nominal_coverage,empirical_coverage_standard,empirical_coverage_normalised,mean_interval_width_standard,mean_interval_width_normalised,n_predictions
0,0.9,0.903,0.9048,3.4762,3.2835,13264


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step9_reliability\step9B_coverage_summary.csv  (1 rows)
  Well calibrated: 90.3% observed vs 90% nominal.

Applicability-domain coverage per organism:


,organism_group,organism_name,reliability_label,n_predictions,mean_absolute_error,mean_similarity,percent_of_predictions
0,Algae,Desmodesmus subspicatus,borderline_moderate_reliability,605,0.7852,0.4548,46.3957
1,Algae,Desmodesmus subspicatus,in_domain_high_reliability,411,0.4683,0.6211,31.5184
2,Algae,Desmodesmus subspicatus,out_of_domain_low_reliability,288,1.0366,0.4060,22.0859
3,Algae,Raphidocelis subcapitata,borderline_moderate_reliability,1461,0.5726,0.4637,43.2249
4,Algae,Raphidocelis subcapitata,in_domain_high_reliability,535,0.4282,0.6320,15.8284
5,Algae,Raphidocelis subcapitata,out_of_domain_low_reliability,1384,0.9696,0.4701,40.9467
6,Algae,Skeletonema costatum,borderline_moderate_reliability,55,0.8274,0.5288,9.2282
7,Algae,Skeletonema costatum,in_domain_high_reliability,3,0.3168,0.8889,0.5034
8,Algae,Skeletonema costatum,out_of_domain_low_reliability,538,1.0939,0.3767,90.2685
9,Fish,Danio rerio,borderline_moderate_reliability,998,0.5921,0.5013,40.8180


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step9_reliability\step9B_ad_coverage.csv  (24 rows)

Accuracy vs applicability-domain coverage trade-off:


,similarity_threshold,n_predictions,percent_of_test_set,r2,mae,rmse,spearman
0,0.00,13264,100.0000,0.4045,0.7616,1.0276,0.6328
1,0.25,12191,91.9104,0.4224,0.7453,1.0070,0.6472
2,0.40,7683,57.9237,0.4591,0.7352,0.9939,0.6724
3,0.55,4026,30.3528,0.4966,0.7289,0.9966,0.7039


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step9_reliability\step9B_accuracy_vs_coverage.csv  (4 rows)


WindowsPath('C:/Users/lab/Desktop/ecotoxicity/working/step9_reliability/step9B_accuracy_vs_coverage.csv')

In [21]:
# ============================================================
# STEP 9C: DESCRIPTOR SHAP AND FEATURE TRIMMING
# ============================================================
# "We can later use descriptor-based SHAP to trim." (suggestion 2)

shap_importance_df = pd.DataFrame()
trim_results_df = pd.DataFrame()

if not (RUN_SHAP and AVAILABLE.get("shap")):
    print("SHAP is unavailable or disabled - Step 9C is skipped.")
else:
    # SHAP is run on the DESCRIPTOR-only model: descriptor attributions are interpretable,
    # individual fingerprint bits are not.
    SHAP_FEATURE_SET = "ext"
    shap_rows = []

    with Timer("Step 9C: SHAP attribution on the extended descriptors"):
        for organism in MODELLED_ORGANISMS:
            organism_data = aquatic[aquatic["organism_name"] == organism].reset_index(drop=True)
            organism_position = {int(r): i for i, r in enumerate(organism_data["row_id"].values)}
            task_id = f"organism__{organism}"

            X_all = build_feature_matrix(organism_data, SHAP_FEATURE_SET)
            y_all = organism_data["model_target"].values

            train_ids, test_ids = get_split_rows(task_id, 1)
            tr = [organism_position[int(r)] for r in train_ids]
            te = [organism_position[int(r)] for r in test_ids]

            model = build_model("RandomForest", tuned_params(task_id, "RandomForest"))
            model.fit(X_all.iloc[tr], y_all[tr])

            sample = X_all.iloc[te]
            if len(sample) > SHAP_MAX_SAMPLES:
                sample = sample.sample(SHAP_MAX_SAMPLES, random_state=RANDOM_STATE)

            try:
                explainer = shap.TreeExplainer(model)
                shap_values = explainer.shap_values(sample, check_additivity=False)
                mean_absolute_shap = np.abs(shap_values).mean(axis=0)
            except Exception as exc:
                print(f"    SHAP failed for {organism} ({exc}); using impurity importance.")
                mean_absolute_shap = model.feature_importances_

            for feature, importance in zip(X_all.columns, mean_absolute_shap):
                shap_rows.append({
                    "organism_name": organism,
                    "organism_group": organism_data["organism_group"].iloc[0],
                    "descriptor": feature,
                    "mean_abs_shap": float(importance),
                })
            print(f"  SHAP done: {organism}")

    shap_importance_df = pd.DataFrame(shap_rows)
    save_table(shap_importance_df, step9_dir, "step9C_shap_per_organism.csv", show=False)

    # ---- global ranking across organisms ----
    # Importances are normalised within each organism first, so a single high-variance
    # organism cannot dominate the global ranking.
    shap_importance_df["normalised"] = shap_importance_df.groupby("organism_name")[
        "mean_abs_shap"
    ].transform(lambda s: s / (s.sum() + 1e-12))

    global_ranking = (
        shap_importance_df.groupby("descriptor")
        .agg(mean_normalised_shap=("normalised", "mean"),
             mean_abs_shap=("mean_abs_shap", "mean"),
             n_organisms=("organism_name", "nunique"))
        .reset_index()
        .sort_values("mean_normalised_shap", ascending=False)
        .reset_index(drop=True)
    )
    global_ranking["rank"] = np.arange(1, len(global_ranking) + 1)

    print("\nTop 25 descriptors by SHAP across all organisms:")
    display(round_df(global_ranking.head(25), digits=6))
    save_table(round_df(global_ranking, digits=6), step9_dir,
               "step9C_shap_global_ranking.csv", show=False)

    # ---- how far can the descriptor set be trimmed? ----
    trim_rows = []
    with Timer("Step 9C: trimmed descriptor sets"):
        for top_k in [20, 50, 100, len(EXTENDED_DESCRIPTORS)]:
            if top_k > len(EXTENDED_DESCRIPTORS):
                continue
            selected = global_ranking.head(top_k)["descriptor"].tolist()

            for organism in MODELLED_ORGANISMS:
                organism_data = aquatic[
                    aquatic["organism_name"] == organism
                ].reset_index(drop=True)
                organism_position = {
                    int(r): i for i, r in enumerate(organism_data["row_id"].values)
                }
                task_id = f"organism__{organism}"

                X_full = build_feature_matrix(organism_data, "ext")
                X_trimmed = X_full[selected]
                y_all = organism_data["model_target"].values

                for split_id in range(1, min(3, N_SPLITS_MAIN) + 1):
                    train_ids, test_ids = get_split_rows(task_id, split_id)
                    tr = [organism_position[int(r)] for r in train_ids]
                    te = [organism_position[int(r)] for r in test_ids]

                    model = build_model("RandomForest",
                                        tuned_params(task_id, "RandomForest"))
                    model.fit(X_trimmed.iloc[tr], y_all[tr])
                    metrics = evaluate_predictions(
                        y_all[te], model.predict(X_trimmed.iloc[te])
                    )
                    trim_rows.append({
                        "organism_name": organism,
                        "n_descriptors": top_k,
                        "split_id": split_id,
                        **{f"test_{k}": v for k, v in metrics.items()},
                    })
            print(f"  evaluated top-{top_k} descriptor set")

    trim_results_df = pd.DataFrame(trim_rows)
    trim_summary = (
        trim_results_df.groupby("n_descriptors")
        .agg(mean_r2=("test_r2", "mean"), mean_mae=("test_mae", "mean"),
             mean_rmse=("test_rmse", "mean"), mean_spearman=("test_spearman", "mean"))
        .reset_index()
    )
    print("\nPerformance vs number of descriptors kept (how far you can trim):")
    display(round_df(trim_summary))
    save_table(round_df(trim_summary), step9_dir, "step9C_descriptor_trimming.csv", show=False)
    save_table(trim_results_df, step9_dir, "step9C_trimming_per_split.csv", show=False)

    # A recommended trimmed set: the smallest that stays within 0.01 R2 of the full set.
    if len(trim_summary) > 1:
        full_r2 = trim_summary[
            trim_summary["n_descriptors"] == trim_summary["n_descriptors"].max()
        ]["mean_r2"].iloc[0]
        acceptable = trim_summary[trim_summary["mean_r2"] >= full_r2 - 0.01]
        if len(acceptable):
            recommended = int(acceptable["n_descriptors"].min())
            print(f"\nRecommended trimmed descriptor set: top {recommended} "
                  f"(within 0.01 R2 of the full {int(trim_summary['n_descriptors'].max())}).")
            save_table(
                global_ranking.head(recommended)[["rank", "descriptor",
                                                  "mean_normalised_shap"]],
                step9_dir, "step9C_recommended_descriptor_set.csv", show=False,
            )


>>> Step 9C: SHAP attribution on the extended descriptors ...
  SHAP done: Danio rerio
  SHAP done: Daphnia magna
  SHAP done: Desmodesmus subspicatus
  SHAP done: Oncorhynchus mykiss
  SHAP done: Oryzias latipes
  SHAP done: Pimephales promelas
  SHAP done: Raphidocelis subcapitata
  SHAP done: Skeletonema costatum
<<< Step 9C: SHAP attribution on the extended descriptors finished in 604.3 s
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step9_reliability\step9C_shap_per_organism.csv  (1328 rows)

Top 25 descriptors by SHAP across all organisms:


,descriptor,mean_normalised_shap,mean_abs_shap,n_organisms,rank
0,MolLogP,0.170594,0.241711,8,1
1,MolWt,0.089046,0.134501,8,2
2,Chi3v,0.043438,0.068869,8,3
3,BertzCT,0.041227,0.061660,8,4
4,PEOE_VSA6,0.037466,0.052483,8,5
5,AvgIpc,0.028119,0.042472,8,6
6,SMR_VSA7,0.028014,0.040174,8,7
7,FractionCSP3,0.026781,0.040459,8,8
8,EState_VSA8,0.021929,0.030381,8,9
9,SlogP_VSA3,0.021828,0.031225,8,10


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step9_reliability\step9C_shap_global_ranking.csv  (166 rows)

>>> Step 9C: trimmed descriptor sets ...
  evaluated top-20 descriptor set
  evaluated top-50 descriptor set
  evaluated top-100 descriptor set
  evaluated top-166 descriptor set
<<< Step 9C: trimmed descriptor sets finished in 87.0 s

Performance vs number of descriptors kept (how far you can trim):


,n_descriptors,mean_r2,mean_mae,mean_rmse,mean_spearman
0,20,0.3762,0.8163,1.0860,0.6286
1,50,0.3861,0.8050,1.0774,0.6387
2,100,0.3868,0.8048,1.0773,0.6405
3,166,0.3890,0.8026,1.0754,0.6412


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step9_reliability\step9C_descriptor_trimming.csv  (4 rows)
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step9_reliability\step9C_trimming_per_split.csv  (96 rows)

Recommended trimmed descriptor set: top 50 (within 0.01 R2 of the full 166).
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step9_reliability\step9C_recommended_descriptor_set.csv  (50 rows)


## Step 10 — Halogen-specific pattern analysis on halogen-only subsets

*Addresses the Phase 2 inference: "The Phase 2 halogenation analysis adds scientific value beyond
model accuracy by exploring interpretable chemical patterns. Subsequent pattern analysis can be
done with individual datasets halogen only datasets."*

Still obeying the rule that halogen labels never enter a prediction model — here they are used
purely to **slice the data for analysis**.

Three analyses:

**10A — Matched dehalogenated-parent comparison.** Strip F/Cl/Br/I from each molecule to get its
dehalogenated parent. Where the same parent has both halogenated and non-halogenated members, the
toxicity difference is attributable to the halogen rather than to a different scaffold. This is the
Phase 2 analysis, re-run here on the molar target with the matched-pair list exported so individual
pairs can be inspected.

**10B — Per-halogen-class subsets.** Cl-only, F-only, Br-only, I-only and mixed-halogen subsets,
each described separately and each given its own model where the subset is large enough. This
answers whether halogen classes differ in how *predictable* they are, not only how toxic they are.

**10C — Where do the models fail on halogenated compounds?** Residuals from Step 9 are broken down
by halogen class, which shows whether halogenated chemistry is systematically harder for the model.

In [22]:
# ============================================================
# STEP 10A: MATCHED DEHALOGENATED-PARENT ANALYSIS
# ============================================================

step10_dir = OUTPUT_DIR / "step10_halogen_patterns"


def dehalogenated_parent(smiles):
    """Remove every F/Cl/Br/I atom to approximate the parent structure."""
    mol = mol_from_smiles(smiles)
    if mol is None:
        return np.nan
    try:
        parent = Chem.DeleteSubstructs(mol, Chem.MolFromSmarts("[F,Cl,Br,I]"))
        Chem.SanitizeMol(parent)
        parent_smiles = Chem.MolToSmiles(parent, canonical=True)
        return parent_smiles if parent_smiles else np.nan
    except Exception:
        return np.nan


with Timer("Step 10A: matched dehalogenated-parent analysis"):
    halogen_data = ml_ready.copy()
    parent_lookup = {
        s: dehalogenated_parent(s) for s in halogen_data["canonical_smiles"].unique()
    }
    halogen_data["dehalogenated_parent"] = halogen_data["canonical_smiles"].map(parent_lookup)


def matched_parent_analysis(data, group_columns, min_pairs=10):
    """
    Within each analysis group, find dehalogenated parents having BOTH halogenated and
    non-halogenated members, and compare their molar potency.
    """
    pair_records, result_records = [], []

    for keys, subset in data.groupby(group_columns):
        keys = keys if isinstance(keys, tuple) else (keys,)
        key_values = dict(zip(group_columns, keys))

        for parent, parent_rows in subset.dropna(
            subset=["dehalogenated_parent"]
        ).groupby("dehalogenated_parent"):
            halogenated = parent_rows.loc[
                parent_rows["is_halogenated"] == True, "model_target"
            ].dropna()
            plain = parent_rows.loc[
                parent_rows["is_halogenated"] == False, "model_target"
            ].dropna()
            if len(halogenated) >= 1 and len(plain) >= 1:
                pair_records.append({
                    **key_values,
                    "dehalogenated_parent": parent,
                    "n_halogenated": len(halogenated),
                    "n_non_halogenated": len(plain),
                    "median_halogenated": halogenated.median(),
                    "median_non_halogenated": plain.median(),
                    "median_difference": halogenated.median() - plain.median(),
                })

    pair_df = pd.DataFrame(pair_records)
    if pair_df.empty:
        return pair_df, pd.DataFrame()

    for keys, group_rows in pair_df.groupby(group_columns):
        keys = keys if isinstance(keys, tuple) else (keys,)
        key_values = dict(zip(group_columns, keys))
        differences = group_rows["median_difference"].dropna()

        if len(differences) >= min_pairs:
            try:
                statistic, p_value = wilcoxon(differences)
            except Exception:
                statistic, p_value = np.nan, np.nan
            direction = ("halogenated_more_toxic" if differences.median() > 0
                         else "halogenated_less_or_equal")
        else:
            statistic, p_value, direction = np.nan, np.nan, "insufficient_matched_pairs"

        result_records.append({
            **key_values,
            "matched_parent_pairs": len(differences),
            "median_difference": differences.median() if len(differences) else np.nan,
            "mean_difference": differences.mean() if len(differences) else np.nan,
            "wilcoxon_statistic": statistic,
            "p_value": p_value,
            "direction": direction,
        })

    return pair_df, pd.DataFrame(result_records)


overall_pairs, overall_matched = matched_parent_analysis(halogen_data, ["unit"])
group_pairs, group_matched = matched_parent_analysis(
    halogen_data, ["organism_group", "unit"]
)
organism_pairs, organism_matched = matched_parent_analysis(
    halogen_data, ["organism_group", "organism_name", "unit"]
)

print("Matched-parent halogenation effect, overall:")
display(round_df(overall_matched))
print("\nBy biological group:")
display(round_df(group_matched))

save_table(overall_pairs, step10_dir, "step10A_overall_matched_pairs.csv", show=False)
save_table(round_df(overall_matched), step10_dir, "step10A_overall_results.csv", show=False)
save_table(group_pairs, step10_dir, "step10A_group_matched_pairs.csv", show=False)
save_table(round_df(group_matched), step10_dir, "step10A_group_results.csv", show=False)
save_table(organism_pairs, step10_dir, "step10A_organism_matched_pairs.csv", show=False)
save_table(round_df(organism_matched), step10_dir, "step10A_organism_results.csv", show=False)


>>> Step 10A: matched dehalogenated-parent analysis ...
<<< Step 10A: matched dehalogenated-parent analysis finished in 3.1 s
Matched-parent halogenation effect, overall:


,unit,matched_parent_pairs,median_difference,mean_difference,wilcoxon_statistic,p_value,direction
0,mg/L,248,0.4732,0.6139,7141.0,0.0000,halogenated_more_toxic
1,mg/kg,13,0.2407,-0.3582,42.0,0.8394,halogenated_more_toxic



By biological group:


,organism_group,unit,matched_parent_pairs,median_difference,mean_difference,wilcoxon_statistic,p_value,direction
0,Algae,mg/L,102,0.6514,0.5634,1307.0,0.0000,halogenated_more_toxic
1,Avian,mg/kg,7,0.0668,-1.2288,NaN,NaN,insufficient_matched_pairs
2,Fish,mg/L,172,0.6538,0.6838,3079.0,0.0000,halogenated_more_toxic
3,Invertebrates,mg/L,38,0.6360,0.5663,152.0,0.0011,halogenated_more_toxic
4,Terrestrial insects and arthropods,mg/kg,6,0.2861,0.3798,NaN,NaN,insufficient_matched_pairs


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step10_halogen_patterns\step10A_overall_matched_pairs.csv  (261 rows)
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step10_halogen_patterns\step10A_overall_results.csv  (2 rows)
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step10_halogen_patterns\step10A_group_matched_pairs.csv  (325 rows)
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step10_halogen_patterns\step10A_group_results.csv  (5 rows)
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step10_halogen_patterns\step10A_organism_matched_pairs.csv  (309 rows)
  saved: C:\Users\lab\Desktop\ecotoxicity\working\step10_halogen_patterns\step10A_organism_results.csv  (17 rows)


WindowsPath('C:/Users/lab/Desktop/ecotoxicity/working/step10_halogen_patterns/step10A_organism_results.csv')

In [23]:
# ============================================================
# STEP 10B: HALOGEN-ONLY SUBSET DATASETS
# ============================================================
# Individual halogen datasets, as raised in the Phase 2 inferences.

HALOGEN_CLASSES = ["Cl_only", "F_only", "Br_only", "I_only", "mixed_halogen", "non_halogenated"]

subset_rows = []
for halogen_class in HALOGEN_CLASSES:
    subset = ml_ready[ml_ready["halogen_type"] == halogen_class]
    if subset.empty:
        continue
    for (group_name, unit), part in subset.groupby(["organism_group", "unit"]):
        subset_rows.append({
            "halogen_class": halogen_class,
            "organism_group": group_name,
            "unit": unit,
            "rows": len(part),
            "unique_chemicals": part["canonical_smiles"].nunique(),
            "unique_scaffolds": part["scaffold"].nunique(),
            "median_pToxicity_molar": part["model_target"].median(),
            "mean_pToxicity_molar": part["model_target"].mean(),
            "std_pToxicity_molar": part["model_target"].std(),
            "q1": part["model_target"].quantile(0.25),
            "q3": part["model_target"].quantile(0.75),
        })

halogen_subsets_df = pd.DataFrame(subset_rows).sort_values(
    ["organism_group", "halogen_class"]
)
print("Halogen-class subsets by biological group:")
display(round_df(halogen_subsets_df))
save_table(round_df(halogen_subsets_df), step10_dir, "step10B_halogen_subsets.csv", show=False)

# Export each halogen-only dataset so it can be fed to ECOSAR / VEGA / T.E.S.T. directly.
halogen_export_dir = step10_dir / "halogen_only_datasets"
halogen_export_dir.mkdir(parents=True, exist_ok=True)
export_columns = [
    "organism_group", "organism_name", "unit", "chemical_name", "canonical_smiles",
    "scaffold", "toxicity_value_raw", "mol_weight", "model_target",
    "halogen_type", "total_halogen_count",
]
for halogen_class in HALOGEN_CLASSES:
    subset = ml_ready[ml_ready["halogen_type"] == halogen_class]
    if len(subset) < 20:
        continue
    available = [c for c in export_columns if c in subset.columns]
    path = halogen_export_dir / f"halogen_dataset_{halogen_class}.csv"
    subset[available].to_csv(path, index=False)
    print(f"  exported {len(subset):>6} rows -> {path.name}")

# ------------------------------------------------------------------
# Can a model predict WITHIN a single halogen class?
# ------------------------------------------------------------------
# Trained and tested inside one halogen class, so halogen status is constant and
# cannot act as a hidden predictor.
within_class_rows = []
MIN_ROWS_WITHIN_CLASS = 200

with Timer("Step 10B: within-halogen-class modelling"):
    for halogen_class in HALOGEN_CLASSES:
        subset = ml_ready[
            (ml_ready["halogen_type"] == halogen_class) & (ml_ready["unit"] == "mg/L")
        ].reset_index(drop=True)
        n_scaffolds = subset["scaffold"].nunique()
        if len(subset) < MIN_ROWS_WITHIN_CLASS or n_scaffolds < 30:
            print(f"  {halogen_class}: skipped - {len(subset)} rows "
                  f"(need {MIN_ROWS_WITHIN_CLASS}), {n_scaffolds} scaffolds (need 30)")
            continue

        X_all = build_feature_matrix(subset, MAIN_FEATURE_SET)
        y_all = subset["model_target"].values
        splitter = GroupShuffleSplit(
            n_splits=min(3, N_SPLITS_MAIN), test_size=TEST_SIZE, random_state=RANDOM_STATE
        )

        for split_id, (tr, te) in enumerate(
            splitter.split(np.zeros(len(subset)), groups=subset["scaffold"]), 1
        ):
            model = build_model("RandomForest")
            model.fit(X_all.iloc[tr], y_all[tr])
            metrics = evaluate_predictions(y_all[te], model.predict(X_all.iloc[te]))
            within_class_rows.append({
                "halogen_class": halogen_class,
                "rows": len(subset),
                "split_id": split_id,
                "train_rows": len(tr),
                "test_rows": len(te),
                **{f"test_{k}": v for k, v in metrics.items()},
            })
        print(f"  {halogen_class}: modelled ({len(subset)} rows)")

within_class_df = pd.DataFrame(within_class_rows)
if len(within_class_df):
    within_class_summary = (
        within_class_df.groupby(["halogen_class", "rows"])
        .agg(r2_mean=("test_r2", "mean"), r2_std=("test_r2", "std"),
             mae_mean=("test_mae", "mean"), rmse_mean=("test_rmse", "mean"),
             spearman_mean=("test_spearman", "mean"))
        .reset_index().sort_values("r2_mean", ascending=False)
    )
    print("\nPredictability within each halogen class:")
    display(round_df(within_class_summary))
    save_table(round_df(within_class_summary), step10_dir,
               "step10B_within_class_performance.csv", show=False)

Halogen-class subsets by biological group:


,halogen_class,organism_group,unit,rows,unique_chemicals,unique_scaffolds,median_pToxicity_molar,mean_pToxicity_molar,std_pToxicity_molar,q1,q3
10,Br_only,Algae,mg/L,111,95,75,4.4168,4.6835,1.3099,3.6060,5.6795
0,Cl_only,Algae,mg/L,794,662,391,4.2921,4.5208,1.2902,3.5467,5.4739
5,F_only,Algae,mg/L,294,272,211,3.7443,4.1371,1.2845,3.3916,4.8166
15,I_only,Algae,mg/L,28,26,20,4.3265,4.3612,1.2229,3.5964,5.0421
20,mixed_halogen,Algae,mg/L,121,110,82,4.8454,4.9518,1.5385,3.7375,5.9412
25,non_halogenated,Algae,mg/L,4351,4011,2682,3.8082,4.0737,1.2292,3.2886,4.7424
11,Br_only,Avian,mg/kg,12,9,7,2.4491,3.2849,2.0990,2.2878,4.1451
1,Cl_only,Avian,mg/kg,159,118,67,2.3317,3.1727,2.8314,1.7460,3.0487
6,F_only,Avian,mg/kg,26,18,17,2.6586,3.6043,2.4026,2.1733,4.6630
16,I_only,Avian,mg/kg,2,2,2,2.0994,2.0994,0.1094,2.0607,2.1381


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step10_halogen_patterns\step10B_halogen_subsets.csv  (30 rows)
  exported   2852 rows -> halogen_dataset_Cl_only.csv
  exported    766 rows -> halogen_dataset_F_only.csv
  exported    309 rows -> halogen_dataset_Br_only.csv
  exported     74 rows -> halogen_dataset_I_only.csv
  exported    378 rows -> halogen_dataset_mixed_halogen.csv
  exported  11099 rows -> halogen_dataset_non_halogenated.csv

>>> Step 10B: within-halogen-class modelling ...
  Cl_only: modelled (2575 rows)
  F_only: modelled (714 rows)
  Br_only: modelled (284 rows)
  I_only: skipped - 68 rows (need 200), 30 scaffolds (need 30)
  mixed_halogen: modelled (312 rows)
  non_halogenated: modelled (10640 rows)
<<< Step 10B: within-halogen-class modelling finished in 12.4 s

Predictability within each halogen class:


,halogen_class,rows,r2_mean,r2_std,mae_mean,rmse_mean,spearman_mean
4,non_halogenated,10640,0.3393,0.0503,0.7893,1.0690,0.6021
1,Cl_only,2575,0.2874,0.0940,0.9955,1.2863,0.5455
3,mixed_halogen,312,0.2303,0.0734,1.1092,1.4173,0.4986
2,F_only,714,0.1736,0.1359,0.9012,1.2314,0.5262
0,Br_only,284,-0.0026,0.2419,0.9772,1.2287,0.2985


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step10_halogen_patterns\step10B_within_class_performance.csv  (5 rows)


In [24]:
# ============================================================
# STEP 10C: MODEL RESIDUALS BY HALOGEN CLASS
# ============================================================
# Does the model struggle more on halogenated chemistry?

if len(reliability_df):
    residual_by_halogen = (
        reliability_df.groupby("halogen_type")
        .agg(
            n_predictions=("row_id", "count"),
            mean_absolute_error=("absolute_error", "mean"),
            median_absolute_error=("absolute_error", "median"),
            mean_signed_error=("absolute_error", "mean"),
            mean_similarity=("max_train_tanimoto", "mean"),
            mean_tree_spread=("tree_spread", "mean"),
            coverage_standard=("covered_standard", "mean"),
        )
        .reset_index().sort_values("mean_absolute_error", ascending=False)
    )

    # Signed bias: positive means the model under-predicts toxicity for that class.
    reliability_df["signed_error"] = (
        reliability_df["observed"] - reliability_df["predicted"]
    )
    bias = (
        reliability_df.groupby("halogen_type")["signed_error"]
        .agg(["mean", "median"]).reset_index()
        .rename(columns={"mean": "mean_signed_error", "median": "median_signed_error"})
    )
    residual_by_halogen = residual_by_halogen.drop(columns=["mean_signed_error"]).merge(
        bias, on="halogen_type", how="left"
    )

    print("Model error broken down by halogen class:")
    print("(positive signed error = the model UNDER-predicts toxicity for that class)")
    display(round_df(residual_by_halogen))
    save_table(round_df(residual_by_halogen), step10_dir,
               "step10C_residuals_by_halogen_class.csv", show=False)

    # Formal test: halogenated vs non-halogenated absolute error
    halogenated_errors = reliability_df.loc[
        reliability_df["is_halogenated"], "absolute_error"
    ]
    plain_errors = reliability_df.loc[
        ~reliability_df["is_halogenated"], "absolute_error"
    ]
    if len(halogenated_errors) > 10 and len(plain_errors) > 10:
        statistic, p_value = mannwhitneyu(
            halogenated_errors, plain_errors, alternative="two-sided"
        )
        error_test = pd.DataFrame([{
            "comparison": "absolute error: halogenated vs non-halogenated",
            "n_halogenated": len(halogenated_errors),
            "n_non_halogenated": len(plain_errors),
            "median_error_halogenated": halogenated_errors.median(),
            "median_error_non_halogenated": plain_errors.median(),
            "mannwhitney_u": statistic,
            "p_value": p_value,
            "interpretation": (
                "halogenated compounds are harder to predict"
                if halogenated_errors.median() > plain_errors.median()
                else "halogenated compounds are no harder to predict"
            ),
        }])
        display(round_df(error_test))
        save_table(round_df(error_test), step10_dir, "step10C_halogen_error_test.csv",
                   show=False)

Model error broken down by halogen class:
(positive signed error = the model UNDER-predicts toxicity for that class)


,halogen_type,n_predictions,mean_absolute_error,median_absolute_error,mean_similarity,mean_tree_spread,coverage_standard,mean_signed_error,median_signed_error
0,I_only,66,1.0811,0.7998,0.2961,1.1097,0.7424,0.4991,0.5153
1,mixed_halogen,239,0.9143,0.7255,0.3835,1.1209,0.8494,0.2879,0.2428
2,Br_only,242,0.8739,0.7116,0.3938,1.0705,0.8884,0.1686,0.1599
3,Cl_only,2153,0.8487,0.6469,0.4192,1.0674,0.8839,0.1206,0.0037
4,F_only,603,0.7865,0.6074,0.4506,1.0410,0.9221,-0.0010,-0.0860
5,non_halogenated,9961,0.7328,0.5311,0.5005,0.9600,0.9086,0.0038,-0.0815


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step10_halogen_patterns\step10C_residuals_by_halogen_class.csv  (6 rows)


,comparison,n_halogenated,n_non_halogenated,median_error_halogenated,median_error_non_halogenated,mannwhitney_u,p_value,interpretation
0,absolute error: halogenated vs non-halogenated,3303,9961,0.6598,0.5311,18162485.0,0.0,halogenated compounds are harder to predict


  saved: C:\Users\lab\Desktop\ecotoxicity\working\step10_halogen_patterns\step10C_halogen_error_test.csv  (1 rows)


## Step 11 — External validation and regulatory-software benchmarking

*Addresses suggestions 9 and 10: "External benchmark datasets are being compiled. At least two
organisms per category… Benchmarking can be done against regulatory software like OECD database,
T.E.S.T software, ECOSAR and VEGA along with academic ecotox models like ADMET SAR, ADMETLAB,
ProTox, Toxtree and aquatic tox. Also, the standard ML models."*

The external datasets are still being compiled, so this step ships the machinery ready to run:

1. **Templates** — `external_validation_template.csv` (the format to put your compiled external
   data into) and `regulatory_benchmark_template.csv` (where to paste the predictions you get back
   from ECOSAR, VEGA, T.E.S.T., ProTox and so on).
2. **`run_external_validation()`** — drop a filled-in external file into the path shown and
   re-run this cell: it standardises the SMILES, converts to the molar target, scores every Phase 3
   strategy on it, and flags which external compounds fall inside the applicability domain.
3. **`compare_against_regulatory_software()`** — once you have the software predictions, this
   produces the head-to-head table of R²/MAE/RMSE/Spearman per organism per tool.
4. **A coverage check** on the ≥2-organisms-per-category requirement, so you can see at a glance
   whether the compiled external set meets it.

In [25]:
# ============================================================
# STEP 11: EXTERNAL VALIDATION AND BENCHMARKING MACHINERY
# ============================================================

step11_dir = OUTPUT_DIR / "step11_external_benchmark"
step11_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 11.1 Templates
# ------------------------------------------------------------------
external_template = pd.DataFrame({
    "chemical_name": ["example_compound_1", "example_compound_2", "example_compound_3"],
    "smiles": ["CCO", "Clc1ccccc1", "O=C(Nc1ccc(Cl)cc1)N(C)C"],
    "organism_group": ["Fish", "Fish", "Algae"],
    "organism_name": ["Danio rerio", "Oncorhynchus mykiss", "Raphidocelis subcapitata"],
    "unit": ["mg/L", "mg/L", "mg/L"],
    "toxicity_value": [100.0, 10.0, 2.5],
    "endpoint": ["LC50_96h", "LC50_96h", "EC50_72h"],
    "source": ["OECD", "ECOTOX", "EFSA"],
})
external_template.to_csv(step11_dir / "external_validation_template.csv", index=False)

regulatory_template = pd.DataFrame({
    "chemical_name": ["example_compound_1", "example_compound_1"],
    "smiles": ["CCO", "CCO"],
    "organism_name": ["Danio rerio", "Danio rerio"],
    "unit": ["mg/L", "mg/L"],
    "observed_toxicity_value": [100.0, 100.0],
    "software": ["ECOSAR", "T.E.S.T"],
    "predicted_toxicity_value": [85.0, 112.0],
    "software_version": ["2.2", "5.1.2"],
    "notes": ["baseline narcosis class", "consensus method"],
})
regulatory_template.to_csv(step11_dir / "regulatory_benchmark_template.csv", index=False)

print("Templates written:")
print(f"  {step11_dir / 'external_validation_template.csv'}")
print(f"  {step11_dir / 'regulatory_benchmark_template.csv'}")
display(external_template)

# Suggested tools, so the benchmark table has a fixed vocabulary.
BENCHMARK_SOFTWARE = [
    "OECD_QSAR_Toolbox", "T.E.S.T", "ECOSAR", "VEGA",
    "ADMETSAR", "ADMETlab", "ProTox", "Toxtree", "AquaticTox",
]
pd.DataFrame({"software": BENCHMARK_SOFTWARE}).to_csv(
    step11_dir / "benchmark_software_list.csv", index=False
)


# ------------------------------------------------------------------
# 11.2 Coverage check: at least two organisms per category?
# ------------------------------------------------------------------
def check_external_coverage(external_df, min_organisms_per_group=2):
    """Verify the '>= 2 organisms per category' requirement from suggestion 9."""
    coverage = (
        external_df.groupby("organism_group")
        .agg(n_organisms=("organism_name", "nunique"),
             n_records=("organism_name", "count"),
             organisms=("organism_name", lambda s: ", ".join(sorted(set(s)))))
        .reset_index()
    )
    coverage["meets_requirement"] = coverage["n_organisms"] >= min_organisms_per_group
    return coverage


# ------------------------------------------------------------------
# 11.3 Score the Phase 3 models on an external file
# ------------------------------------------------------------------
def prepare_external(external_df):
    """Standardise an external file into the notebook's internal format."""
    data = external_df.copy()
    smiles_column = "smiles" if "smiles" in data.columns else "canonical_smiles"
    data["canonical_smiles"] = data[smiles_column].apply(canonicalize_smiles)
    data["valid_smiles"] = data["canonical_smiles"].notna()

    descriptors = pd.DataFrame(
        data["canonical_smiles"].apply(compute_basic_descriptors).tolist()
    )
    data = pd.concat([data.reset_index(drop=True), descriptors], axis=1)
    data = pd.concat(
        [data, pd.DataFrame(data["canonical_smiles"].apply(halogen_counts).tolist())], axis=1
    )
    data["scaffold"] = data["canonical_smiles"].apply(get_scaffold)
    data["toxicity_value_raw"] = pd.to_numeric(data["toxicity_value"], errors="coerce")
    data["model_target"] = data.apply(convert_to_molar_target, axis=1)
    data["organism_ready_rows"] = 0

    usable = data[data["valid_smiles"] & data["model_target"].notna()].reset_index(drop=True)
    print(f"  external rows: {len(data)}  usable after SMILES/target checks: {len(usable)}")
    return usable


def run_external_validation(external_path):
    """
    Train on ALL internal data for each organism and score the external compounds.

    External compounds that share a scaffold with the internal training data are flagged,
    because those are not a truly independent test.
    """
    external_df = pd.read_csv(external_path)
    external = prepare_external(external_df)

    coverage = check_external_coverage(external)
    print("\nExternal dataset coverage:")
    display(coverage)
    save_table(coverage, step11_dir, "step11_external_coverage.csv", show=False)

    # Extended descriptors for the external chemicals must be added to the lookup.
    global EXTENDED_LOOKUP, UNIQUE_SMILES_ORDER, SMILES_TO_POSITION, _FP_CACHE
    new_smiles = [
        s for s in external["canonical_smiles"].unique() if s not in EXTENDED_LOOKUP.index
    ]
    if new_smiles:
        values = np.array([compute_extended_descriptors(s) for s in new_smiles], dtype=float)
        frame = pd.DataFrame(values, columns=ALL_RDKIT_DESCRIPTOR_NAMES, index=new_smiles)
        frame = frame.replace([np.inf, -np.inf], np.nan)[EXTENDED_DESCRIPTORS]
        frame = frame.fillna(EXTENDED_LOOKUP.median())
        EXTENDED_LOOKUP = pd.concat([EXTENDED_LOOKUP, frame])

        # Extend the fingerprint cache to cover the new chemicals.
        UNIQUE_SMILES_ORDER = UNIQUE_SMILES_ORDER + new_smiles
        SMILES_TO_POSITION = {s: i for i, s in enumerate(UNIQUE_SMILES_ORDER)}
        _FP_CACHE = {}

    results, predictions = [], []

    for organism, external_subset in external.groupby("organism_name"):
        internal = aquatic[aquatic["organism_name"] == organism].reset_index(drop=True)
        if len(internal) < 50:
            print(f"  {organism}: too little internal data ({len(internal)}) - skipped")
            continue

        group_name = internal["organism_group"].iloc[0]
        vocabulary = GROUP_ORGANISMS.get(group_name, [organism])
        task_id = f"organism__{organism}"

        internal_scaffolds = set(internal["scaffold"])
        external_subset = external_subset.reset_index(drop=True)
        external_subset["scaffold_seen_in_training"] = (
            external_subset["scaffold"].isin(internal_scaffolds)
        )

        y_external = external_subset["model_target"].values

        # --- independent organism model ---
        X_internal = build_feature_matrix(internal, MAIN_FEATURE_SET)
        X_external = build_feature_matrix(external_subset, MAIN_FEATURE_SET)
        independent = build_model("RandomForest", tuned_params(task_id, "RandomForest"))
        independent.fit(X_internal, internal["model_target"].values)
        strategy_predictions = {"independent_organism": independent.predict(X_external)}

        # --- species-aware pooled model ---
        pool = aquatic[aquatic["organism_group"] == group_name].reset_index(drop=True)
        X_pool = build_species_aware_matrix(pool, vocabulary)
        X_external_aware = build_species_aware_matrix(external_subset, vocabulary)
        aware = build_model(
            "RandomForest", tuned_params(f"group__{group_name}__mg/L", "RandomForest")
        )
        aware.fit(X_pool, pool["model_target"].values)
        strategy_predictions["species_aware_pooled"] = aware.predict(X_external_aware)

        # --- applicability domain for the external compounds ---
        similarity = max_tanimoto_to_train(
            morgan_bit_matrix(external_subset["canonical_smiles"]),
            morgan_bit_matrix(internal["canonical_smiles"]),
        )

        for strategy, y_pred in strategy_predictions.items():
            metrics = evaluate_predictions(y_external, y_pred)
            results.append({
                "organism_group": group_name,
                "organism_name": organism,
                "strategy": strategy,
                "n_external": len(external_subset),
                "n_novel_scaffolds": int(
                    (~external_subset["scaffold_seen_in_training"]).sum()
                ),
                "mean_similarity_to_training": float(similarity.mean()),
                **{f"external_{k}": v for k, v in metrics.items()},
            })
            for i in range(len(external_subset)):
                predictions.append({
                    "organism_name": organism,
                    "strategy": strategy,
                    "chemical_name": external_subset.loc[i, "chemical_name"],
                    "canonical_smiles": external_subset.loc[i, "canonical_smiles"],
                    "observed": float(y_external[i]),
                    "predicted": float(y_pred[i]),
                    "absolute_error": abs(float(y_external[i]) - float(y_pred[i])),
                    "max_train_tanimoto": float(similarity[i]),
                    "scaffold_seen_in_training": bool(
                        external_subset.loc[i, "scaffold_seen_in_training"]
                    ),
                    "in_applicability_domain": bool(similarity[i] >= 0.40),
                })
        print(f"  {organism}: validated on {len(external_subset)} external records")

    results_df = pd.DataFrame(results)
    predictions_external = pd.DataFrame(predictions)

    if len(results_df):
        save_table(round_df(results_df), step11_dir, "step11_external_results.csv")
        save_table(predictions_external, step11_dir, "step11_external_predictions.csv",
                   show=False)

        in_domain = predictions_external[predictions_external["in_applicability_domain"]]
        if len(in_domain) > 10:
            print("\nExternal performance restricted to in-domain compounds:")
            for strategy, subset in in_domain.groupby("strategy"):
                metrics = evaluate_predictions(subset["observed"], subset["predicted"])
                print(f"  {strategy:<25} n={len(subset):>4}  "
                      f"R2={metrics['r2']:.3f}  MAE={metrics['mae']:.3f}  "
                      f"rho={metrics['spearman']:.3f}")

    return results_df, predictions_external


# ------------------------------------------------------------------
# 11.4 Head-to-head against regulatory software
# ------------------------------------------------------------------
def compare_against_regulatory_software(benchmark_path, our_predictions):
    """
    Compare Phase 3 predictions with the predictions exported from the regulatory tools.

    `benchmark_path` must follow regulatory_benchmark_template.csv. Software predictions
    are given in the ORIGINAL units (mg/L) and are converted to the molar target here so
    the comparison is like-for-like.
    """
    benchmark = pd.read_csv(benchmark_path)
    benchmark["canonical_smiles"] = benchmark["smiles"].apply(canonicalize_smiles)
    molecular_weights = benchmark["canonical_smiles"].apply(
        lambda s: compute_basic_descriptors(s)["mol_weight"]
    )
    benchmark["mol_weight"] = molecular_weights

    def to_molar(value, mw):
        if pd.isna(value) or pd.isna(mw) or value <= 0 or mw <= 0:
            return np.nan
        return -np.log10((value / 1000.0) / mw)

    benchmark["observed_target"] = [
        to_molar(v, m) for v, m in zip(benchmark["observed_toxicity_value"],
                                       benchmark["mol_weight"])
    ]
    benchmark["predicted_target"] = [
        to_molar(v, m) for v, m in zip(benchmark["predicted_toxicity_value"],
                                       benchmark["mol_weight"])
    ]

    rows = []
    usable = benchmark.dropna(subset=["observed_target", "predicted_target"])
    for (software, organism), subset in usable.groupby(["software", "organism_name"]):
        if len(subset) < 5:
            continue
        metrics = evaluate_predictions(subset["observed_target"], subset["predicted_target"])
        rows.append({
            "source": software, "organism_name": organism,
            "n_compounds": len(subset), **metrics,
        })

    if our_predictions is not None and len(our_predictions):
        for (strategy, organism), subset in our_predictions.groupby(
            ["strategy", "organism_name"]
        ):
            if len(subset) < 5:
                continue
            metrics = evaluate_predictions(subset["observed"], subset["predicted"])
            rows.append({
                "source": f"Phase3_{strategy}", "organism_name": organism,
                "n_compounds": len(subset), **metrics,
            })

    comparison = pd.DataFrame(rows).sort_values(["organism_name", "r2"], ascending=[True, False])
    if len(comparison):
        save_table(round_df(comparison), step11_dir, "step11_regulatory_comparison.csv")
    return comparison


# ------------------------------------------------------------------
# 11.5 Run it now if the files are present
# ------------------------------------------------------------------
EXTERNAL_DATA_PATH = Path("/kaggle/input/external-validation/external_validation.csv")
REGULATORY_DATA_PATH = Path("/kaggle/input/external-validation/regulatory_predictions.csv")

external_results_df = pd.DataFrame()
external_predictions_df = pd.DataFrame()

if EXTERNAL_DATA_PATH.exists():
    print(f"\nExternal dataset found: {EXTERNAL_DATA_PATH}")
    with Timer("Step 11: external validation"):
        external_results_df, external_predictions_df = run_external_validation(
            EXTERNAL_DATA_PATH
        )

    if REGULATORY_DATA_PATH.exists():
        print(f"\nRegulatory predictions found: {REGULATORY_DATA_PATH}")
        regulatory_comparison = compare_against_regulatory_software(
            REGULATORY_DATA_PATH, external_predictions_df
        )
        display(round_df(regulatory_comparison))
else:
    print("\nNo external dataset attached yet - this is expected while it is being compiled.")
    print("When the files are ready:")
    print("  1. Fill in external_validation_template.csv with the compiled data.")
    print("  2. Attach it to the notebook as a Kaggle dataset named 'external-validation'.")
    print("  3. Re-run this cell. Everything downstream works unchanged.")

Templates written:
  C:\Users\lab\Desktop\ecotoxicity\working\step11_external_benchmark\external_validation_template.csv
  C:\Users\lab\Desktop\ecotoxicity\working\step11_external_benchmark\regulatory_benchmark_template.csv


,chemical_name,smiles,organism_group,organism_name,unit,toxicity_value,endpoint,source
0,example_compound_1,CCO,Fish,Danio rerio,mg/L,100.0,LC50_96h,OECD
1,example_compound_2,Clc1ccccc1,Fish,Oncorhynchus mykiss,mg/L,10.0,LC50_96h,ECOTOX
2,example_compound_3,O=C(Nc1ccc(Cl)cc1)N(C)C,Algae,Raphidocelis subcapitata,mg/L,2.5,EC50_72h,EFSA



No external dataset attached yet - this is expected while it is being compiled.
When the files are ready:
  1. Fill in external_validation_template.csv with the compiled data.
  2. Attach it to the notebook as a Kaggle dataset named 'external-validation'.
  3. Re-run this cell. Everything downstream works unchanged.


## Step 12 — Final tables and figures

Everything the report needs, gathered in one folder: numbered tables as CSV and publication-quality
figures at 300 dpi. A `run_manifest.json` records the exact configuration, library versions and
timestamp that produced them.

In [26]:
# ============================================================
# STEP 12: FINAL TABLES AND FIGURES
# ============================================================

step12_dir = OUTPUT_DIR / "step12_final_tables_figures"
step12_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 12.1 Numbered tables
# ------------------------------------------------------------------
final_tables = {
    "Table1_dataset_summary.csv": dataset_summary,
    "Table2_feature_set_comparison.csv": feature_screen_summary,
    "Table3_tuned_hyperparameters.csv": tuning_df if len(tuning_df) else pd.DataFrame(),
    "Table4_strategy_comparison.csv": overall_strategy,
    "Table5_strategy_by_dataset_size.csv": by_size,
    "Table6_paired_strategy_tests.csv": paired_tests_df,
    "Table7_metrics_with_CIs.csv": report_table,
    "Table8_best_per_organism.csv": best_per_organism,
    "Table9_ensemble_vs_single.csv": ensemble_vs_single,
    "Table10_error_by_similarity.csv": error_by_similarity,
    "Table11_conformal_coverage.csv": coverage_summary,
    "Table12_matched_halogenation.csv": group_matched,
    "Table13_halogen_subsets.csv": halogen_subsets_df,
}
if len(multitask_df):
    final_tables["Table14_multitask_nn.csv"] = multitask_summary
if RUN_SHAP and len(shap_importance_df):
    final_tables["Table15_shap_descriptor_ranking.csv"] = global_ranking.head(50)
    final_tables["Table16_descriptor_trimming.csv"] = trim_summary
if len(external_results_df):
    final_tables["Table17_external_validation.csv"] = external_results_df

for filename, table in final_tables.items():
    if table is not None and len(table):
        round_df(table).to_csv(step12_dir / filename, index=False)
        print(f"  {filename}  ({len(table)} rows)")

# ------------------------------------------------------------------
# 12.2 Figures
# ------------------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
    "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True,
})


def save_figure(fig, name):
    path = step12_dir / name
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"  saved {name}")


# --- Figure 1: feature-set comparison ---
fig, ax = plt.subplots(figsize=(9, 5))
plot_data = feature_screen_summary.sort_values("mean_spearman")
colors = ["#2d6a9f" if str(f).startswith("ext") else "#9f6b2d"
          for f in plot_data["feature_set"]]
ax.barh(plot_data["feature_set"], plot_data["mean_spearman"], color=colors)
ax.set_xlabel("Mean Spearman correlation (scaffold-held-out)")
ax.set_title("Feature-set comparison: extended RDKit descriptors vs the Phase-2 set")
handles = [
    plt.Rectangle((0, 0), 1, 1, color="#2d6a9f"),
    plt.Rectangle((0, 0), 1, 1, color="#9f6b2d"),
]
ax.legend(handles, ["extended RDKit (~200)", "basic 11 (Phase 2)"], loc="lower right")
save_figure(fig, "Figure1_feature_set_comparison.png")

# --- Figure 2: strategy comparison per organism ---
pivot = (
    strategy_summary[strategy_summary["model"] == SPECIES_AWARE_MODELS[0]]
    .pivot_table(index="organism_name", columns="strategy", values="r2_mean")
)
if len(pivot):
    pivot = pivot.sort_values(
        "independent_organism" if "independent_organism" in pivot.columns else pivot.columns[0]
    )
    fig, ax = plt.subplots(figsize=(11, 5.5))
    x = np.arange(len(pivot))
    width = 0.8 / max(len(pivot.columns), 1)
    for i, column in enumerate(pivot.columns):
        ax.bar(x + i * width - 0.4 + width / 2, pivot[column], width, label=column)
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=30, ha="right")
    ax.set_ylabel("Mean R² (held-out scaffolds)")
    ax.set_title("Modelling strategy comparison on identical scaffold splits")
    ax.legend(fontsize=8)
    ax.axhline(0, color="black", linewidth=0.8)
    save_figure(fig, "Figure2_strategy_comparison.png")

# --- Figure 3: does species-awareness help small datasets? ---
size_pivot = by_size.pivot_table(
    index="dataset_size_category", columns="strategy", values="mean_r2"
)
if len(size_pivot):
    order = [c for c in ["very_small", "small", "medium", "strong"] if c in size_pivot.index]
    size_pivot = size_pivot.loc[order] if order else size_pivot
    fig, ax = plt.subplots(figsize=(9, 5))
    size_pivot.plot(kind="bar", ax=ax, width=0.8)
    ax.set_ylabel("Mean R²")
    ax.set_xlabel("Dataset size category")
    ax.set_title("Does species-aware pooling help organisms with limited data?")
    ax.legend(fontsize=8)
    plt.xticks(rotation=0)
    save_figure(fig, "Figure3_strategy_by_dataset_size.png")

# --- Figure 4: error vs chemical similarity ---
if len(error_by_similarity):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    ax1.bar(error_by_similarity["similarity_band"],
            error_by_similarity["mean_absolute_error"], color="#b5485d")
    ax1.set_ylabel("Mean absolute error (log units)")
    ax1.set_title("Error vs nearest-neighbour similarity")
    ax1.tick_params(axis="x", rotation=30)

    ax2.plot(error_by_similarity["similarity_band"],
             error_by_similarity["coverage_standard"], "o-", label="standard conformal")
    ax2.plot(error_by_similarity["similarity_band"],
             error_by_similarity["coverage_normalised"], "s-", label="normalised conformal")
    ax2.axhline(1 - CONFORMAL_ALPHA, linestyle="--", color="black",
                label=f"nominal {1 - CONFORMAL_ALPHA:.0%}")
    ax2.set_ylabel("Empirical coverage")
    ax2.set_title("Prediction-interval coverage by similarity band")
    ax2.tick_params(axis="x", rotation=30)
    ax2.legend(fontsize=8)
    save_figure(fig, "Figure4_reliability_and_coverage.png")

# --- Figure 5: observed vs predicted, with intervals ---
if len(reliability_df):
    fig, ax = plt.subplots(figsize=(6.5, 6))
    sample = reliability_df.sample(min(2000, len(reliability_df)), random_state=RANDOM_STATE)
    scatter = ax.scatter(
        sample["observed"], sample["predicted"], c=sample["max_train_tanimoto"],
        cmap="viridis", s=12, alpha=0.6,
    )
    limits = [
        min(sample["observed"].min(), sample["predicted"].min()),
        max(sample["observed"].max(), sample["predicted"].max()),
    ]
    ax.plot(limits, limits, "k--", linewidth=1)
    ax.set_xlabel("Observed pToxicity (molar)")
    ax.set_ylabel("Predicted pToxicity (molar)")
    ax.set_title("Observed vs predicted, coloured by similarity to training set")
    plt.colorbar(scatter, ax=ax, label="max Tanimoto to training set")
    save_figure(fig, "Figure5_observed_vs_predicted.png")

# --- Figure 6: matched halogenation effect ---
if len(group_matched) and group_matched["median_difference"].notna().any():
    plot_data = group_matched.dropna(subset=["median_difference"]).copy()
    plot_data["label"] = plot_data["organism_group"] + " (" + plot_data["unit"] + ")"
    plot_data = plot_data.sort_values("median_difference")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    colors = ["#2d6a9f" if v > 0 else "#9f2d2d" for v in plot_data["median_difference"]]
    ax.barh(plot_data["label"], plot_data["median_difference"], color=colors)
    ax.axvline(0, linestyle="--", color="black")
    ax.set_xlabel("Median difference in pToxicity: halogenated − non-halogenated")
    ax.set_title("Scaffold-matched halogenation effect (positive = halogenated more toxic)")
    save_figure(fig, "Figure6_matched_halogenation.png")

# --- Figure 7: SHAP descriptor ranking ---
if RUN_SHAP and len(shap_importance_df):
    top = global_ranking.head(20).sort_values("mean_normalised_shap")
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(top["descriptor"], top["mean_normalised_shap"], color="#3d7a5a")
    ax.set_xlabel("Mean normalised |SHAP| across organisms")
    ax.set_title("Top 20 descriptors driving the toxicity predictions")
    save_figure(fig, "Figure7_shap_descriptor_ranking.png")

    if len(trim_summary) > 1:
        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.plot(trim_summary["n_descriptors"], trim_summary["mean_r2"], "o-")
        ax.set_xlabel("Number of descriptors kept (SHAP-ranked)")
        ax.set_ylabel("Mean R²")
        ax.set_title("How far the descriptor set can be trimmed")
        save_figure(fig, "Figure8_descriptor_trimming.png")

  Table1_dataset_summary.csv  (21 rows)
  Table2_feature_set_comparison.csv  (8 rows)
  Table3_tuned_hyperparameters.csv  (66 rows)
  Table4_strategy_comparison.csv  (12 rows)
  Table5_strategy_by_dataset_size.csv  (8 rows)
  Table6_paired_strategy_tests.csv  (9 rows)
  Table7_metrics_with_CIs.csv  (104 rows)
  Table8_best_per_organism.csv  (8 rows)
  Table9_ensemble_vs_single.csv  (3 rows)
  Table10_error_by_similarity.csv  (5 rows)
  Table11_conformal_coverage.csv  (1 rows)
  Table12_matched_halogenation.csv  (5 rows)
  Table13_halogen_subsets.csv  (30 rows)
  Table14_multitask_nn.csv  (8 rows)
  Table15_shap_descriptor_ranking.csv  (50 rows)
  Table16_descriptor_trimming.csv  (4 rows)
  saved Figure1_feature_set_comparison.png
  saved Figure2_strategy_comparison.png
  saved Figure3_strategy_by_dataset_size.png
  saved Figure4_reliability_and_coverage.png
  saved Figure5_observed_vs_predicted.png
  saved Figure6_matched_halogenation.png
  saved Figure7_shap_descriptor_ranking.png
  s

In [27]:
# ============================================================
# STEP 12 (continued): RUN MANIFEST AND WRITTEN SUMMARY
# ============================================================

import platform
import sklearn

manifest = {
    "notebook": "Phase 3 - Ecotoxicity Modelling",
    "generated_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "run_profile": RUN_PROFILE,
    "random_state": RANDOM_STATE,
    "configuration": {
        "outer_splits": N_SPLITS_MAIN,
        "inner_folds": N_INNER_FOLDS,
        "optuna_trials": OPTUNA_TRIALS,
        "test_size": TEST_SIZE,
        "fingerprint_sizes_tested": FP_BITS_TO_TEST,
        "main_feature_set": MAIN_FEATURE_SET,
        "conformal_alpha": CONFORMAL_ALPHA,
        "exclude_explicit_halogen_descriptors": EXCLUDE_EXPLICIT_HALOGEN_DESCRIPTORS,
        "multitask_splits": MTL_MAX_SPLITS,
        "bootstrap_resamples": N_BOOTSTRAP,
    },
    "data": {
        "ml_ready_rows": int(len(ml_ready)),
        "unique_chemicals": int(ml_ready["canonical_smiles"].nunique()),
        "unique_scaffolds": int(ml_ready["scaffold"].nunique()),
        "modelled_organisms": MODELLED_ORGANISMS,
        "modelled_groups": [f"{g} | {u}" for g, u in MODELLED_GROUPS],
        "extended_descriptors_kept": len(EXTENDED_DESCRIPTORS),
    },
    "environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "rdkit": getattr(__import__("rdkit"), "__version__", "unknown"),
        "optional_libraries": AVAILABLE,
    },
}

with open(OUTPUT_DIR / "run_manifest.json", "w") as handle:
    json.dump(manifest, handle, indent=2, default=str)

print("Run manifest written to:", OUTPUT_DIR / "run_manifest.json")

# ------------------------------------------------------------------
# Written summary with the headline numbers filled in
# ------------------------------------------------------------------
def safe(value, fmt="{:.4f}"):
    try:
        return fmt.format(float(value))
    except Exception:
        return "n/a"


best_feature = feature_screen_summary.iloc[0]
best_strategy = overall_strategy.sort_values("mean_r2", ascending=False).iloc[0]

summary_lines = [
    "PHASE 3 ECOTOXICITY MODELLING - RESULTS SUMMARY",
    "=" * 70,
    f"Generated : {manifest['generated_utc']} UTC",
    f"Profile   : {RUN_PROFILE}  ({N_SPLITS_MAIN} outer splits, {N_INNER_FOLDS} inner folds)",
    f"Data      : {len(ml_ready)} records, {ml_ready['canonical_smiles'].nunique()} chemicals, "
    f"{ml_ready['scaffold'].nunique()} scaffolds",
    "",
    "1. FEATURES (suggestion 2)",
    f"   Best feature set      : {best_feature['feature_set']} "
    f"({int(best_feature['n_features'])} features)",
    f"   Mean Spearman         : {safe(best_feature['mean_spearman'])}",
    f"   Mean R2               : {safe(best_feature['mean_r2'])}",
    f"   Extended descriptors kept after cleaning: {len(EXTENDED_DESCRIPTORS)}",
    "",
    "2. TUNING (suggestion 1)",
    f"   Tuned parameter sets saved: {len(BEST_PARAMS)}",
    f"   Model families tuned      : {', '.join(MODELS_TO_TUNE)}",
    "",
    "3. SPECIES-AWARE MODELLING (suggestion 3)",
    f"   Best overall strategy : {best_strategy['strategy']} ({best_strategy['model']})",
    f"   Mean R2               : {safe(best_strategy['mean_r2'])}",
    f"   Mean Spearman         : {safe(best_strategy['mean_spearman'])}",
    "",
    "4. ENSEMBLE (suggestion 7)",
    f"   Best ensemble/single  : {ensemble_vs_single.iloc[0]['strategy']} "
    f"(R2 = {safe(ensemble_vs_single.iloc[0]['mean_r2'])})",
    "",
    "5. RELIABILITY (suggestion 8)",
    f"   Tanimoto verified against RDKit : max difference "
    f"{safe(max_difference, '{:.2e}')}",
    f"   Nominal interval coverage       : {1 - CONFORMAL_ALPHA:.0%}",
    f"   Empirical coverage (standard)   : "
    f"{safe(coverage_summary['empirical_coverage_standard'].iloc[0], '{:.1%}')}",
    f"   Empirical coverage (normalised) : "
    f"{safe(coverage_summary['empirical_coverage_normalised'].iloc[0], '{:.1%}')}",
    "",
    "6. HALOGENATION (Phase 2 inference)",
]

for _, row in group_matched.iterrows():
    summary_lines.append(
        f"   {row['organism_group']:<36} {row['matched_parent_pairs']:>4} matched pairs, "
        f"median difference {safe(row['median_difference'], '{:+.3f}')}, "
        f"p = {safe(row['p_value'], '{:.3g}')}"
    )

if len(multitask_df):
    summary_lines += [
        "",
        "7. NEURAL BASELINE (suggestion 4)",
        f"   Multitask network mean R2 : {safe(multitask_df['test_r2'].mean())}",
        f"   Evaluated on {MTL_MAX_SPLITS} split(s) per organism",
    ]

summary_lines += [
    "",
    "8. EXTERNAL VALIDATION (suggestions 9, 10)",
    f"   Status: {'completed' if len(external_results_df) else 'machinery ready, awaiting the compiled datasets'}",
    "",
    "=" * 70,
    "Tables are in step12_final_tables_figures/. The exact configuration that produced",
    "these numbers is recorded in run_manifest.json.",
]

summary_text = "\n".join(summary_lines)
with open(step12_dir / "Phase3_results_summary.txt", "w") as handle:
    handle.write(summary_text)

print("\n" + summary_text)

Run manifest written to: C:\Users\lab\Desktop\ecotoxicity\working\run_manifest.json

PHASE 3 ECOTOXICITY MODELLING - RESULTS SUMMARY
Generated : 2026-11-17 22:53:57 UTC
Profile   : standard  (5 outer splits, 3 inner folds)
Data      : 15478 records, 8112 chemicals, 4840 scaffolds

1. FEATURES (suggestion 2)
   Best feature set      : ext_morgan1024 (1190 features)
   Mean Spearman         : 0.6305
   Mean R2               : 0.3739
   Extended descriptors kept after cleaning: 166

2. TUNING (suggestion 1)
   Tuned parameter sets saved: 66
   Model families tuned      : RandomForest, ExtraTrees, HistGB, XGBoost, LightGBM, CatBoost

3. SPECIES-AWARE MODELLING (suggestion 3)
   Best overall strategy : hierarchical_residual (RandomForest)
   Mean R2               : 0.4073
   Mean Spearman         : 0.6524

4. ENSEMBLE (suggestion 7)
   Best ensemble/single  : simple_average (R2 = 0.3971)

5. RELIABILITY (suggestion 8)
   Tanimoto verified against RDKit : max difference 0.00e+00
   Nominal i

## Step 13 — Download everything

Run this last cell, then click the link to download a single ZIP containing every table, figure,
saved parameter set and the run manifest.

In [28]:
# ============================================================
# STEP 13: PACKAGE THE OUTPUTS FOR DOWNLOAD
# ============================================================

import shutil
from IPython.display import FileLink, display

zip_path = shutil.make_archive(
    base_name="/kaggle/working/ecotox_phase3_outputs",
    format="zip",
    root_dir=str(OUTPUT_DIR),
)

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"ZIP created: {zip_path}  ({size_mb:.1f} MB)")
print("\nContents:")
for folder in sorted(OUTPUT_DIR.iterdir()):
    if folder.is_dir():
        n_files = len(list(folder.rglob("*")))
        print(f"  {folder.name}/  ({n_files} files)")

print("\nFor the professor, the folder that matters is step12_final_tables_figures/.")
display(FileLink(zip_path))

ZIP created: C:\kaggle\working\ecotox_phase3_outputs.zip  (14.0 MB)

Contents:
  step10_halogen_patterns/  (17 files)
  step11_external_benchmark/  (3 files)
  step12_final_tables_figures/  (25 files)
  step1_data/  (2 files)
  step2_features/  (2 files)
  step3_splits/  (6 files)
  step4_tuning/  (3 files)
  step5_species_aware/  (6 files)
  step6_multitask_nn/  (4 files)
  step7_metrics/  (4 files)
  step8_stacking/  (5 files)
  step9_reliability/  (11 files)

For the professor, the folder that matters is step12_final_tables_figures/.


C:\kaggle\working\ecotox_phase3_outputs.zip

---

## How to read the results

**The three questions Phase 3 was set up to answer**

1. *Do richer descriptors help?* → `Table2_feature_set_comparison.csv` and `Figure1`. The
   comparison is like-for-like: one fixed Random Forest, identical scaffold splits, only the
   features change.
2. *Do species-aware pooled models beat separate organism models?* → `Table4`, `Table5`, `Table6`
   and `Figure2`/`Figure3`. `Table6` gives the paired Wilcoxon test across matched splits, which is
   the statistically correct comparison; `Table5` answers the small-data question specifically.
3. *Can the predictions be trusted?* → `Table10`, `Table11` and `Figure4`. Error rises as test
   compounds get chemically further from the training set, and the conformal intervals report their
   actual coverage rather than a nominal claim.

**Honest reporting.** If a Phase 3 method does not beat Phase 2, the notebook says so in plain
words in its verdict blocks. A well-documented negative result on the same splits is a real finding
and is worth more than a comparison tuned until it wins.

**Known limitations to state in the write-up**

- mg/kg organisms (avian, terrestrial arthropods) are too small for individual models and stay in
  the descriptive analysis only.
- Conformal coverage is guaranteed under exchangeability; scaffold-based splitting deliberately
  breaks that, so observed coverage can sit below nominal. That gap is itself a useful measure of
  how far the chemical shift goes.
- The halogenation analysis is correlational. Matched dehalogenated parents control for scaffold,
  not for every confounder.
- External validation is pending the compiled datasets; Step 11 runs unchanged once they arrive.

**Reference for the species-aware design:** Yin, Z. & Li, J. *Multimodal Deep Learning for
Species-Aware Prediction of Chemical Toxicity to Aquatic Plants.* Journal of Hazardous Materials
(2026). doi:10.1016/j.jhazmat.2026.143864 — conditions a shared chemical encoder on taxonomic
identity and evaluates under chemical-cold and species-cold splits, the same logic applied here in
tabular form in Steps 5 and 6.